Load and validate the DEG dataset

In [1]:

# ── Load and validate the DEG dataset ──────────────────────────────────────────
deg_file <- "/mnt/user-uploads/DEG_OSD498_510_radiation_effect.csv"

# Read the CSV; the first column is unnamed (it holds the AGI gene ID)
deg <- read.csv(deg_file, header = TRUE, stringsAsFactors = FALSE)

# Inspect structure
cat("=== Dimensions ===\n")
cat("Rows:", nrow(deg), " Cols:", ncol(deg), "\n")
cat("\n=== Column names ===\n")
print(colnames(deg))
cat("\n=== First 3 rows ===\n")
print(head(deg, 3))
cat("\n=== Last 3 rows ===\n")
print(tail(deg, 3))

# The first column is unnamed — it's the gene ID. Rename it.
colnames(deg)[1] <- "gene_id"
cat("\n=== After renaming first column ===\n")
print(colnames(deg))

# Check the DEG flag column (last column, unnamed)
cat("\n=== DEG flag values ===\n")
print(table(deg[, ncol(deg)]))

# Rename the last column to 'deg_flag'
colnames(deg)[ncol(deg)] <- "deg_flag"
cat("\n=== Final column names ===\n")
print(colnames(deg))

# Validate AGI ID format (AT{1-5}Gnnnnn)
agi_pattern <- "^AT[1-5]G[0-9]{5}$"
valid_ids <- grepl(agi_pattern, deg$gene_id)
cat("\n=== AGI ID validation ===\n")
cat("Valid AGI IDs:", sum(valid_ids), "out of", nrow(deg), "\n")
cat("Invalid examples:", paste(head(deg$gene_id[!valid_ids], 10), collapse=", "), "\n")

# Count DEGs
n_deg <- sum(deg$deg_flag == "yes", na.rm = TRUE)
cat("\n=== DEG count ===\n")
cat("Total DEGs (flag='yes'):", n_deg, "\n")
cat("Non-DEGs (flag='no'):", sum(deg$deg_flag == "no", na.rm = TRUE), "\n")

# Check for missing values in key columns
cat("\n=== Missing values ===\n")
for (col in c("baseMean","log2FoldChange","padj","deg_flag")) {
  cat(col, ":", sum(is.na(deg[[col]])), "NA\n")
}

# Check value ranges
cat("\n=== log2FoldChange range ===\n")
cat("Min:", min(deg$log2FoldChange, na.rm=TRUE), " Max:", max(deg$log2FoldChange, na.rm=TRUE), "\n")
cat("\n=== padj range ===\n")
cat("Min:", min(deg$padj, na.rm=TRUE), " Max:", max(deg$padj, na.rm=TRUE), "\n")

cat("\n=== Dataset loaded and validated successfully ===\n")


=== Dimensions ===
Rows: 23573  Cols: 8 

=== Column names ===
[1] "X"              "baseMean"       "log2FoldChange" "lfcSE"         
[5] "stat"           "pvalue"         "padj"           "X.1"           

=== First 3 rows ===
          X baseMean log2FoldChange     lfcSE     stat   pvalue     padj X.1
1 AT5G60250 373.9543       4.474915 0.3280360 13.64154 2.27e-42 5.34e-38 yes
2 AT3G27630 367.0555       5.769355 0.4284310 13.46624 2.47e-41 2.91e-37 yes
3 AT1G20750 101.5512       6.216640 0.4723034 13.16239 1.44e-39 1.14e-35 yes

=== Last 3 rows ===
              X  baseMean log2FoldChange      lfcSE         stat    pvalue
23571 AT5G53500 517.16162       1.96e-05 0.05362018  0.000365013 0.9997088
23572 AT2G31305  94.10068      -1.11e-05 0.05148909 -0.000215724 0.9998279
23573 AT3G01990  33.49981       1.53e-06 0.11626778  0.000013200 0.9999895
           padj X.1
23571 0.9997936  no
23572 0.9998703  no
23573 0.9999895  no

=== After renaming first column ===
[1] "gene_id"        "bas

Download AtGenExpress atlas from GEO

In [3]:

# ── Download AtGenExpress developmental atlas from GEO ─────────────────────────
# GSE5629 = seedlings and whole plants
# GSE5630 = leaves
# GSE5632 = flowers and pollen
# GSE5633 = shoots and stems
# GSE5634 = roots (separate series)
# GSE5631 = siliques/seeds

suppressPackageStartupMessages({
  library(GEOquery)
})

# Use a local cache directory
geodir <- "/workspace/geo_cache"
dir.create(geodir, showWarnings = FALSE, recursive = TRUE)

# Function to download and extract expression + sample metadata
get_geo_series <- function(gse_id) {
  cat("Downloading", gse_id, "...\n")
  gse <- getGEO(gse_id, destdir = geodir, getGPL = TRUE)
  # getGEO returns a list (one element per platform)
  if (is.list(gse)) gse <- gse[[1]]
  cat("  Class:", class(gse), "\n")
  cat("  Samples:", ncol(gse), "\n")
  cat("  Features:", nrow(gse), "\n")
  return(gse)
}

# Download each series
series_ids <- c("GSE5629", "GSE5630", "GSE5631", "GSE5632", "GSE5633", "GSE5634")
geo_data <- list()

for (sid in series_ids) {
  geo_data[[sid]] <- tryCatch(
    get_geo_series(sid),
    error = function(e) { cat("  ERROR for", sid, ":", conditionMessage(e), "\n"); NULL }
  )
}

cat("\n=== Download summary ===\n")
for (sid in names(geo_data)) {
  if (!is.null(geo_data[[sid]])) {
    cat(sid, ": OK (", ncol(geo_data[[sid]]), "samples,", nrow(geo_data[[sid]]), "features)\n")
  } else {
    cat(sid, ": FAILED\n")
  }
}


Found 1 file(s)

GSE5629_series_matrix.txt.gz

  Class: ExpressionSet 
  Samples: 24 
  Features: 22810 
Found 1 file(s)

GSE5630_series_matrix.txt.gz

Using locally cached version of GPL198 found here:
/workspace/geo_cache/GPL198.soft.gz 

  Class: ExpressionSet 
  Samples: 60 
  Features: 22810 
Found 1 file(s)

GSE5631_series_matrix.txt.gz

Using locally cached version of GPL198 found here:
/workspace/geo_cache/GPL198.soft.gz 

  Class: ExpressionSet 
  Samples: 21 
  Features: 22810 
Found 1 file(s)

GSE5632_series_matrix.txt.gz

Using locally cached version of GPL198 found here:
/workspace/geo_cache/GPL198.soft.gz 

  Class: ExpressionSet 
  Samples: 66 
  Features: 22810 
Found 1 file(s)

GSE5633_series_matrix.txt.gz

Using locally cached version of GPL198 found here:
/workspace/geo_cache/GPL198.soft.gz 

  Class: ExpressionSet 
  Samples: 42 
  Features: 22810 
Found 1 file(s)

GSE5634_series_matrix.txt.gz

Using locally cached version of GPL198 found here:
/workspace/geo_cache/

Extract expression and metadata from GEO series

In [5]:

# ── Extract expression matrices and sample metadata from all series ────────────
suppressPackageStartupMessages({
  library(Biobase)
})

# Combine all series into one expression matrix + sample metadata
all_expr <- list()
all_meta <- list()

for (sid in names(geo_data)) {
  gse <- geo_data[[sid]]
  if (is.null(gse)) next
  
  # Expression matrix (rows = probes, cols = samples)
  expr <- exprs(gse)
  cat(sid, "expression matrix:", nrow(expr), "x", ncol(expr), "\n")
  
  # Sample metadata from pData
  pdata <- pData(gse)
  cat("  pData columns:", ncol(pdata), "\n")
  
  # The key column is usually 'title' or 'source_name_ch1' or 'characteristics_ch1'
  # Let's look at what's available
  if (sid == "GSE5629") {
    cat("  Sample titles:\n")
    print(pdata$title)
    cat("\n  source_name_ch1:\n")
    if ("source_name_ch1" %in% colnames(pdata)) print(pdata$source_name_ch1)
  }
  
  all_expr[[sid]] <- expr
  all_meta[[sid]] <- pdata
}

cat("\n=== Total samples across all series ===\n")
total_samples <- sum(sapply(all_expr, ncol))
cat(total_samples, "samples\n")


GSE5629 expression matrix: 22810 x 24 
  pData columns: 35 
  Sample titles:
 [1] "ATGE_7_A2"  "ATGE_7_B2"  "ATGE_7_C2"  "ATGE_22_A"  "ATGE_22_B" 
 [6] "ATGE_22_C"  "ATGE_23_A"  "ATGE_23_B"  "ATGE_23_C"  "ATGE_24_A" 
[11] "ATGE_24_B"  "ATGE_24_C"  "ATGE_96_A"  "ATGE_96_B"  "ATGE_96_C" 
[16] "ATGE_97_A"  "ATGE_97_B"  "ATGE_97_C"  "ATGE_100_A" "ATGE_100_B"
[21] "ATGE_100_C" "ATGE_101_A" "ATGE_101_B" "ATGE_101_C"

  source_name_ch1:
 [1] "Col-0" "Col-0" "Col-0" "Col-0" "Col-0" "Col-0" "Col-0" "Col-0" "Col-0"
[10] "Col-0" "Col-0" "Col-0" "Col-0" "Col-0" "Col-0" "Col-0" "Col-0" "Col-0"
[19] "Col-0" "Col-0" "Col-0" "Col-0" "Col-0" "Col-0"
GSE5630 expression matrix: 22810 x 60 
  pData columns: 38 
GSE5631 expression matrix: 22810 x 21 
  pData columns: 36 
GSE5632 expression matrix: 22810 x 66 
  pData columns: 38 
GSE5633 expression matrix: 22810 x 42 
  pData columns: 38 
GSE5634 expression matrix: 22810 x 24 
  pData columns: 35 

=== Total samples across all series ===
237 samples


Examine sample metadata for tissue mapping

In [7]:

# ── Examine sample metadata to build tissue mapping ────────────────────────────
# AtGenExpress uses ATGE sample numbers. We need to map these to tissues.
# Let's look at the characteristics columns for each series

for (sid in names(geo_data)) {
  gse <- geo_data[[sid]]
  pdata <- pData(gse)
  cat("=== ", sid, " ===\n", sep="")
  cat("Titles:\n")
  print(pdata$title)
  
  # Look at characteristics columns
  char_cols <- grep("characteristics", colnames(pdata), value=TRUE)
  for (cc in char_cols) {
    vals <- unique(pdata[[cc]])
    if (length(vals) <= 10) {
      cat("  ", cc, ":\n")
      print(vals)
    }
  }
  
  # Also check description/molecule columns
  desc_cols <- grep("description|source|title|label", colnames(pdata), value=TRUE, ignore.case=TRUE)
  for (dc in desc_cols) {
    vals <- unique(pdata[[dc]])
    if (length(vals) <= 15) {
      cat("  ", dc, ":\n")
      print(vals)
    }
  }
  cat("\n")
}


=== GSE5629 ===
Titles:
 [1] "ATGE_7_A2"  "ATGE_7_B2"  "ATGE_7_C2"  "ATGE_22_A"  "ATGE_22_B" 
 [6] "ATGE_22_C"  "ATGE_23_A"  "ATGE_23_B"  "ATGE_23_C"  "ATGE_24_A" 
[11] "ATGE_24_B"  "ATGE_24_C"  "ATGE_96_A"  "ATGE_96_B"  "ATGE_96_C" 
[16] "ATGE_97_A"  "ATGE_97_B"  "ATGE_97_C"  "ATGE_100_A" "ATGE_100_B"
[21] "ATGE_100_C" "ATGE_101_A" "ATGE_101_B" "ATGE_101_C"
   characteristics_ch1 :
[1] "Tissue: seedling, green parts"                                                
[2] "Tissue: developmental drift; whole plant after transition, but before bolting"
   source_name_ch1 :
[1] "Col-0"
   label_ch1 :
[1] "biotin"
   description :
[1] "Photoperiod: continuous light"
   description.1 :
[1] "Relative humidity: 65%"         "Substrate: 0.5x GM"            
[3] "Substrate: 0.5x GM, 5% sucrose"
   description.2 :
[1] "Substrate: soil"   "Temperature: 20oC"
   description.3 :
[1] "Temperature: 23oC"                                                                                          
[2] "Tissu

Extract tissue labels from all samples

In [9]:

# ── Build tissue mapping for all samples ───────────────────────────────────────
# Extract tissue info from characteristics_ch1 columns for each series

extract_tissue <- function(pdata) {
  # Find all characteristics columns
  char_cols <- grep("characteristics_ch1", colnames(pdata), value=TRUE)
  tissue <- rep(NA_character_, nrow(pdata))
  
  for (cc in char_cols) {
    vals <- pdata[[cc]]
    # Look for "Tissue: ..." pattern
    tissue_matches <- grepl("Tissue:", vals)
    if (any(tissue_matches)) {
      extracted <- sub(".*Tissue:\\s*", "", vals)
      extracted <- sub("\\s*;.*", "", extracted)  # remove anything after ;
      extracted[!tissue_matches] <- NA
      # Fill in where we don't have tissue yet
      need <- is.na(tissue)
      tissue[need] <- extracted[need]
    }
  }
  return(tissue)
}

# Build combined sample metadata
all_samples <- data.frame(
  sample_id = character(),
  series = character(),
  tissue_raw = character(),
  stringsAsFactors = FALSE
)

for (sid in names(geo_data)) {
  gse <- geo_data[[sid]]
  pdata <- pData(gse)
  tissue <- extract_tissue(pdata)
  
  df <- data.frame(
    sample_id = pdata$title,
    series = sid,
    tissue_raw = tissue,
    stringsAsFactors = FALSE
  )
  all_samples <- rbind(all_samples, df)
}

cat("=== Total samples ===\n")
cat(nrow(all_samples), "\n")
cat("\n=== Samples with tissue annotation ===\n")
cat(sum(!is.na(all_samples$tissue_raw)), "out of", nrow(all_samples), "\n")

cat("\n=== Unique tissue labels ===\n")
tissue_table <- sort(table(all_samples$tissue_raw), decreasing=TRUE)
print(tissue_table)

cat("\n=== Samples missing tissue ===\n")
missing <- all_samples[is.na(all_samples$tissue_raw), ]
print(missing[, c("sample_id", "series")])


=== Total samples ===
237 

=== Samples with tissue annotation ===
237 out of 237 

=== Unique tissue labels ===

shoot apex, inflorescence (after bolting) 
                                       24 
               flower stage 12 equivalent 
                                       21 
                                     root 
                                       15 
                    seedling, green parts 
                                       15 
                      developmental drift 
                                        9 
           veg rosette (for phase change) 
                                        9 
                                    roots 
                                        6 
                        rosette leaf # 12 
                                        6 
                rosette leaf #4, 1cm long 
                                        6 
                         rosette leaf # 4 
                                        4 
                          

Build hierarchical tissue mapping

In [11]:

# ── Build hierarchical tissue mapping: broad_organ → sub_tissue ────────────────

# Map each raw tissue label to (broad_organ, sub_tissue)
build_tissue_map <- function(raw_tissue) {
  t <- tolower(trimws(raw_tissue))
  
  # ROOT
  if (grepl("root", t)) {
    return(list(broad = "Root", sub = "root"))
  }
  
  # SEEDLING
  if (grepl("seedling", t)) {
    return(list(broad = "Seedling", sub = "seedling_green_parts"))
  }
  if (grepl("developmental drift", t)) {
    return(list(broad = "Seedling", sub = "whole_plant_pre_bolting"))
  }
  
  # LEAF / SHOOT
  if (grepl("cotyledon", t)) {
    return(list(broad = "Leaf_Shoot", sub = "cotyledon"))
  }
  if (grepl("hypocotyl", t)) {
    return(list(broad = "Leaf_Shoot", sub = "hypocotyl"))
  }
  if (grepl("senescing", t)) {
    return(list(broad = "Leaf_Shoot", sub = "senescing_leaf"))
  }
  if (grepl("rosette leaf", t)) {
    return(list(broad = "Leaf_Shoot", sub = "rosette_leaf"))
  }
  if (grepl("^leaf", t) || grepl("leaves 1", t)) {
    return(list(broad = "Leaf_Shoot", sub = "rosette_leaf"))
  }
  if (grepl("cauline", t)) {
    return(list(broad = "Leaf_Shoot", sub = "cauline_leaf"))
  }
  if (grepl("veg rosette", t)) {
    return(list(broad = "Leaf_Shoot", sub = "rosette_vegetative"))
  }
  if (grepl("shoot apex, vegetative", t) && !grepl("inflorescence", t)) {
    if (grepl("young leaves", t)) return(list(broad = "Leaf_Shoot", sub = "shoot_apex_vegetative_with_leaves"))
    return(list(broad = "Leaf_Shoot", sub = "shoot_apex_vegetative"))
  }
  if (grepl("shoot apex, transition", t)) {
    return(list(broad = "Leaf_Shoot", sub = "shoot_apex_transition"))
  }
  if (grepl("stem", t) || grepl("1st node", t)) {
    return(list(broad = "Leaf_Shoot", sub = "stem"))
  }
  
  # FLOWER
  if (grepl("pollen", t)) {
    return(list(broad = "Flower", sub = "mature_pollen"))
  }
  if (grepl("carpel", t)) {
    return(list(broad = "Flower", sub = "carpel"))
  }
  if (grepl("petal", t)) {
    return(list(broad = "Flower", sub = "petal"))
  }
  if (grepl("sepal", t)) {
    return(list(broad = "Flower", sub = "sepal"))
  }
  if (grepl("stamen", t)) {
    return(list(broad = "Flower", sub = "stamen"))
  }
  if (grepl("pedicel", t)) {
    return(list(broad = "Flower", sub = "pedicel"))
  }
  if (grepl("flower stage 12 equivalent", t) || grepl("^flower$", t)) {
    return(list(broad = "Flower", sub = "flower_whole"))
  }
  if (grepl("flowers stage 9", t)) {
    return(list(broad = "Flower", sub = "flower_stage_9"))
  }
  if (grepl("flowers stage 10", t)) {
    return(list(broad = "Flower", sub = "flower_stage_10_11"))
  }
  if (grepl("flowers stage 12", t)) {
    return(list(broad = "Flower", sub = "flower_stage_12"))
  }
  if (grepl("flowers stage 15", t)) {
    return(list(broad = "Flower", sub = "flower_stage_15"))
  }
  if (grepl("shoot apex, inflorescence", t)) {
    return(list(broad = "Flower", sub = "inflorescence_apex"))
  }
  
  # SEED / SILIQUE
  if (grepl("silique", t)) {
    if (grepl("stage 3", t)) return(list(broad = "Seed_Silique", sub = "silique_stage_3"))
    if (grepl("stage 4", t)) return(list(broad = "Seed_Silique", sub = "silique_stage_4"))
    if (grepl("stage 5", t)) return(list(broad = "Seed_Silique", sub = "silique_stage_5"))
    return(list(broad = "Seed_Silique", sub = "silique"))
  }
  if (grepl("seeds?", t)) {
    if (grepl("stage 6", t)) return(list(broad = "Seed_Silique", sub = "seed_stage_6"))
    if (grepl("stage 7", t)) return(list(broad = "Seed_Silique", sub = "seed_stage_7"))
    if (grepl("stage 8", t)) return(list(broad = "Seed_Silique", sub = "seed_stage_8"))
    if (grepl("stage 9", t)) return(list(broad = "Seed_Silique", sub = "seed_stage_9"))
    if (grepl("stage 10", t)) return(list(broad = "Seed_Silique", sub = "seed_stage_10"))
    return(list(broad = "Seed_Silique", sub = "seed"))
  }
  
  # Fallback
  return(list(broad = "Other", sub = "unknown"))
}

# Apply mapping
broad_organs <- character(nrow(all_samples))
sub_tissues <- character(nrow(all_samples))

for (i in seq_len(nrow(all_samples))) {
  mapping <- build_tissue_map(all_samples$tissue_raw[i])
  broad_organs[i] <- mapping$broad
  sub_tissues[i] <- mapping$sub
}

all_samples$broad_organ <- broad_organs
all_samples$sub_tissue <- sub_tissues

cat("=== Broad organ distribution ===\n")
print(sort(table(all_samples$broad_organ), decreasing=TRUE))

cat("\n=== Sub-tissue distribution ===\n")
print(sort(table(all_samples$sub_tissue), decreasing=TRUE))

cat("\n=== Hierarchical view ===\n")
for (bo in unique(all_samples$broad_organ)) {
  subs <- unique(all_samples$sub_tissue[all_samples$broad_organ == bo])
  cat(bo, ":\n")
  for (s in subs) {
    n <- sum(all_samples$broad_organ == bo & all_samples$sub_tissue == s)
    cat("  ", s, " (", n, " samples)\n", sep="")
  }
}


=== Broad organ distribution ===

      Flower   Leaf_Shoot Seed_Silique     Seedling         Root 
          90           78           24           24           21 

=== Sub-tissue distribution ===

                     rosette_leaf                      flower_whole 
                               42                                24 
               inflorescence_apex                              root 
                               24                                21 
             seedling_green_parts                           silique 
                               15                                15 
               rosette_vegetative           whole_plant_pre_bolting 
                                9                                 9 
                           carpel                             petal 
                                6                                 6 
                            sepal                            stamen 
                                6        

Fix tissue mapping for seed stages

In [13]:

# ── Fix: check "seeds" BEFORE "silique" in the mapping ─────────────────────────
build_tissue_map_fixed <- function(raw_tissue) {
  t <- tolower(trimws(raw_tissue))
  
  # ROOT
  if (grepl("root", t)) {
    return(list(broad = "Root", sub = "root"))
  }
  
  # SEEDLING
  if (grepl("seedling", t)) {
    return(list(broad = "Seedling", sub = "seedling_green_parts"))
  }
  if (grepl("developmental drift", t)) {
    return(list(broad = "Seedling", sub = "whole_plant_pre_bolting"))
  }
  
  # LEAF / SHOOT
  if (grepl("cotyledon", t)) {
    return(list(broad = "Leaf_Shoot", sub = "cotyledon"))
  }
  if (grepl("hypocotyl", t)) {
    return(list(broad = "Leaf_Shoot", sub = "hypocotyl"))
  }
  if (grepl("senescing", t)) {
    return(list(broad = "Leaf_Shoot", sub = "senescing_leaf"))
  }
  if (grepl("rosette leaf", t)) {
    return(list(broad = "Leaf_Shoot", sub = "rosette_leaf"))
  }
  if (grepl("^leaf", t) || grepl("leaves 1", t)) {
    return(list(broad = "Leaf_Shoot", sub = "rosette_leaf"))
  }
  if (grepl("cauline", t)) {
    return(list(broad = "Leaf_Shoot", sub = "cauline_leaf"))
  }
  if (grepl("veg rosette", t)) {
    return(list(broad = "Leaf_Shoot", sub = "rosette_vegetative"))
  }
  if (grepl("shoot apex, vegetative", t) && !grepl("inflorescence", t)) {
    if (grepl("young leaves", t)) return(list(broad = "Leaf_Shoot", sub = "shoot_apex_vegetative_with_leaves"))
    return(list(broad = "Leaf_Shoot", sub = "shoot_apex_vegetative"))
  }
  if (grepl("shoot apex, transition", t)) {
    return(list(broad = "Leaf_Shoot", sub = "shoot_apex_transition"))
  }
  if (grepl("stem", t) || grepl("1st node", t)) {
    return(list(broad = "Leaf_Shoot", sub = "stem"))
  }
  
  # FLOWER
  if (grepl("pollen", t)) {
    return(list(broad = "Flower", sub = "mature_pollen"))
  }
  if (grepl("carpel", t)) {
    return(list(broad = "Flower", sub = "carpel"))
  }
  if (grepl("petal", t)) {
    return(list(broad = "Flower", sub = "petal"))
  }
  if (grepl("sepal", t)) {
    return(list(broad = "Flower", sub = "sepal"))
  }
  if (grepl("stamen", t)) {
    return(list(broad = "Flower", sub = "stamen"))
  }
  if (grepl("pedicel", t)) {
    return(list(broad = "Flower", sub = "pedicel"))
  }
  if (grepl("flower stage 12 equivalent", t) || grepl("^flower$", t)) {
    return(list(broad = "Flower", sub = "flower_whole"))
  }
  if (grepl("flowers stage 9", t)) {
    return(list(broad = "Flower", sub = "flower_stage_9"))
  }
  if (grepl("flowers stage 10", t)) {
    return(list(broad = "Flower", sub = "flower_stage_10_11"))
  }
  if (grepl("flowers stage 12", t)) {
    return(list(broad = "Flower", sub = "flower_stage_12"))
  }
  if (grepl("flowers stage 15", t)) {
    return(list(broad = "Flower", sub = "flower_stage_15"))
  }
  if (grepl("shoot apex, inflorescence", t)) {
    return(list(broad = "Flower", sub = "inflorescence_apex"))
  }
  
  # SEED / SILIQUE — check "seeds, stage" BEFORE "silique" 
  # because "seeds, stage X, w/o siliques" contains the word "siliques"
  if (grepl("^seeds?,", t)) {
    if (grepl("stage 6", t)) return(list(broad = "Seed_Silique", sub = "seed_stage_6"))
    if (grepl("stage 7", t)) return(list(broad = "Seed_Silique", sub = "seed_stage_7"))
    if (grepl("stage 8", t)) return(list(broad = "Seed_Silique", sub = "seed_stage_8"))
    if (grepl("stage 9", t)) return(list(broad = "Seed_Silique", sub = "seed_stage_9"))
    if (grepl("stage 10", t)) return(list(broad = "Seed_Silique", sub = "seed_stage_10"))
    return(list(broad = "Seed_Silique", sub = "seed"))
  }
  if (grepl("silique", t)) {
    if (grepl("stage 3", t)) return(list(broad = "Seed_Silique", sub = "silique_stage_3"))
    if (grepl("stage 4", t)) return(list(broad = "Seed_Silique", sub = "silique_stage_4"))
    if (grepl("stage 5", t)) return(list(broad = "Seed_Silique", sub = "silique_stage_5"))
    return(list(broad = "Seed_Silique", sub = "silique"))
  }
  
  # Fallback
  return(list(broad = "Other", sub = "unknown"))
}

# Re-apply mapping
for (i in seq_len(nrow(all_samples))) {
  mapping <- build_tissue_map_fixed(all_samples$tissue_raw[i])
  all_samples$broad_organ[i] <- mapping$broad
  all_samples$sub_tissue[i] <- mapping$sub
}

cat("=== Fixed broad organ distribution ===\n")
print(sort(table(all_samples$broad_organ), decreasing=TRUE))

cat("\n=== Fixed sub-tissue distribution ===\n")
print(sort(table(all_samples$sub_tissue), decreasing=TRUE))

cat("\n=== Fixed hierarchical view ===\n")
for (bo in sort(unique(all_samples$broad_organ))) {
  subs <- sort(unique(all_samples$sub_tissue[all_samples$broad_organ == bo]))
  cat(bo, ":\n")
  for (s in subs) {
    n <- sum(all_samples$broad_organ == bo & all_samples$sub_tissue == s)
    cat("  ", s, " (", n, " samples)\n", sep="")
  }
}

# Check for any "Other/unknown"
cat("\n=== Unmapped samples ===\n")
other <- all_samples[all_samples$broad_organ == "Other", ]
print(other[, c("sample_id", "series", "tissue_raw")])


=== Fixed broad organ distribution ===

      Flower   Leaf_Shoot Seed_Silique     Seedling         Root 
          90           78           24           24           21 

=== Fixed sub-tissue distribution ===

                     rosette_leaf                      flower_whole 
                               42                                24 
               inflorescence_apex                              root 
                               24                                21 
             seedling_green_parts                rosette_vegetative 
                               15                                 9 
          whole_plant_pre_bolting                            carpel 
                                9                                 6 
                            petal                             sepal 
                                6                                 6 
                           stamen                              stem 
                             

Combine expression and map probes to AGI IDs

In [15]:

# ── Combine all expression matrices and map probes to AGI IDs ──────────────────
suppressPackageStartupMessages({
  library(ath1121501.db)
  library(AnnotationDbi)
})

# Combine expression matrices from all series
# Each series has the same 22810 probes (ATH1 platform)
combined_expr <- NULL
sample_order <- c()  # track which sample belongs to which series

for (sid in names(geo_data)) {
  gse <- geo_data[[sid]]
  expr <- exprs(gse)
  if (is.null(combined_expr)) {
    combined_expr <- expr
  } else {
    # cbind — probes should be in the same order (same platform)
    combined_expr <- cbind(combined_expr, expr)
  }
  sample_order <- c(sample_order, rep(sid, ncol(expr)))
}

cat("=== Combined expression matrix ===\n")
cat("Dimensions:", nrow(combined_expr), "probes x", ncol(combined_expr), "samples\n")

# Verify sample order matches all_samples
# all_samples was built in the same order as geo_data, so they should match
cat("Sample order matches:", all(sample_order == all_samples$series), "\n")

# Map column names of combined_expr to all_samples
colnames(combined_expr) <- all_samples$sample_id

# ── Map ATH1 probe IDs to AGI gene IDs ─────────────────────────────────────────
# Get the probe-to-gene mapping from ath1121501.db
probe_ids <- rownames(combined_expr)
cat("\n=== Probe IDs ===\n")
cat("Total probes:", length(probe_ids), "\n")
cat("Examples:", paste(head(probe_ids, 5), collapse=", "), "\n")

# Get AGI mappings
agi_map <- AnnotationDbi::select(ath1121501.db, 
                                  keys = probe_ids, 
                                  columns = c("PROBEID", "ENTREZID", "SYMBOL", "GENENAME"),
                                  keytype = "PROBEID")

# Also get the Arabidopsis locus IDs (AGI)
# ath1121501.db maps to ENTREZID; we need AGI. Let's check available columns
cat("\n=== Available columns in ath1121501.db ===\n")
print(columns(ath1121501.db))

# Try to get AGI directly
agi_map2 <- AnnotationDbi::select(ath1121501.db,
                                   keys = probe_ids,
                                   columns = c("PROBEID", "ENTREZID", "SYMBOL"),
                                   keytype = "PROBEID")

cat("\n=== Probe mapping summary ===\n")
cat("Probes with ENTREZID:", sum(!is.na(agi_map2$ENTREZID)), "\n")
cat("Probes with SYMBOL:", sum(!is.na(agi_map2$SYMBOL)), "\n")

# The SYMBOL in ath1121501.db often contains the AGI locus ID
cat("\n=== SYMBOL examples ===\n")
print(head(agi_map2$SYMBOL[!is.na(agi_map2$SYMBOL)], 20))


=== Combined expression matrix ===
Dimensions: 22810 probes x 237 samples
Sample order matches: TRUE 

=== Probe IDs ===
Total probes: 22810 
Examples: 244901_at, 244902_at, 244903_at, 244904_at, 244905_at 
'select()' returned 1:many mapping between keys and columns


=== Available columns in ath1121501.db ===
 [1] "ARACYC"       "ARACYCENZYME" "ENTREZID"     "ENZYME"       "EVIDENCE"    
 [6] "EVIDENCEALL"  "GENENAME"     "GO"           "GOALL"        "ONTOLOGY"    
[11] "ONTOLOGYALL"  "PATH"         "PMID"         "PROBEID"      "REFSEQ"      
[16] "SYMBOL"       "TAIR"        
'select()' returned 1:many mapping between keys and columns


=== Probe mapping summary ===
Probes with ENTREZID: 32641 
Probes with SYMBOL: 24493 

=== SYMBOL examples ===
 [1] "ORF25"   "NAD4L"   "ORF149"  "ORF275"  "ORF122C" "ORF240A" "ORF120" 
 [8] "ORF107D" "ORF100A" "ORF119"  "ORF170"  "CCB382"  "ORF121B" "ORF107E"
[15] "ORF158"  "ORF187"  "ORF184"  "ORF106D" "CCB203"  "NAD3"   


Get AGI locus IDs from TAIR column

In [17]:

# ── Get AGI locus IDs from the TAIR column ─────────────────────────────────────
agi_map_tair <- AnnotationDbi::select(ath1121501.db,
                                       keys = probe_ids,
                                       columns = c("PROBEID", "TAIR", "SYMBOL", "GENENAME"),
                                       keytype = "PROBEID")

cat("=== TAIR column examples ===\n")
print(head(agi_map_tair$TAIR[!is.na(agi_map_tair$TAIR)], 20))

cat("\n=== TAIR mapping summary ===\n")
cat("Total probes:", nrow(agi_map_tair), "\n")
cat("Probes with TAIR:", sum(!is.na(agi_map_tair$TAIR)), "\n")
cat("Unique TAIR IDs:", length(unique(agi_map_tair$TAIR[!is.na(agi_map_tair$TAIR)])), "\n")

# Check if TAIR values look like AGI IDs (AT{1-5}Gnnnnn)
tair_vals <- agi_map_tair$TAIR[!is.na(agi_map_tair$TAIR)]
agi_like <- grepl("^AT[1-5]G[0-9]{5}", tair_vals)
cat("TAIR values matching AGI pattern:", sum(agi_like), "out of", length(tair_vals), "\n")

# Show some non-AGI TAIR values
non_agi <- tair_vals[!agi_like]
cat("Non-AGI TAIR examples:", paste(head(non_agi, 20), collapse=", "), "\n")


'select()' returned 1:many mapping between keys and columns

=== TAIR column examples ===
 [1] "ATMG00640" "ATMG00650" "ATMG00660" "ATMG00670" "ATMG00680" "ATMG00690"
 [7] "ATMG00710" "ATMG00720" "ATMG00740" "ATMG00750" "ATMG00820" "ATMG00830"
[13] "ATMG00840" "ATMG00850" "ATMG00860" "ATMG00880" "ATMG00870" "ATMG00890"
[19] "AT2G07768" "ATMG00960"

=== TAIR mapping summary ===
Total probes: 34760 
Probes with TAIR: 33143 
Unique TAIR IDs: 21891 
TAIR values matching AGI pattern: 32890 out of 33143 
Non-AGI TAIR examples: ATMG00640, ATMG00650, ATMG00660, ATMG00670, ATMG00680, ATMG00690, ATMG00710, ATMG00720, ATMG00740, ATMG00750, ATMG00820, ATMG00830, ATMG00840, ATMG00850, ATMG00860, ATMG00880, ATMG00870, ATMG00890, ATMG00960, ATMG00990 


Collapse probes to genes and build tissue expression matrix

In [19]:

# ── Collapse probes to genes and build tissue-level expression matrix ──────────

# Keep only nuclear AGI IDs (AT{1-5}Gnnnnn), drop organellar (ATCG/ATMG)
agi_map_clean <- agi_map_tair[!is.na(agi_map_tair$TAIR), ]
agi_map_clean <- agi_map_clean[grepl("^AT[1-5]G[0-9]{5}", agi_map_clean$TAIR), ]

# Some probes map to multiple genes; keep first mapping per probe
agi_map_clean <- agi_map_clean[!duplicated(agi_map_clean$PROBEID), ]

cat("=== Clean probe-to-gene mapping ===\n")
cat("Probes with nuclear AGI:", nrow(agi_map_clean), "\n")
cat("Unique genes:", length(unique(agi_map_clean$TAIR)), "\n")

# Subset expression matrix to mapped probes
mapped_probes <- agi_map_clean$PROBEID
expr_mapped <- combined_expr[mapped_probes, , drop = FALSE]
rownames(expr_mapped) <- agi_map_clean$TAIR[match(mapped_probes, agi_map_clean$PROBEID)]

# If multiple probes map to the same gene, take the one with highest mean expression
gene_ids <- rownames(expr_mapped)
dup_genes <- unique(gene_ids[duplicated(gene_ids)])
cat("\n=== Genes with multiple probes ===\n")
cat("Duplicated genes:", length(dup_genes), "\n")

if (length(dup_genes) > 0) {
  # For each duplicated gene, keep the probe with highest mean expression
  keep_rows <- seq_len(nrow(expr_mapped))
  mean_expr <- rowMeans(expr_mapped, na.rm = TRUE)
  
  for (g in dup_genes) {
    idx <- which(gene_ids == g)
    best <- idx[which.max(mean_expr[idx])]
    keep_rows <- setdiff(keep_rows, setdiff(idx, best))
  }
  
  expr_mapped <- expr_mapped[keep_rows, , drop = FALSE]
  cat("After deduplication:", nrow(expr_mapped), "genes\n")
}

cat("\n=== Final gene-level expression matrix ===\n")
cat("Dimensions:", nrow(expr_mapped), "genes x", ncol(expr_mapped), "samples\n")
cat("Gene ID examples:", paste(head(rownames(expr_mapped), 5), collapse=", "), "\n")

# ── Build tissue-level expression matrix (mean per sub_tissue) ─────────────────
# For each sub_tissue, average expression across all samples of that tissue
sub_tissues <- unique(all_samples$sub_tissue)
sub_tissue_expr <- matrix(NA, nrow = nrow(expr_mapped), ncol = length(sub_tissues))
rownames(sub_tissue_expr) <- rownames(expr_mapped)
colnames(sub_tissue_expr) <- sub_tissues

for (st in sub_tissues) {
  samples_st <- all_samples$sample_id[all_samples$sub_tissue == st]
  # Make sure these columns exist
  samples_st <- intersect(samples_st, colnames(expr_mapped))
  if (length(samples_st) > 0) {
    sub_tissue_expr[, st] <- rowMeans(expr_mapped[, samples_st, drop = FALSE], na.rm = TRUE)
  }
}

cat("\n=== Sub-tissue expression matrix ===\n")
cat("Dimensions:", nrow(sub_tissue_expr), "genes x", ncol(sub_tissue_expr), "sub-tissues\n")

# Also build broad-organ level (mean per broad_organ)
broad_organs <- unique(all_samples$broad_organ)
broad_organ_expr <- matrix(NA, nrow = nrow(expr_mapped), ncol = length(broad_organs))
rownames(broad_organ_expr) <- rownames(expr_mapped)
colnames(broad_organ_expr) <- broad_organs

for (bo in broad_organs) {
  samples_bo <- all_samples$sample_id[all_samples$broad_organ == bo]
  samples_bo <- intersect(samples_bo, colnames(expr_mapped))
  if (length(samples_bo) > 0) {
    broad_organ_expr[, bo] <- rowMeans(expr_mapped[, samples_bo, drop = FALSE], na.rm = TRUE)
  }
}

cat("\n=== Broad-organ expression matrix ===\n")
cat("Dimensions:", nrow(broad_organ_expr), "genes x", ncol(broad_organ_expr), "broad organs\n")
print(head(broad_organ_expr[, 1:5], 3))

# Save the atlas data for reuse
saveRDS(list(
  sub_tissue_expr = sub_tissue_expr,
  broad_organ_expr = broad_organ_expr,
  all_samples = all_samples,
  agi_map_clean = agi_map_clean
), "/workspace/atgenexpress_atlas.rds")

cat("\n=== Atlas saved to /workspace/atgenexpress_atlas.rds ===\n")


=== Clean probe-to-gene mapping ===
Probes with nuclear AGI: 21045 
Unique genes: 20833 

=== Genes with multiple probes ===
Duplicated genes: 196 
After deduplication: 20833 genes

=== Final gene-level expression matrix ===
Dimensions: 20833 genes x 237 samples
Gene ID examples: AT2G07768, AT2G07751, AT2G07674, AT2G07776, AT2G07732 

=== Sub-tissue expression matrix ===
Dimensions: 20833 genes x 33 sub-tissues

=== Broad-organ expression matrix ===
Dimensions: 20833 genes x 5 broad organs
           Seedling Leaf_Shoot      Root    Flower Seed_Silique
AT2G07768  10.02431   9.630731  14.64738  14.94571     20.86445
AT2G07751 124.18204 124.371355 149.83692 173.57268    157.99836
AT2G07674  18.04145  20.036461  13.60768  26.28298     13.35861

=== Atlas saved to /workspace/atgenexpress_atlas.rds ===


Compute Tau index and assign DEGs to tissues

In [21]:

# ── Compute Tau tissue-specificity index and assign DEGs to tissues ────────────
# Tau = Σ(1 − xᵢ/max(x)) / (n − 1)
# where xᵢ = expression in tissue i, n = number of tissues
# Range: 0 (ubiquitous) to 1 (perfectly tissue-specific)

# Load atlas data
atlas <- readRDS("/workspace/atgenexpress_atlas.rds")
sub_tissue_expr <- atlas$sub_tissue_expr
broad_organ_expr <- atlas$broad_organ_expr

# ── Tau at sub-tissue level ────────────────────────────────────────────────────
compute_tau <- function(expr_matrix) {
  # expr_matrix: genes (rows) x tissues (cols)
  # Returns vector of Tau scores per gene
  
  # Handle any negative or zero values — shift to positive
  expr_matrix[expr_matrix < 0] <- 0
  
  # For each gene, compute Tau
  tau <- apply(expr_matrix, 1, function(x) {
    mx <- max(x, na.rm = TRUE)
    if (mx == 0 || is.na(mx)) return(NA)
    n <- sum(!is.na(x))
    if (n <= 1) return(NA)
    sum(1 - (x / mx), na.rm = TRUE) / (n - 1)
  })
  
  return(tau)
}

# Compute Tau at both levels
tau_subtissue <- compute_tau(sub_tissue_expr)
tau_broad <- compute_tau(broad_organ_expr)

cat("=== Tau index summary (sub-tissue level) ===\n")
cat("Genes with valid Tau:", sum(!is.na(tau_subtissue)), "\n")
cat("Mean:", round(mean(tau_subtissue, na.rm=TRUE), 4), "\n")
cat("Median:", round(median(tau_subtissue, na.rm=TRUE), 4), "\n")
cat("Min:", round(min(tau_subtissue, na.rm=TRUE), 4), "\n")
cat("Max:", round(max(tau_subtissue, na.rm=TRUE), 4), "\n")
cat("Quantiles:\n")
print(round(quantile(tau_subtissue, na.rm=TRUE), 4))

cat("\n=== Tau index summary (broad-organ level) ===\n")
cat("Genes with valid Tau:", sum(!is.na(tau_broad)), "\n")
cat("Mean:", round(mean(tau_broad, na.rm=TRUE), 4), "\n")
cat("Median:", round(median(tau_broad, na.rm=TRUE), 4), "\n")
cat("Quantiles:\n")
print(round(quantile(tau_broad, na.rm=TRUE), 4))

# ── Assign each gene to its predominant tissue ─────────────────────────────────
# For tissue-specific genes (Tau >= threshold), assign to tissue with highest expression
# For constitutive genes (Tau < threshold), label as "constitutive"

TAU_THRESHOLD <- 0.6  # standard threshold from literature

# Sub-tissue assignment
assign_tissue <- function(expr_matrix, tau_values, threshold) {
  # For each gene, find the tissue with max expression
  predominant_tissue <- colnames(expr_matrix)[apply(expr_matrix, 1, function(x) {
    if (all(is.na(x)) || max(x, na.rm=TRUE) == 0) return(NA)
    which.max(x)
  })]
  
  # Classification: tissue-specific vs constitutive
  specificity <- ifelse(tau_values >= threshold, "tissue_specific", "constitutive")
  specificity[is.na(tau_values)] <- "unannotated"
  
  return(list(predominant_tissue = predominant_tissue, specificity = specificity))
}

# Sub-tissue level
sub_assignment <- assign_tissue(sub_tissue_expr, tau_subtissue, TAU_THRESHOLD)

# Broad-organ level  
broad_assignment <- assign_tissue(broad_organ_expr, tau_broad, TAU_THRESHOLD)

# Build the gene annotation table
gene_annotation <- data.frame(
  gene_id = rownames(sub_tissue_expr),
  tau_subtissue = tau_subtissue[rownames(sub_tissue_expr)],
  tau_broad = tau_broad[rownames(sub_tissue_expr)],
  predominant_subtissue = sub_assignment$predominant_tissue,
  predominant_broad_organ = broad_assignment$predominant_tissue,
  specificity = sub_assignment$specificity,
  stringsAsFactors = FALSE
)

# For broad organ, map sub-tissue to broad organ
sub_to_broad <- unique(all_samples[, c("sub_tissue", "broad_organ")])
names(sub_to_broad) <- c("sub_tissue", "broad_organ")
gene_annotation$predominant_broad_organ <- sub_to_broad$broad_organ[
  match(gene_annotation$predominant_subtissue, sub_to_broad$sub_tissue)]

cat("\n=== Specificity classification ===\n")
print(table(gene_annotation$specificity))

cat("\n=== Predominant broad organ distribution ===\n")
print(table(gene_annotation$predominant_broad_organ, useNA = "ifany"))

cat("\n=== Predominant sub-tissue distribution (top 15) ===\n")
print(sort(table(gene_annotation$predominant_subtissue, useNA = "ifany"), decreasing=TRUE)[1:15])

# ── Merge with DEG data ────────────────────────────────────────────────────────
# Load the DEG data (re-read to get clean version)
deg <- read.csv("/mnt/user-uploads/DEG_OSD498_510_radiation_effect.csv", 
                header = TRUE, stringsAsFactors = FALSE)
colnames(deg)[1] <- "gene_id"
colnames(deg)[ncol(deg)] <- "deg_flag"

# Merge
deg_annotated <- merge(deg, gene_annotation, by = "gene_id", all.x = TRUE)

# Flag genes not in atlas as "unannotated"
deg_annotated$specificity[is.na(deg_annotated$specificity)] <- "unannotated"
deg_annotated$predominant_broad_organ[is.na(deg_annotated$predominant_broad_organ)] <- "unannotated"
deg_annotated$predominant_subtissue[is.na(deg_annotated$predominant_subtissue)] <- "unannotated"

# Add regulation direction
deg_annotated$regulation <- ifelse(deg_annotated$deg_flag == "yes",
                                    ifelse(deg_annotated$log2FoldChange > 0, "up", "down"),
                                    "non_DEG")

cat("\n=== Annotated DEG dataset ===\n")
cat("Total genes:", nrow(deg_annotated), "\n")
cat("DEGs:", sum(deg_annotated$deg_flag == "yes"), "\n")

cat("\n=== DEG specificity breakdown ===\n")
deg_only <- deg_annotated[deg_annotated$deg_flag == "yes", ]
print(table(deg_only$specificity))

cat("\n=== DEG predominant broad organ ===\n")
print(table(deg_only$predominant_broad_organ, useNA = "ifany"))

cat("\n=== DEG predominant sub-tissue (top 15) ===\n")
print(sort(table(deg_only$predominant_subtissue, useNA = "ifany"), decreasing=TRUE)[1:15])

cat("\n=== DEG regulation by broad organ ===\n")
print(table(deg_only$predominant_broad_organ, deg_only$regulation))

# Save
saveRDS(deg_annotated, "/workspace/deg_annotated.rds")
cat("\n=== Saved annotated DEG data ===\n")


=== Tau index summary (sub-tissue level) ===
Genes with valid Tau: 20833 
Mean: 0.6622 
Median: 0.6611 
Min: 0.1265 
Max: 0.9983 
Quantiles:
    0%    25%    50%    75%   100% 
0.1265 0.5192 0.6611 0.8105 0.9983 

=== Tau index summary (broad-organ level) ===
Genes with valid Tau: 20833 
Mean: 0.4597 
Median: 0.4163 
Quantiles:
    0%    25%    50%    75%   100% 
0.0087 0.2702 0.4163 0.6140 0.9998 

=== Specificity classification ===

   constitutive tissue_specific 
           8061           12772 

=== Predominant broad organ distribution ===

      Flower   Leaf_Shoot         Root Seed_Silique     Seedling 
        7334         6965         1520         4773          241 

=== Predominant sub-tissue distribution (top 15) ===

        mature_pollen         seed_stage_10        senescing_leaf 
                 3580                  1659                  1616 
                 root shoot_apex_transition    inflorescence_apex 
                 1520                  1500                 

Export hierarchical tissue-specific CSV files

In [23]:

# ── Export tissue-specific CSV files in hierarchical structure ─────────────────
deg_annotated <- readRDS("/workspace/deg_annotated.rds")

# Base output directory
out_base <- "/mnt/results/tissue_specific_degs"
dir.create(out_base, showWarnings = FALSE, recursive = TRUE)

# Filter to DEGs only
degs_only <- deg_annotated[deg_annotated$deg_flag == "yes", ]

# Reorder columns for clarity
col_order <- c("gene_id", "baseMean", "log2FoldChange", "lfcSE", "stat", 
               "pvalue", "padj", "deg_flag", "regulation",
               "tau_subtissue", "tau_broad", "specificity",
               "predominant_broad_organ", "predominant_subtissue")
degs_only <- degs_only[, col_order]

cat("=== Exporting tissue-specific DEG files ===\n")
cat("Total DEGs to distribute:", nrow(degs_only), "\n\n")

# ── 1. Export per broad-organ / sub-tissue CSVs ────────────────────────────────
# Only for tissue-specific DEGs (skip constitutive and unannotated for tissue folders)
tissue_specific_degs <- degs_only[degs_only$specificity == "tissue_specific", ]

exported_files <- c()

for (bo in sort(unique(tissue_specific_degs$predominant_broad_organ))) {
  # Create broad organ directory
  organ_dir <- file.path(out_base, bo)
  dir.create(organ_dir, showWarnings = FALSE, recursive = TRUE)
  
  # Export all DEGs for this broad organ
  organ_degs <- tissue_specific_degs[tissue_specific_degs$predominant_broad_organ == bo, ]
  organ_file <- file.path(organ_dir, paste0(bo, "_all_DEGs.csv"))
  write.csv(organ_degs, organ_file, row.names = FALSE)
  exported_files <- c(exported_files, organ_file)
  cat(bo, ": ", nrow(organ_degs), " DEGs -> ", basename(organ_file), "\n", sep="")
  
  # Export per sub-tissue
  for (st in sort(unique(organ_degs$predominant_subtissue))) {
    st_degs <- organ_degs[organ_degs$predominant_subtissue == st, ]
    st_file <- file.path(organ_dir, paste0(st, "_DEGs.csv"))
    write.csv(st_degs, st_file, row.names = FALSE)
    exported_files <- c(exported_files, st_file)
    cat("  ", st, ": ", nrow(st_degs), " DEGs\n", sep="")
  }
}

# ── 2. Export constitutive DEGs ───────────────────────────────────────────────
constitutive_degs <- degs_only[degs_only$specificity == "constitutive", ]
const_file <- file.path(out_base, "constitutive_DEGs.csv")
write.csv(constitutive_degs, const_file, row.names = FALSE)
exported_files <- c(exported_files, const_file)
cat("\nConstitutive: ", nrow(constitutive_degs), " DEGs -> constitutive_DEGs.csv\n", sep="")

# ── 3. Export unannotated DEGs ─────────────────────────────────────────────────
unannotated_degs <- degs_only[degs_only$specificity == "unannotated", ]
unannot_file <- file.path(out_base, "unannotated_DEGs.csv")
write.csv(unannotated_degs, unannot_file, row.names = FALSE)
exported_files <- c(exported_files, unannot_file)
cat("Unannotated: ", nrow(unannotated_degs), " DEGs -> unannotated_DEGs.csv\n", sep="")

# ── 4. Export master annotated file (all genes, not just DEGs) ─────────────────
master_file <- file.path(out_base, "all_degs_with_tissue_annotation.csv")
write.csv(deg_annotated[, col_order], master_file, row.names = FALSE)
exported_files <- c(exported_files, master_file)
cat("\nMaster file (all genes): ", nrow(deg_annotated), " genes -> all_degs_with_tissue_annotation.csv\n", sep="")

cat("\n=== Total files exported:", length(exported_files), "===\n")
cat("\n=== File listing ===\n")
for (f in sort(exported_files)) {
  cat(gsub(out_base, "tissue_specific_degs", f), "\n")
}


=== Exporting tissue-specific DEG files ===
Total DEGs to distribute: 6942 

Flower: 1192 DEGs -> Flower_all_DEGs.csv
  carpel: 36 DEGs
  flower_stage_10_11: 21 DEGs
  flower_stage_12: 2 DEGs
  flower_stage_15: 10 DEGs
  flower_stage_9: 36 DEGs
  inflorescence_apex: 201 DEGs
  mature_pollen: 530 DEGs
  pedicel: 69 DEGs
  petal: 101 DEGs
  sepal: 120 DEGs
  stamen: 66 DEGs
Leaf_Shoot: 1399 DEGs -> Leaf_Shoot_all_DEGs.csv
  cauline_leaf: 77 DEGs
  cotyledon: 169 DEGs
  hypocotyl: 114 DEGs
  rosette_leaf: 38 DEGs
  rosette_vegetative: 129 DEGs
  senescing_leaf: 436 DEGs
  shoot_apex_transition: 212 DEGs
  shoot_apex_vegetative: 63 DEGs
  shoot_apex_vegetative_with_leaves: 59 DEGs
  stem: 102 DEGs
Root: 432 DEGs -> Root_all_DEGs.csv
  root: 432 DEGs
Seed_Silique: 609 DEGs -> Seed_Silique_all_DEGs.csv
  seed_stage_10: 180 DEGs
  seed_stage_6: 96 DEGs
  seed_stage_7: 62 DEGs
  seed_stage_8: 54 DEGs
  seed_stage_9: 116 DEGs
  silique_stage_3: 33 DEGs
  silique_stage_4: 49 DEGs
  silique_stage

Generate tissue DEG summary table

In [25]:

# ── Generate summary table: DEG counts per tissue with up/down breakdown ───────
deg_annotated <- readRDS("/workspace/deg_annotated.rds")
degs_only <- deg_annotated[deg_annotated$deg_flag == "yes", ]

# ── Sub-tissue level summary ───────────────────────────────────────────────────
build_summary <- function(df, group_col) {
  summary_list <- list()
  for (grp in sort(unique(df[[group_col]]))) {
    subset_df <- df[df[[group_col]] == grp, ]
    summary_list[[grp]] <- data.frame(
      tissue = grp,
      total_DEGs = nrow(subset_df),
      upregulated = sum(subset_df$regulation == "up"),
      downregulated = sum(subset_df$regulation == "down"),
      median_log2FC = round(median(subset_df$log2FoldChange, na.rm = TRUE), 4),
      mean_log2FC = round(mean(subset_df$log2FoldChange, na.rm = TRUE), 4),
      mean_tau_subtissue = round(mean(subset_df$tau_subtissue, na.rm = TRUE), 4),
      mean_tau_broad = round(mean(subset_df$tau_broad, na.rm = TRUE), 4),
      stringsAsFactors = FALSE
    )
  }
  do.call(rbind, summary_list)
}

# Sub-tissue summary
sub_summary <- build_summary(degs_only, "predominant_subtissue")
sub_summary <- sub_summary[order(-sub_summary$total_DEGs), ]

cat("=== Sub-tissue summary ===\n")
print(sub_summary)

# Broad-organ summary
broad_summary <- build_summary(degs_only, "predominant_broad_organ")
broad_summary <- broad_summary[order(-broad_summary$total_DEGs), ]

cat("\n=== Broad-organ summary ===\n")
print(broad_summary)

# ── Combined summary with hierarchical structure ───────────────────────────────
# Build a hierarchical summary: broad_organ -> sub_tissue
hierarchical_summary <- data.frame(
  broad_organ = character(),
  sub_tissue = character(),
  total_DEGs = integer(),
  upregulated = integer(),
  downregulated = integer(),
  median_log2FC = numeric(),
  mean_log2FC = numeric(),
  mean_tau_subtissue = numeric(),
  stringsAsFactors = FALSE
)

# Add tissue-specific DEGs grouped hierarchically
tissue_specific <- degs_only[degs_only$specificity == "tissue_specific", ]

for (bo in sort(unique(tissue_specific$predominant_broad_organ))) {
  for (st in sort(unique(tissue_specific$predominant_subtissue[tissue_specific$predominant_broad_organ == bo]))) {
    subset_df <- tissue_specific[tissue_specific$predominant_broad_organ == bo & 
                                  tissue_specific$predominant_subtissue == st, ]
    hierarchical_summary <- rbind(hierarchical_summary, data.frame(
      broad_organ = bo,
      sub_tissue = st,
      total_DEGs = nrow(subset_df),
      upregulated = sum(subset_df$regulation == "up"),
      downregulated = sum(subset_df$regulation == "down"),
      median_log2FC = round(median(subset_df$log2FoldChange, na.rm = TRUE), 4),
      mean_log2FC = round(mean(subset_df$log2FoldChange, na.rm = TRUE), 4),
      mean_tau_subtissue = round(mean(subset_df$tau_subtissue, na.rm = TRUE), 4),
      stringsAsFactors = FALSE
    ))
  }
}

# Add constitutive and unannotated rows
for (spec in c("constitutive", "unannotated")) {
  subset_df <- degs_only[degs_only$specificity == spec, ]
  hierarchical_summary <- rbind(hierarchical_summary, data.frame(
    broad_organ = spec,
    sub_tissue = "(all)",
    total_DEGs = nrow(subset_df),
    upregulated = sum(subset_df$regulation == "up"),
    downregulated = sum(subset_df$regulation == "down"),
    median_log2FC = round(median(subset_df$log2FoldChange, na.rm = TRUE), 4),
    mean_log2FC = round(mean(subset_df$log2FoldChange, na.rm = TRUE), 4),
    mean_tau_subtissue = round(mean(subset_df$tau_subtissue, na.rm = TRUE), 4),
    stringsAsFactors = FALSE
  ))
}

# Add total row
hierarchical_summary <- rbind(hierarchical_summary, data.frame(
  broad_organ = "TOTAL",
  sub_tissue = "(all)",
  total_DEGs = nrow(degs_only),
  upregulated = sum(degs_only$regulation == "up"),
  downregulated = sum(degs_only$regulation == "down"),
  median_log2FC = round(median(degs_only$log2FoldChange, na.rm = TRUE), 4),
  mean_log2FC = round(mean(degs_only$log2FoldChange, na.rm = TRUE), 4),
  mean_tau_subtissue = round(mean(degs_only$tau_subtissue, na.rm = TRUE), 4),
  stringsAsFactors = FALSE
))

cat("\n=== Hierarchical summary ===\n")
print(hierarchical_summary)

# Export
write.csv(hierarchical_summary, "/mnt/results/tissue_deg_summary.csv", row.names = FALSE)
cat("\n=== Summary saved to /mnt/results/tissue_deg_summary.csv ===\n")


=== Sub-tissue summary ===
                                                             tissue total_DEGs
unannotated                                             unannotated       1123
senescing_leaf                                       senescing_leaf        665
mature_pollen                                         mature_pollen        636
root                                                           root        517
shoot_apex_transition                         shoot_apex_transition        414
inflorescence_apex                               inflorescence_apex        397
seed_stage_10                                         seed_stage_10        320
shoot_apex_vegetative                         shoot_apex_vegetative        258
cotyledon                                                 cotyledon        234
rosette_vegetative                               rosette_vegetative        226
seed_stage_9                                           seed_stage_9        183
seed_stage_6             

Generate bar chart of DEG counts by organ

In [27]:

# ── Visualization 1: Bar chart of DEG counts per broad organ (stacked up/down) ─
suppressPackageStartupMessages({
  library(ggplot2)
})

deg_annotated <- readRDS("/workspace/deg_annotated.rds")
degs_only <- deg_annotated[deg_annotated$deg_flag == "yes", ]

# Prepare data — exclude unannotated for the tissue plot
tissue_degs <- degs_only[degs_only$specificity == "tissue_specific", ]

# Order broad organs by total DEG count
organ_order <- names(sort(table(tissue_degs$predominant_broad_organ), decreasing = TRUE))
tissue_degs$predominant_broad_organ <- factor(tissue_degs$predominant_broad_organ, levels = organ_order)

# Color palette (colorblind-friendly)
up_color <- "#0072B2"   # blue
down_color <- "#D55E00" # vermillion

p1 <- ggplot(tissue_degs, aes(x = predominant_broad_organ, fill = regulation)) +
  geom_bar(position = "stack", width = 0.7) +
  scale_fill_manual(values = c("up" = up_color, "down" = down_color),
                    labels = c("up" = "Upregulated", "down" = "Downregulated"),
                    name = "Regulation") +
  labs(
    title = "Radiation-Responsive DEGs by Tissue (Broad Organ)",
    subtitle = "Arabidopsis thaliana — OSD-498/510 radiation effect",
    x = "Broad Organ",
    y = "Number of DEGs"
  ) +
  theme_minimal(base_size = 14) +
  theme(
    text = element_text(family = "Liberation Sans"),
    plot.title = element_text(face = "bold", size = 16),
    plot.subtitle = element_text(size = 11, color = "grey40"),
    axis.text.x = element_text(angle = 30, hjust = 1, size = 12),
    legend.position = "top",
    panel.grid.minor = element_blank()
  ) +
  geom_text(stat = "count", aes(label = after_stat(count)), 
            position = position_stack(vjust = 0.5), size = 3.5, color = "white")

ggsave("/mnt/results/fig1_deg_counts_by_organ.png", p1, width = 8, height = 6, dpi = 300, bg = "white")
cat("Saved fig1_deg_counts_by_organ.png\n")


Warning message:
“package ‘ggplot2’ was built under R version 4.4.3”
Saved fig1_deg_counts_by_organ.png


Generate volcano plots per organ

In [29]:

# ── Visualization 2: Volcano plots per broad organ (multi-panel) ───────────────
suppressPackageStartupMessages({
  library(ggplot2)
  library(ggrepel)
})

# Prepare data — tissue-specific DEGs only, exclude unannotated
tissue_degs <- degs_only[degs_only$specificity == "tissue_specific", ]

# For volcano plots, we need all genes (not just DEGs) colored by tissue assignment
# Use the full annotated dataset
all_genes <- deg_annotated
all_genes$neg_log10_padj <- -log10(all_genes$padj)

# For each broad organ, create a volcano plot
organs <- sort(unique(tissue_degs$predominant_broad_organ))

# Create a combined plot with one panel per organ
volcano_list <- list()

for (bo in organs) {
  # Genes assigned to this organ
  organ_genes <- all_genes[all_genes$predominant_broad_organ == bo, ]
  
  # Also include constitutive genes as grey background
  constitutive <- all_genes[all_genes$specificity == "constitutive", ]
  
  # Combine: constitutive as background, organ-specific as colored
  plot_data <- rbind(
    data.frame(constitutive[, c("log2FoldChange", "neg_log10_padj", "deg_flag", "regulation")], 
               category = "constitutive"),
    data.frame(organ_genes[, c("log2FoldChange", "neg_log10_padj", "deg_flag", "regulation")],
               category = bo)
  )
  
  # Cap neg_log10_padj for visualization
  plot_data$neg_log10_padj[plot_data$neg_log10_padj > 50] <- 50
  
  p <- ggplot(plot_data, aes(x = log2FoldChange, y = neg_log10_padj, color = category)) +
    geom_point(data = subset(plot_data, category == "constitutive"), 
               color = "grey80", alpha = 0.3, size = 0.8) +
    geom_point(data = subset(plot_data, category == bo & deg_flag == "no"),
               color = "grey60", alpha = 0.4, size = 0.8) +
    geom_point(data = subset(plot_data, category == bo & deg_flag == "yes" & regulation == "up"),
               color = up_color, alpha = 0.7, size = 1.2) +
    geom_point(data = subset(plot_data, category == bo & deg_flag == "yes" & regulation == "down"),
               color = down_color, alpha = 0.7, size = 1.2) +
    geom_hline(yintercept = -log10(0.05), linetype = "dashed", color = "grey50", linewidth = 0.4) +
    geom_vline(xintercept = 0, linetype = "dotted", color = "grey50", linewidth = 0.4) +
    labs(
      title = bo,
      x = "log2 Fold Change",
      y = "-log10(padj)"
    ) +
    theme_minimal(base_size = 11) +
    theme(
      text = element_text(family = "Liberation Sans"),
      plot.title = element_text(face = "bold", size = 12, hjust = 0.5),
      legend.position = "none",
      panel.grid.minor = element_blank()
    ) +
    coord_cartesian(xlim = c(-4, 7), ylim = c(0, 52))
  
  volcano_list[[bo]] <- p
}

# Combine using patchwork-like approach with cowplot or gridExtra
# Use gridExtra
suppressPackageStartupMessages(library(gridExtra))

# Create a legend plot
legend_data <- data.frame(
  x = c(1, 2, 3),
  label = c("Upregulated DEG", "Downregulated DEG", "Non-DEG / Constitutive"),
  color = c(up_color, down_color, "grey70")
)

combined_volcano <- do.call(grid.arrange, c(volcano_list, ncol = 3, 
  top = textGrob("Volcano Plots by Tissue: Radiation-Responsive Genes", 
                 gp = gpar(fontface = "bold", fontsize = 14, fontfamily = "Liberation Sans"))))

ggsave("/mnt/results/fig2_volcano_plots_by_organ.png", 
       arrangeGrob(grobs = volcano_list, ncol = 3,
         top = textGrob("Volcano Plots by Tissue: Radiation-Responsive Genes",
                        gp = gpar(fontface = "bold", fontsize = 14, fontfamily = "Liberation Sans"))),
       width = 14, height = 10, dpi = 300, bg = "white")
cat("Saved fig2_volcano_plots_by_organ.png\n")


Warning message:
“package ‘ggrepel’ was built under R version 4.4.3”
Warning message:
“package ‘gridExtra’ was built under R version 4.4.3”
Error in h(simpleError(msg, call)): error in evaluating the argument 'args' in selecting a method for function 'do.call': could not find function "textGrob"
Traceback:

1. .handleSimpleError(function (cond) 
 . .Internal(C_tryCatchHelper(addr, 1L, cond)), "could not find function \"textGrob\"", 
 .     base::quote(textGrob("Volcano Plots by Tissue: Radiation-Responsive Genes", 
 .         gp = gpar(fontface = "bold", fontsize = 14, fontfamily = "Liberation Sans"))))
2. h(simpleError(msg, call))
3. .handleSimpleError(function (cnd) 
 . {
 .     watcher$capture_plot_and_output()
 .     cnd <- sanitize_call(cnd)
 .     watcher$push(cnd)
 .     switch(on_error, continue = invokeRestart("eval_continue"), 
 .         stop = invokeRestart("eval_stop"), error = NULL)
 . }, "error in evaluating the argument 'args' in selecting a method for function 'do.call

Regenerate volcano plots with grid loaded

In [31]:

# ── Fix: load grid package and regenerate volcano plots ────────────────────────
suppressPackageStartupMessages({
  library(grid)
  library(gridExtra)
  library(ggplot2)
})

# Rebuild volcano plots (variables still in scope from previous cell)
# Rebuild plot_data for each organ
volcano_list <- list()

for (bo in organs) {
  organ_genes <- all_genes[all_genes$predominant_broad_organ == bo, ]
  
  plot_data <- rbind(
    data.frame(constitutive[, c("log2FoldChange", "neg_log10_padj", "deg_flag", "regulation")],
               category = "constitutive"),
    data.frame(organ_genes[, c("log2FoldChange", "neg_log10_padj", "deg_flag", "regulation")],
               category = bo)
  )
  
  plot_data$neg_log10_padj[plot_data$neg_log10_padj > 50] <- 50
  
  p <- ggplot(plot_data, aes(x = log2FoldChange, y = neg_log10_padj)) +
    geom_point(data = subset(plot_data, category == "constitutive"),
               color = "grey80", alpha = 0.3, size = 0.8) +
    geom_point(data = subset(plot_data, category == bo & deg_flag == "no"),
               color = "grey60", alpha = 0.4, size = 0.8) +
    geom_point(data = subset(plot_data, category == bo & deg_flag == "yes" & regulation == "up"),
               color = up_color, alpha = 0.7, size = 1.2) +
    geom_point(data = subset(plot_data, category == bo & deg_flag == "yes" & regulation == "down"),
               color = down_color, alpha = 0.7, size = 1.2) +
    geom_hline(yintercept = -log10(0.05), linetype = "dashed", color = "grey50", linewidth = 0.4) +
    geom_vline(xintercept = 0, linetype = "dotted", color = "grey50", linewidth = 0.4) +
    labs(
      title = bo,
      x = "log2 Fold Change",
      y = "-log10(padj)"
    ) +
    theme_minimal(base_size = 11) +
    theme(
      text = element_text(family = "Liberation Sans"),
      plot.title = element_text(face = "bold", size = 12, hjust = 0.5),
      legend.position = "none",
      panel.grid.minor = element_blank()
    ) +
    coord_cartesian(xlim = c(-4, 7), ylim = c(0, 52))
  
  volcano_list[[bo]] <- p
}

# Save with grid textGrob for title
ggsave("/mnt/results/fig2_volcano_plots_by_organ.png",
       arrangeGrob(grobs = volcano_list, ncol = 3,
         top = textGrob("Volcano Plots by Tissue: Radiation-Responsive Genes",
                        gp = gpar(fontface = "bold", fontsize = 14, fontfamily = "Liberation Sans"))),
       width = 14, height = 10, dpi = 300, bg = "white")
cat("Saved fig2_volcano_plots_by_organ.png\n")


Saved fig2_volcano_plots_by_organ.png


Generate heatmap of top DEGs by tissue

In [33]:

# ── Visualization 3: Heatmap of top DEGs x tissues (expression specificity) ───
suppressPackageStartupMessages({
  library(ComplexHeatmap)
  library(circlize)
})

# Load atlas
atlas <- readRDS("/workspace/atgenexpress_atlas.rds")
sub_tissue_expr <- atlas$sub_tissue_expr
deg_annotated <- readRDS("/workspace/deg_annotated.rds")
degs_only <- deg_annotated[deg_annotated$deg_flag == "yes", ]

# Select top 50 DEGs by padj (most significant)
top_degs <- degs_only[order(degs_only$padj), ][1:50, ]

# Get their expression in the atlas (sub-tissue level)
top_genes <- top_degs$gene_id
genes_in_atlas <- intersect(top_genes, rownames(sub_tissue_expr))

# Expression matrix for these genes
expr_top <- sub_tissue_expr[genes_in_atlas, , drop = FALSE]

# Z-score normalize per gene (row)
expr_z <- t(scale(t(expr_top)))

# Order columns by broad organ for visual grouping
sub_to_broad <- unique(atlas$all_samples[, c("sub_tissue", "broad_organ")])
col_order <- sub_to_broad[order(sub_to_broad$broad_organ), "sub_tissue"]
col_order <- intersect(col_order, colnames(expr_z))
expr_z <- expr_z[, col_order]

# Create column annotation (broad organ)
col_anno_data <- data.frame(
  broad_organ = sub_to_broad$broad_organ[match(col_order, sub_to_broad$sub_tissue)],
  row.names = col_order
)

# Color palette for broad organs
organ_colors <- c(
  "Root" = "#0072B2",
  "Seedling" = "#009E73",
  "Leaf_Shoot" = "#E69F00",
  "Flower" = "#CC79A7",
  "Seed_Silique" = "#D55E00"
)

col_anno <- HeatmapAnnotation(
  Broad_Organ = col_anno_data$broad_organ,
  col = list(Broad_Organ = organ_colors),
  annotation_name_gp = gpar(fontsize = 9, fontfamily = "Liberation Sans"),
  annotation_legend_param = list(title_gp = gpar(fontsize = 10, fontfamily = "Liberation Sans"),
                                  labels_gp = gpar(fontsize = 9, fontfamily = "Liberation Sans"))
)

# Row annotation: regulation direction
row_anno_data <- data.frame(
  regulation = top_degs$regulation[match(genes_in_atlas, top_degs$gene_id)],
  row.names = genes_in_atlas
)

row_anno <- rowAnnotation(
  Regulation = row_anno_data$regulation,
  col = list(Regulation = c("up" = up_color, "down" = down_color)),
  annotation_name_gp = gpar(fontsize = 9, fontfamily = "Liberation Sans"),
  annotation_legend_param = list(title_gp = gpar(fontsize = 10, fontfamily = "Liberation Sans"),
                                  labels_gp = gpar(fontsize = 9, fontfamily = "Liberation Sans"))
)

# Create heatmap
png("/mnt/results/fig3_heatmap_top_degs.png", width = 12, height = 8, units = "in", res = 300, bg = "white")

ht <- Heatmap(expr_z,
  name = "Z-score\nExpression",
  col = colorRamp2(c(-2, 0, 2), c(down_color, "white", up_color)),
  top_annotation = col_anno,
  right_annotation = row_anno,
  cluster_columns = FALSE,
  cluster_rows = TRUE,
  show_row_names = TRUE,
  show_column_names = TRUE,
  row_names_gp = gpar(fontsize = 7, fontfamily = "Liberation Sans"),
  column_names_gp = gpar(fontsize = 7, fontfamily = "Liberation Sans"),
  column_names_rot = 45,
  column_title = "Top 50 DEGs by Significance — Tissue Expression Specificity",
  column_title_gp = gpar(fontface = "bold", fontsize = 13, fontfamily = "Liberation Sans"),
  heatmap_legend_param = list(title_gp = gpar(fontsize = 10, fontfamily = "Liberation Sans"),
                               labels_gp = gpar(fontsize = 9, fontfamily = "Liberation Sans"))
)

draw(ht)
dev.off()
cat("Saved fig3_heatmap_top_degs.png\n")


Warning message:
“package ‘circlize’ was built under R version 4.4.3”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“font family 'Liberation Sans' not found in PostScript font database”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“font family 'Liberation Sans' not found in PostScript font database”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“font family 'Liberation Sans' not found in PostScript font database”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“font family 'Liberation Sans' not found in PostScript font database”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“font family 'Liberation Sans' not found in PostScript font database”
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
“font family 'Liberation Sans' not found in PostScript font database”
Warning message in

Prepare data for UpSet diagram

In [35]:

# ── Visualization 4: UpSet diagram of DEG overlaps across broad organs ─────────
suppressPackageStartupMessages({
  library(UpSetR)
})

deg_annotated <- readRDS("/workspace/deg_annotated.rds")
degs_only <- deg_annotated[deg_annotated$deg_flag == "yes", ]

# For UpSet: create binary matrix of gene x broad_organ membership
# Only tissue-specific DEGs (each gene belongs to exactly one organ in our assignment)
# But we can also show constitutive + unannotated as separate sets

# Actually, since each tissue-specific gene is assigned to ONE organ, 
# the overlaps would be empty. Let's instead show the overlap at the GENE level:
# which genes are DEGs AND tissue-specific to each organ?

# Better approach: show the overlap of DEG sets across organs using ALL DEGs
# that have expression in each organ above a threshold.
# 
# Alternative: show the overlap between tissue-specific, constitutive, and unannotated categories
# That's not very informative either.
#
# Best approach for UpSet: For each broad organ, define the set of DEGs whose 
# atlas expression is above median in that organ (i.e., "expressed" in that organ).
# This shows which DEGs are active in multiple organs.

atlas <- readRDS("/workspace/atgenexpress_atlas.rds")
broad_organ_expr <- atlas$broad_organ_expr

# Get DEGs that are in the atlas
degs_in_atlas <- degs_only[deggs_only$gene_id %in% rownames(broad_organ_expr), ]
degs_in_atlas <- degs_only[deggs_only$gene_id %in% rownames(broad_organ_expr), ]

# Fix: typo above, redo properly
degs_in_atlas <- degs_only[deggs_only$gene_id %in% rownames(broad_organ_expr), ]


Error in h(simpleError(msg, call)): error in evaluating the argument 'x' in selecting a method for function '%in%': object 'deggs_only' not found
Traceback:

1. `[.data.frame`(degs_only, deggs_only$gene_id %in% rownames(broad_organ_expr), 
 .     )
2. deggs_only$gene_id %in% rownames(broad_organ_expr)
3. .handleSimpleError(function (cond) 
 . .Internal(C_tryCatchHelper(addr, 1L, cond)), "object 'deggs_only' not found", 
 .     base::quote(eval(expr, envir)))
4. h(simpleError(msg, call))
5. .handleSimpleError(function (cnd) 
 . {
 .     watcher$capture_plot_and_output()
 .     cnd <- sanitize_call(cnd)
 .     watcher$push(cnd)
 .     switch(on_error, continue = invokeRestart("eval_continue"), 
 .         stop = invokeRestart("eval_stop"), error = NULL)
 . }, "error in evaluating the argument 'x' in selecting a method for function '%in%': object 'deggs_only' not found", 
 .     base::quote(h(simpleError(msg, call))))

Prepare UpSet data attempt 2

In [37]:

# ── UpSet diagram: DEG overlaps across broad organs ────────────────────────────
suppressPackageStartupMessages({
  library(UpSetR)
})

deg_annotated <- readRDS("/workspace/deg_annotated.rds")
degs_only <- deg_annotated[deg_annotated$deg_flag == "yes", ]
atlas <- readRDS("/workspace/atgenexpress_atlas.rds")
broad_organ_expr <- atlas$broad_organ_expr

# Get DEGs that are in the atlas
degs_in_atlas <- degs_only[deggs_only$gene_id %in% rownames(broad_organ_expr), ]


Error in h(simpleError(msg, call)): error in evaluating the argument 'x' in selecting a method for function '%in%': object 'deggs_only' not found
Traceback:

1. `[.data.frame`(degs_only, deggs_only$gene_id %in% rownames(broad_organ_expr), 
 .     )
2. deggs_only$gene_id %in% rownames(broad_organ_expr)
3. .handleSimpleError(function (cond) 
 . .Internal(C_tryCatchHelper(addr, 1L, cond)), "object 'deggs_only' not found", 
 .     base::quote(eval(expr, envir)))
4. h(simpleError(msg, call))
5. .handleSimpleError(function (cnd) 
 . {
 .     watcher$capture_plot_and_output()
 .     cnd <- sanitize_call(cnd)
 .     watcher$push(cnd)
 .     switch(on_error, continue = invokeRestart("eval_continue"), 
 .         stop = invokeRestart("eval_stop"), error = NULL)
 . }, "error in evaluating the argument 'x' in selecting a method for function '%in%': object 'deggs_only' not found", 
 .     base::quote(h(simpleError(msg, call))))

Generate UpSet diagram of organ overlaps

In [39]:

# ── UpSet diagram: DEG overlaps across broad organs ────────────────────────────
suppressPackageStartupMessages({
  library(UpSetR)
})

deg_annotated <- readRDS("/workspace/deg_annotated.rds")
degs_only <- deg_annotated[deg_annotated$deg_flag == "yes", ]
atlas <- readRDS("/workspace/atgenexpress_atlas.rds")
broad_organ_expr <- atlas$broad_organ_expr

# Get DEGs that are in the atlas
degs_in_atlas <- degs_only[degs_only$gene_id %in% rownames(broad_organ_expr), ]
cat("DEGs in atlas:", nrow(degs_in_atlas), "\n")

# For each broad organ, define the set of DEGs "expressed" in that organ
# (expression above the gene's median across all organs — i.e., relatively high)
# This shows which DEGs are active in multiple organs

# Get expression for DEGs
expr_degs <- broad_organ_expr[degs_in_atlas$gene_id, , drop = FALSE]
cat("Expression matrix for DEGs:", nrow(expr_degs), "x", ncol(expr_degs), "\n")

# For each gene, determine which organs it's "expressed" in (above gene median)
gene_medians <- apply(expr_degs, 1, median, na.rm = TRUE)

# Build binary matrix: gene x organ (1 = expressed above median, 0 = not)
binary_matrix <- matrix(0, nrow = nrow(expr_degs), ncol = ncol(expr_degs))
rownames(binary_matrix) <- rownames(expr_degs)
colnames(binary_matrix) <- colnames(expr_degs)

for (i in seq_len(nrow(expr_degs))) {
  binary_matrix[i, ] <- as.integer(expr_degs[i, ] > gene_medians[i])
}

# Convert to data frame for UpSetR
binary_df <- as.data.frame(binary_matrix)
binary_df$gene_id <- rownames(binary_df)

cat("\n=== Organ set sizes (DEGs expressed above median) ===\n")
print(colSums(binary_matrix))

# Create UpSet plot
png("/mnt/results/fig4_upset_organ_overlaps.png", width = 10, height = 6, units = "in", res = 300, bg = "white")

upset_plot <- upset(binary_df, 
  sets = colnames(expr_degs),
  order.by = "freq",
  nsets = 5,
  nintersects = 20,
  sets.bar.color = c("#0072B2", "#009E73", "#E69F00", "#CC79A7", "#D55E00"),
  main.bar.color = "grey30",
  matrix.color = "#0279EE",
  point.size = 3,
  line.size = 0.8,
  text.scale = 1.2,
  mb.ratio = c(0.6, 0.4),
  mainbar.y.label = "DEG Intersection Size",
  sets.x.label = "DEGs Expressed Above Median",
  labels = c("Intersection", "Set")
)

print(upset_plot)
dev.off()
cat("\nSaved fig4_upset_organ_overlaps.png\n")


DEGs in atlas: 5819 
Expression matrix for DEGs: 5819 x 5 

=== Organ set sizes (DEGs expressed above median) ===
    Seedling   Leaf_Shoot         Root       Flower Seed_Silique 
        2041         2178         2717         2629         2073 
Error in upset(binary_df, sets = colnames(expr_degs), order.by = "freq", : unused argument (labels = c("Intersection", "Set"))
Traceback:


Generate UpSet diagram fixed

In [41]:

# ── Fix UpSet diagram (remove invalid 'labels' argument) ───────────────────────
png("/mnt/results/fig4_upset_organ_overlaps.png", width = 10, height = 6, units = "in", res = 300, bg = "white")

upset_plot <- upset(binary_df, 
  sets = colnames(expr_degs),
  order.by = "freq",
  nsets = 5,
  nintersects = 20,
  sets.bar.color = c("#0072B2", "#009E73", "#E69F00", "#CC79A7", "#D55E00"),
  main.bar.color = "grey30",
  matrix.color = "#0279EE",
  point.size = 3,
  line.size = 0.8,
  text.scale = 1.2,
  mb.ratio = c(0.6, 0.4),
  mainbar.y.label = "DEG Intersection Size",
  sets.x.label = "DEGs Expressed Above Median"
)

print(upset_plot)
dev.off()
cat("Saved fig4_upset_organ_overlaps.png\n")


Warning message:
“`aes_string()` was deprecated in ggplot2 3.0.0.
ℹ Please use tidy evaluation idioms with `aes()`.
ℹ See also `vignette("ggplot2-in-packages")` for more information.
ℹ The deprecated feature was likely used in the UpSetR package.
  Please report the issue at <https://github.com/hms-dbmi/UpSetR/issues>.”
Warning message:
“Using `size` aesthetic for lines was deprecated in ggplot2 3.4.0.
ℹ Please use `linewidth` instead.
ℹ The deprecated feature was likely used in the UpSetR package.
  Please report the issue at <https://github.com/hms-dbmi/UpSetR/issues>.”
Warning message:
“The `size` argument of `element_line()` is deprecated as of ggplot2 3.4.0.
ℹ Please use the `linewidth` argument instead.
ℹ The deprecated feature was likely used in the UpSetR package.
  Please report the issue at <https://github.com/hms-dbmi/UpSetR/issues>.”
Saved fig4_upset_organ_overlaps.png


Set up PDF generation infrastructure

In [43]:

# ── Generate the PDF instruction manual ────────────────────────────────────────
# Mixed-level educational document using ReportLab (per pdf-report-generation skill)

import os
from reportlab.lib.pagesizes import letter
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
                                 PageBreak, HRFlowable, KeepTogether, Image)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.colors import HexColor
from reportlab.lib.enums import TA_JUSTIFY, TA_CENTER, TA_LEFT
from reportlab.lib.units import inch

# ── Phylo Brand Colors ──
PHYLO_BLACK     = HexColor("#000000")
PHYLO_WARM_GRAY = HexColor("#ECE9E2")
PHYLO_OFF_WHITE = HexColor("#FAF9F3")
PHYLO_GOLD      = HexColor("#D4A04A")
HEADING_COLOR   = HexColor("#111111")
BODY_TEXT       = HexColor("#2C2A26")
MUTED_TEXT      = HexColor("#8A8378")
TABLE_HEADER_BG = PHYLO_GOLD
TABLE_HEADER_FG = HexColor("#FFFFFF")
TABLE_ALT_ROW   = HexColor("#F9F7F3")
TABLE_BORDER    = HexColor("#D5CFC5")
DIVIDER_COLOR   = PHYLO_GOLD
CALLOUT_BG      = PHYLO_OFF_WHITE
CALLOUT_BORDER  = PHYLO_GOLD

# ── Fonts ──
FONT_HEADING = "Helvetica-Bold"
FONT_BODY    = "Helvetica"
FONT_ITALIC  = "Helvetica-Oblique"
FONT_MONO    = "Courier"

# ── Styles ──
styles = getSampleStyleSheet()

styles.add(ParagraphStyle(name="ReportTitle", fontName=FONT_HEADING, fontSize=26,
    textColor=HEADING_COLOR, spaceBefore=0, spaceAfter=6, leading=32))
styles.add(ParagraphStyle(name="Subtitle", fontName=FONT_BODY, fontSize=12,
    textColor=PHYLO_GOLD, spaceAfter=4))
styles.add(ParagraphStyle(name="Attribution", fontName=FONT_ITALIC, fontSize=10,
    textColor=MUTED_TEXT, spaceAfter=8))
styles.add(ParagraphStyle(name="SectionHead", fontName=FONT_HEADING, fontSize=18,
    textColor=HEADING_COLOR, spaceBefore=24, spaceAfter=10))
styles.add(ParagraphStyle(name="SubHead", fontName=FONT_HEADING, fontSize=13,
    textColor=HEADING_COLOR, spaceBefore=16, spaceAfter=6))
styles.add(ParagraphStyle(name="Body", fontName=FONT_BODY, fontSize=10.5,
    textColor=BODY_TEXT, alignment=TA_JUSTIFY, spaceAfter=8, leading=15))
styles.add(ParagraphStyle(name="Caption", fontName=FONT_ITALIC, fontSize=9,
    textColor=MUTED_TEXT, alignment=TA_CENTER, spaceAfter=14))
styles.add(ParagraphStyle(name="CalloutText", fontName=FONT_BODY, fontSize=10,
    textColor=BODY_TEXT, alignment=TA_LEFT, spaceAfter=4, leading=14))
styles.add(ParagraphStyle(name="CalloutTitle", fontName=FONT_HEADING, fontSize=10.5,
    textColor=PHYLO_GOLD, spaceAfter=4))
styles.add(ParagraphStyle(name="CodeStyle", fontName=FONT_MONO, fontSize=8.5,
    textColor=BODY_TEXT, alignment=TA_LEFT, spaceAfter=4, leading=11,
    leftIndent=12, rightIndent=12))
styles.add(ParagraphStyle(name="BulletStyle", fontName=FONT_BODY, fontSize=10.5,
    textColor=BODY_TEXT, alignment=TA_LEFT, spaceAfter=4, leading=15,
    leftIndent=20, bulletIndent=8))
styles.add(ParagraphStyle(name="TableCell", fontName=FONT_BODY, fontSize=9,
    textColor=BODY_TEXT, alignment=TA_LEFT, leading=12))
styles.add(ParagraphStyle(name="TableHeader", fontName=FONT_HEADING, fontSize=9,
    textColor=TABLE_HEADER_FG, alignment=TA_LEFT, leading=12))

# ── Helper functions ──
def divider(width=480):
    return HRFlowable(width=width, thickness=1, color=DIVIDER_COLOR,
                      spaceAfter=10, spaceBefore=4)

def callout_box(title, text, width=460):
    """Beginner-friendly sidebar callout with gold left border."""
    content = []
    if title:
        content.append(Paragraph(title, styles["CalloutTitle"]))
    content.append(Paragraph(text, styles["CalloutText"]))
    t = Table([[content]], colWidths=[width])
    t.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, -1), CALLOUT_BG),
        ("BOX", (0, 0), (-1, -1), 0.5, TABLE_BORDER),
        ("LINEBEFORE", (0, 0), (0, -1), 3, PHYLO_GOLD),
        ("TOPPADDING", (0, 0), (-1, -1), 10),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 10),
        ("LEFTPADDING", (0, 0), (-1, -1), 14),
        ("RIGHTPADDING", (0, 0), (-1, -1), 14),
    ]))
    t.hAlign = "CENTER"
    return t

def code_block(code_text, width=460):
    """Monospace code block with light background."""
    # Escape XML special chars
    escaped = code_text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    # Preserve line breaks
    escaped = escaped.replace("\n", "<br/>")
    p = Paragraph(escaped, styles["CodeStyle"])
    t = Table([[p]], colWidths=[width])
    t.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, -1), HexColor("#F5F3EE")),
        ("BOX", (0, 0), (-1, -1), 0.5, TABLE_BORDER),
        ("TOPPADDING", (0, 0), (-1, -1), 8),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 8),
        ("LEFTPADDING", (0, 0), (-1, -1), 10),
        ("RIGHTPADDING", (0, 0), (-1, -1), 10),
    ]))
    t.hAlign = "CENTER"
    return t

def make_table(headers, rows, col_widths):
    """Standard Phylo-styled table."""
    data = [[Paragraph(f'<b>{h}</b>', styles["TableHeader"]) for h in headers]]
    for row in rows:
        data.append([Paragraph(str(c), styles["TableCell"]) for c in row])
    t = Table(data, colWidths=col_widths, repeatRows=1)
    t.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), TABLE_HEADER_BG),
        ("TEXTCOLOR", (0, 0), (-1, 0), TABLE_HEADER_FG),
        *[("BACKGROUND", (0, i), (-1, i), TABLE_ALT_ROW) for i in range(2, len(data), 2)],
        ("GRID", (0, 0), (-1, -1), 0.5, TABLE_BORDER),
        ("BOX", (0, 0), (-1, -1), 0.75, TABLE_BORDER),
        ("TOPPADDING", (0, 0), (-1, -1), 5),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
        ("LEFTPADDING", (0, 0), (-1, -1), 6),
        ("RIGHTPADDING", (0, 0), (-1, -1), 6),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
    ]))
    t.hAlign = "CENTER"
    return t

# ── Page header/footer ──
def page_header_footer(canvas, doc):
    canvas.saveState()
    w, h = letter
    canvas.setFont("Helvetica", 9)
    canvas.setFillColor(MUTED_TEXT)
    canvas.drawString(60, h - 40, "Tissue-Specific DEG Analysis: Instruction Manual")
    canvas.setStrokeColor(PHYLO_GOLD)
    canvas.setLineWidth(1)
    canvas.line(60, h - 48, w - 60, h - 48)
    canvas.setStrokeColor(TABLE_BORDER)
    canvas.setLineWidth(0.75)
    canvas.line(60, 40, w - 60, 40)
    canvas.setFont("Helvetica", 8)
    canvas.setFillColor(MUTED_TEXT)
    canvas.drawCentredString(w / 2, 26, f"Page {doc.page}")
    canvas.restoreState()

# ── Build the document ──
output_path = "/mnt/results/instruction_manual_tissue_specific_deg.pdf"
doc = SimpleDocTemplate(output_path, pagesize=letter,
                        topMargin=52, bottomMargin=52,
                        leftMargin=60, rightMargin=60)
story = []

print("Building PDF manual...")


Building PDF manual...


Add title page and sections 1-3 to PDF

In [45]:

# ── PAGE 1: Title Page ─────────────────────────────────────────────────────────
story.append(Spacer(1, 40))
story.append(Paragraph("Tissue-Specific Separation of Radiation-Responsive DEGs", styles["ReportTitle"]))
story.append(Paragraph("An Educational Instruction Manual for R-Based Transcriptomic Analysis", styles["Subtitle"]))
story.append(Spacer(1, 8))
story.append(Paragraph("<i>Generated by Biomni (Phylo)  |  2026-07-09</i>", styles["Attribution"]))
story.append(Spacer(1, 24))

# Brief abstract
story.append(Paragraph(
    "This manual accompanies an R script that separates differentially expressed genes (DEGs) "
    "from an Arabidopsis thaliana radiation experiment into tissue-specific response groups. "
    "The dataset comes from NASA GeneLab studies OSD-498 and OSD-510, which examined the "
    "transcriptional effects of ionizing radiation on whole Arabidopsis seedlings. Because the "
    "original experiment used whole seedlings (not separated tissues), we use the AtGenExpress "
    "developmental expression atlas to infer which tissues each radiation-responsive gene is "
    "normally most active in, then group DEGs accordingly.", styles["Body"]))

story.append(divider())

# ── Table of Contents ──
story.append(Paragraph("Contents", styles["SubHead"]))
toc_items = [
    "1.  Introduction: The Experiment and the Data",
    "2.  Background: Differential Expression and DESeq2",
    "3.  The Tissue-Specificity Concept: Expression Atlases and the Tau Index",
    "4.  Code Walkthrough: Section-by-Section Explanation",
    "5.  Understanding the Outputs: CSVs, Summary Table, and Figures",
    "6.  How to Run the Code: Prerequisites and Execution",
    "7.  Interpretation Guide: What Tissue-Specific Radiation Responses Mean",
    "8.  Exercises and Extensions for Further Exploration",
]
for item in toc_items:
    story.append(Paragraph(item, styles["BulletStyle"]))

story.append(PageBreak())

# ── SECTION 1: Introduction ────────────────────────────────────────────────────
story.append(Paragraph("1. Introduction: The Experiment and the Data", styles["SectionHead"]))
story.append(divider())

story.append(Paragraph("1.1 The Biological Question", styles["SubHead"]))
story.append(Paragraph(
    "When plants are exposed to ionizing radiation (such as gamma rays or cosmic radiation), "
    "their cells activate DNA damage response pathways, alter cell cycle progression, and "
    "modulate stress-responsive gene expression. But different plant tissues and organs may "
    "respond differently to radiation. Root cells, which are underground and normally shielded, "
    "might mount a different transcriptional response than leaf cells, which are adapted to "
    "handle light and oxidative stress. Understanding tissue-specific radiation responses is "
    "important for space agriculture, where plants will grow in high-radiation environments "
    "beyond Earth's protective magnetic field.", styles["Body"]))

story.append(Paragraph("1.2 The Dataset", styles["SubHead"]))
story.append(Paragraph(
    "The input file <font face='Courier'>DEG_OSD498_510_radiation_effect.csv</font> contains "
    "the results of a differential expression analysis performed with DESeq2, a widely used "
    "R package for analyzing RNA-seq count data. The data comes from NASA GeneLab studies "
    "OSD-498 and OSD-510, which profiled the transcriptome of Arabidopsis thaliana seedlings "
    "exposed to ionizing radiation.", styles["Body"]))

story.append(Paragraph("The file contains 23,573 genes and 8 columns:", styles["Body"]))

deg_table = make_table(
    ["Column", "Description", "What It Tells You"],
    [
        ["gene_id", "AGI locus identifier (e.g., AT5G60250)", "Unique gene name in Arabidopsis"],
        ["baseMean", "Average normalized expression across all samples", "How highly expressed the gene is overall"],
        ["log2FoldChange", "Log2 ratio of expression (irradiated / control)", "Direction and magnitude of change; positive = up, negative = down"],
        ["lfcSE", "Standard error of the log2 fold change", "Uncertainty in the fold change estimate"],
        ["stat", "Wald test statistic", "Used to compute the p-value"],
        ["pvalue", "Raw p-value from the Wald test", "Statistical significance before correction"],
        ["padj", "Adjusted p-value (Benjamini-Hochberg FDR)", "Significance after multiple testing correction"],
        ["deg_flag", "yes / no flag", "Whether the gene is a differentially expressed gene (DEG)"],
    ],
    [90, 160, 210]
)
story.append(deg_table)
story.append(Spacer(1, 8))

story.append(Paragraph(
    "Of the 23,573 genes, <b>6,942 are flagged as DEGs</b> (deg_flag = 'yes'). These are the "
    "genes whose expression significantly changed in response to radiation.", styles["Body"]))

story.append(callout_box(
    "BEGINNER'S CORNER: What is Arabidopsis thaliana?",
    "Arabidopsis thaliana is a small flowering plant widely used as a model organism in plant "
    "biology, much like E. coli is used in microbiology or Drosophila in genetics. It has a "
    "small genome (~135 megabases, ~27,000 genes), a short life cycle (~6 weeks), and is easy "
    "to grow in the lab. Every gene in Arabidopsis has a unique identifier called an AGI code "
    "(Arabidopsis Genome Initiative), formatted as AT{chromosome}G{number}, e.g., AT5G60250 "
    "means chromosome 5, gene 60250."
))

story.append(callout_box(
    "BEGINNER'S CORNER: What is ionizing radiation?",
    "Ionizing radiation carries enough energy to damage DNA directly, causing single- and "
    "double-strand breaks. In space, astronauts and plants are exposed to galactic cosmic "
    "rays (high-energy protons and heavy ions) and solar particle events. On Earth, gamma "
    "rays from radioactive sources (like Cobalt-60) are used to simulate some aspects of "
    "space radiation in lab experiments. Plants have evolved DNA damage response pathways "
    "(involving proteins like ATM, ATR, and SOG1) that detect damage and activate repair genes."
))

story.append(PageBreak())

# ── SECTION 2: Background ──────────────────────────────────────────────────────
story.append(Paragraph("2. Background: Differential Expression and DESeq2", styles["SectionHead"]))
story.append(divider())

story.append(Paragraph("2.1 What is Differential Expression?", styles["SubHead"]))
story.append(Paragraph(
    "Differential expression analysis compares gene expression levels between two or more "
    "conditions (e.g., irradiated vs. control plants) to identify genes that are significantly "
    "upregulated (expressed more) or downregulated (expressed less) in one condition relative "
    "to the other. The output is a table with a fold change and a p-value for every gene.", styles["Body"]))

story.append(Paragraph("2.2 Key Concepts in the DESeq2 Output", styles["SubHead"]))

story.append(Paragraph("<b>Log2 Fold Change (log2FC):</b> This is the most important column. "
    "It tells you the direction and magnitude of expression change. A log2FC of +1 means the "
    "gene's expression doubled; -1 means it halved; +2 means it quadrupled. We use log2 "
    "(rather than raw fold change) because it makes up- and down-regulation symmetric: "
    "+2 and -2 represent the same magnitude of change in opposite directions.", styles["Body"]))

story.append(Paragraph("<b>Adjusted p-value (padj):</b> When testing thousands of genes "
    "simultaneously, some will appear significant by chance alone. The adjusted p-value "
    "(using the Benjamini-Hochberg procedure) controls the false discovery rate (FDR). "
    "A common threshold is padj &lt; 0.05, meaning we expect at most 5% of the genes we "
    "call significant to be false positives.", styles["Body"]))

story.append(callout_box(
    "BEGINNER'S CORNER: Why log2 instead of raw fold change?",
    "If a gene goes from 100 to 200 reads, that's a 2-fold increase. If it goes from 100 to 50, "
    "that's a 0.5-fold change. On a linear scale, 'doubling' (2x) and 'halving' (0.5x) look "
    "very different in magnitude. But on a log2 scale: log2(2) = +1 and log2(0.5) = -1. "
    "Now they're symmetric! This makes it much easier to visualize and compare up- and "
    "down-regulation on the same plot (like a volcano plot)."
))

story.append(Paragraph("2.3 The Challenge: No Tissue Information", styles["SubHead"]))
story.append(Paragraph(
    "The critical challenge with this dataset is that it comes from <b>whole seedling</b> "
    "experiments. The researchers harvested entire Arabidopsis seedlings (roots, leaves, "
    "stems, and all) and extracted RNA from the pooled tissue. This means the DEG file "
    "contains no tissue column — we cannot directly tell whether a radiation-responsive gene "
    "is primarily active in roots, leaves, flowers, or elsewhere.", styles["Body"]))

story.append(Paragraph(
    "To address this, we use an <b>expression atlas</b> approach: we look up where each gene "
    "is normally most highly expressed (across many tissues, measured in a separate reference "
    "dataset), and use that information to infer which tissue's biology the gene is most "
    "relevant to. This is an inference, not a direct measurement, but it is a well-established "
    "approach in plant genomics.", styles["Body"]))

story.append(PageBreak())

# ── SECTION 3: Tissue-Specificity Concept ──────────────────────────────────────
story.append(Paragraph("3. The Tissue-Specificity Concept: Expression Atlases and the Tau Index", styles["SectionHead"]))
story.append(divider())

story.append(Paragraph("3.1 The AtGenExpress Developmental Atlas", styles["SubHead"]))
story.append(Paragraph(
    "The AtGenExpress project (Schmid et al. 2005, Nature Genetics) is the canonical reference "
    "for Arabidopsis tissue-specific gene expression. It profiled gene expression across 79+ "
    "diverse tissue samples covering the entire plant life cycle, from seeds to senescing "
    "leaves, using the Affymetrix ATH1 microarray. The data is publicly available from the "
    "Gene Expression Omnibus (GEO) under accession numbers GSE5629 through GSE5634.", styles["Body"]))

story.append(Paragraph(
    "We download six GEO series, each covering a different organ system:", styles["Body"]))

atlas_table = make_table(
    ["GEO Accession", "Tissue Coverage", "Samples"],
    [
        ["GSE5629", "Seedlings and whole plants", "24"],
        ["GSE5630", "Leaves (rosette, cauline, senescing)", "60"],
        ["GSE5631", "Roots", "21"],
        ["GSE5632", "Flowers and pollen (all stages)", "66"],
        ["GSE5633", "Shoots, stems, and shoot apex", "42"],
        ["GSE5634", "Siliques and seeds (all stages)", "24"],
    ],
    [120, 260, 80]
)
story.append(atlas_table)
story.append(Spacer(1, 8))

story.append(Paragraph(
    "In total, this gives us 237 tissue-specific expression profiles across 20,833 genes. "
    "We organize these into a <b>hierarchical tissue structure</b>:", styles["Body"]))

hierarchy_table = make_table(
    ["Broad Organ", "Sub-Tissues", "Samples"],
    [
        ["Root", "root", "21"],
        ["Seedling", "seedling_green_parts, whole_plant_pre_bolting", "24"],
        ["Leaf_Shoot", "cotyledon, hypocotyl, rosette_leaf, cauline_leaf, senescing_leaf, rosette_vegetative, stem, shoot_apex (vegetative/transition)", "78"],
        ["Flower", "sepal, petal, stamen, carpel, mature_pollen, pedicel, flower_stage_9/10/12/15, inflorescence_apex", "90"],
        ["Seed_Silique", "silique_stage_3/4/5, seed_stage_6/7/8/9/10", "24"],
    ],
    [80, 300, 80]
)
story.append(hierarchy_table)
story.append(Spacer(1, 8))

story.append(Paragraph("3.2 The Tau Tissue-Specificity Index", styles["SubHead"]))
story.append(Paragraph(
    "To quantify how tissue-specific a gene's expression is, we compute the <b>Tau index</b>, "
    "a standard metric used in genomics. Tau ranges from 0 to 1:", styles["Body"]))

story.append(code_block(
    "Tau = sum(1 - x_i / max(x)) / (n - 1)\n\n"
    "where:\n"
    "  x_i = expression of the gene in tissue i\n"
    "  n   = number of tissues\n"
    "  max(x) = highest expression across all tissues"
))

story.append(Paragraph(
    "<b>Tau = 0</b> means the gene is expressed equally in all tissues (ubiquitous/constitutive). "
    "<b>Tau = 1</b> means the gene is expressed in only one tissue (perfectly tissue-specific). "
    "We use a threshold of <b>Tau &ge; 0.6</b> to classify a gene as tissue-specific; genes "
    "below this threshold are labeled 'constitutive' (broadly expressed).", styles["Body"]))

story.append(callout_box(
    "INTERMEDIATE NOTE: Why Tau and not just 'highest expression tissue'?",
    "Simply assigning a gene to the tissue where it is most expressed would be misleading for "
    "housekeeping genes that are expressed everywhere at similar levels. The Tau index first "
    "asks 'is this gene actually tissue-specific?' before making an assignment. A gene with "
    "Tau = 0.15 (expressed nearly everywhere) should not be assigned to any single tissue, "
    "even if one tissue happens to have marginally higher expression. Tau separates truly "
    "tissue-specific genes from broadly expressed ones."
))

story.append(callout_box(
    "ADVANCED NOTE: Limitations of the atlas approach",
    "This method infers tissue relevance from where genes are normally expressed in unstressed "
    "plants. It does not measure where the radiation response actually occurs. A gene that is "
    "normally root-specific might be induced in leaves by radiation. The atlas tells us the "
    "'default' tissue context of each gene, not the treatment-specific context. For direct "
    "tissue-level radiation responses, you would need to irradiate and harvest tissues "
    "separately (as in some newer experiments). Also, the ATH1 array covers ~21,000 of "
    "~27,000 Arabidopsis genes; genes not on the array are flagged 'unannotated'."
))

story.append(PageBreak())

print("Sections 1-3 added. Continuing...")


Sections 1-3 added. Continuing...


Add section 4 code walkthrough to PDF

In [47]:

# ── SECTION 4: Code Walkthrough ────────────────────────────────────────────────
story.append(Paragraph("4. Code Walkthrough: Section-by-Section Explanation", styles["SectionHead"]))
story.append(divider())

story.append(Paragraph(
    "This section walks through the R script (<font face='Courier'>tissue_specific_deg_analysis.R</font>) "
    "block by block. The script is organized into 13 numbered sections. We explain what each "
    "section does, why it does it that way, and what the key R functions mean.", styles["Body"]))

# Section 0
story.append(Paragraph("4.1 Section 0: Package Installation and Loading", styles["SubHead"]))
story.append(Paragraph(
    "The script begins by checking whether all required R packages are installed and loading them. "
    "The key packages are:", styles["Body"]))

pkg_table = make_table(
    ["Package", "Purpose"],
    [
        ["GEOquery", "Downloads GEO datasets from NCBI"],
        ["ath1121501.db", "Maps ATH1 microarray probe IDs to Arabidopsis gene IDs"],
        ["AnnotationDbi", "Interface for querying annotation databases"],
        ["ComplexHeatmap", "Creates publication-quality heatmaps"],
        ["circlize", "Color mapping for heatmaps"],
        ["ggplot2", "General-purpose plotting (bar charts, volcano plots)"],
        ["UpSetR", "Creates UpSet diagrams for set intersection visualization"],
        ["gridExtra", "Arranges multiple plots on one page"],
    ],
    [120, 340]
)
story.append(pkg_table)
story.append(Spacer(1, 8))

story.append(code_block(
    '# Install missing packages automatically\n'
    'for (pkg in required_cran) {\n'
    '  if (!requireNamespace(pkg, quietly = TRUE))\n'
    '    install.packages(pkg, repos = "https://cloud.r-project.org")\n'
    '}'
))

story.append(Paragraph(
    "The <font face='Courier'>requireNamespace()</font> function checks if a package is available "
    "without loading it. If it returns FALSE, we install the package. This makes the script "
    "reproducible on any machine.", styles["Body"]))

# Section 1
story.append(Paragraph("4.2 Section 1: Loading and Validating the DEG Dataset", styles["SubHead"]))
story.append(Paragraph(
    "We read the CSV file and rename the unnamed first and last columns. The first column "
    "contains gene IDs; the last contains the yes/no DEG flag.", styles["Body"]))

story.append(code_block(
    'deg <- read.csv(deg_file, header = TRUE, stringsAsFactors = FALSE)\n'
    'colnames(deg)[1] <- "gene_id"\n'
    'colnames(deg)[ncol(deg)] <- "deg_flag"'
))

story.append(Paragraph(
    "We also validate the AGI gene ID format. Nuclear genes follow the pattern AT{1-5}Gnnnnn. "
    "Genes starting with ATCG (chloroplast) or ATMG (mitochondria) are organellar genes that "
    "won't be in the AtGenExpress nuclear atlas — they are flagged as 'unannotated' later.", styles["Body"]))

story.append(code_block(
    '# Check for nuclear AGI IDs (AT1G through AT5G)\n'
    'valid_nuclear <- grepl("^AT[1-5]G[0-9]{5}$", deg$gene_id)\n'
    '# Add regulation direction\n'
    'deg$regulation <- ifelse(deg$deg_flag == "yes",\n'
    '  ifelse(deg$log2FoldChange > 0, "up", "down"), "non_DEG")'
))

story.append(callout_box(
    "BEGINNER'S CORNER: What does grepl() do?",
    "<font face='Courier'>grepl()</font> checks whether each string matches a pattern. The pattern "
    "<font face='Courier'>^AT[1-5]G[0-9]{5}$</font> means: start (^) with 'AT', then a digit 1-5, "
    "then 'G', then exactly 5 digits, then end ($). This matches valid nuclear gene IDs like "
    "AT3G27630 but rejects ATCG00090 (chloroplast) or ATMG01090 (mitochondria)."
))

# Section 2-3
story.append(Paragraph("4.3 Sections 2-3: Downloading AtGenExpress and Building Tissue Labels", styles["SubHead"]))
story.append(Paragraph(
    "We download six GEO series using <font face='Courier'>getGEO()</font> from the GEOquery package. "
    "Each series returns an ExpressionSet object containing the expression matrix and sample metadata. "
    "We then extract tissue labels from the sample metadata and map them to a hierarchical structure "
    "of broad organs and sub-tissues.", styles["Body"]))

story.append(code_block(
    '# Download a GEO series\n'
    'gse <- getGEO("GSE5629", destdir = geodir, getGPL = TRUE)\n\n'
    '# Extract tissue from the characteristics column\n'
    'extracted <- sub(".*Tissue:\\\\s*", "", vals)  # remove "Tissue: " prefix\n'
    'extracted <- sub("\\\\s*;.*", "", extracted)   # remove anything after ;'
))

story.append(Paragraph(
    "The <font face='Courier'>build_tissue_map()</font> function uses a series of if/else statements "
    "with <font face='Courier'>grepl()</font> to classify each raw tissue label (e.g., 'flowers stage "
    "12, sepals') into a broad organ ('Flower') and sub-tissue ('sepal'). This is a rule-based "
    "mapping because the AtGenExpress tissue labels are free-text descriptions, not controlled "
    "vocabulary.", styles["Body"]))

# Section 4
story.append(Paragraph("4.4 Section 4: Mapping Microarray Probes to Gene IDs", styles["SubHead"]))
story.append(Paragraph(
    "The ATH1 microarray uses probe set IDs (e.g., 244901_at), not gene names. We use the "
    "<font face='Courier'>ath1121501.db</font> annotation package to map probes to AGI gene IDs "
    "via the TAIR column. Some genes have multiple probes; we keep the one with the highest "
    "average expression.", styles["Body"]))

story.append(code_block(
    '# Map probes to AGI gene IDs\n'
    'agi_map <- AnnotationDbi::select(ath1121501.db,\n'
    '  keys = probe_ids, columns = c("PROBEID", "TAIR"), keytype = "PROBEID")\n\n'
    '# Keep only nuclear AGI IDs\n'
    'agi_map <- agi_map[grepl("^AT[1-5]G[0-9]{5}", agi_map$TAIR), ]'
))

# Section 5
story.append(Paragraph("4.5 Section 5: Building Tissue-Level Expression Matrices", styles["SubHead"]))
story.append(Paragraph(
    "We average expression across all biological replicates within each tissue group, producing "
    "one expression value per gene per tissue. This is done at both the sub-tissue level (33 tissues) "
    "and the broad-organ level (5 organs).", styles["Body"]))

story.append(code_block(
    '# Average expression per sub-tissue\n'
    'sub_tissue_expr <- sapply(sub_tissues, function(st) {\n'
    '  samples_st <- all_samples$sample_id[all_samples$sub_tissue == st]\n'
    '  rowMeans(expr_mapped[, samples_st, drop = FALSE], na.rm = TRUE)\n'
    '})'
))

# Section 6
story.append(Paragraph("4.6 Section 6: Computing the Tau Index", styles["SubHead"]))
story.append(Paragraph(
    "The <font face='Courier'>compute_tau()</font> function applies the Tau formula to each gene "
    "(each row of the expression matrix). The <font face='Courier'>apply()</font> function with "
    "MARGIN = 1 iterates over rows.", styles["Body"]))

story.append(code_block(
    'compute_tau <- function(expr_matrix) {\n'
    '  expr_matrix[expr_matrix < 0] <- 0  # clamp negatives\n'
    '  apply(expr_matrix, 1, function(x) {\n'
    '    mx <- max(x, na.rm = TRUE)\n'
    '    if (mx == 0) return(NA)\n'
    '    n <- sum(!is.na(x))\n'
    '    sum(1 - (x / mx), na.rm = TRUE) / (n - 1)\n'
    '  })\n'
    '}'
))

story.append(Paragraph(
    "After computing Tau, we classify each gene as 'tissue_specific' (Tau &ge; 0.6), "
    "'constitutive' (Tau &lt; 0.6), or 'unannotated' (not in the atlas). We also identify "
    "the predominant tissue for each gene (the tissue with the highest expression).", styles["Body"]))

# Section 7
story.append(Paragraph("4.7 Section 7: Merging DEG Data with Tissue Annotations", styles["SubHead"]))
story.append(Paragraph(
    "We merge the DESeq2 results with the tissue annotations using <font face='Courier'>merge()</font>, "
    "which joins two data frames by a shared column (gene_id). Genes not in the atlas are flagged "
    "as 'unannotated'.", styles["Body"]))

# Sections 8-9
story.append(Paragraph("4.8 Sections 8-9: Exporting CSVs and Summary Table", styles["SubHead"]))
story.append(Paragraph(
    "We create a hierarchical folder structure and write one CSV per tissue. The summary table "
    "aggregates DEG counts, up/down breakdown, and statistics per tissue.", styles["Body"]))

# Sections 10-13
story.append(Paragraph("4.9 Sections 10-13: Generating Visualizations", styles["SubHead"]))
story.append(Paragraph(
    "Four figures are generated: (1) a stacked bar chart of DEG counts per organ, (2) volcano "
    "plots per organ, (3) a heatmap of the top 50 DEGs across all sub-tissues, and (4) an UpSet "
    "diagram showing organ expression overlaps. Each uses ggplot2 or ComplexHeatmap with "
    "colorblind-friendly palettes.", styles["Body"]))

story.append(PageBreak())

print("Section 4 added. Continuing...")


Section 4 added. Continuing...


Add sections 5-8 and build the PDF

In [49]:

# ── SECTION 5: Understanding the Outputs ───────────────────────────────────────
story.append(Paragraph("5. Understanding the Outputs: CSVs, Summary Table, and Figures", styles["SectionHead"]))
story.append(divider())

story.append(Paragraph("5.1 Hierarchical CSV Files", styles["SubHead"]))
story.append(Paragraph(
    "The script creates a folder called <font face='Courier'>tissue_specific_degs/</font> with "
    "the following structure:", styles["Body"]))

story.append(code_block(
    "tissue_specific_degs/\n"
    "  ├── Flower/\n"
    "  │   ├── Flower_all_DEGs.csv        (all flower-specific DEGs)\n"
    "  │   ├── carpel_DEGs.csv            (DEGs specific to carpels)\n"
    "  │   ├── petal_DEGs.csv\n"
    "  │   ├── sepal_DEGs.csv\n"
    "  │   ├── stamen_DEGs.csv\n"
    "  │   ├── mature_pollen_DEGs.csv\n"
    "  │   └── ... (one per flower sub-tissue)\n"
    "  ├── Leaf_Shoot/\n"
    "  │   ├── Leaf_Shoot_all_DEGs.csv\n"
    "  │   ├── rosette_leaf_DEGs.csv\n"
    "  │   ├── senescing_leaf_DEGs.csv\n"
    "  │   └── ...\n"
    "  ├── Root/\n"
    "  │   └── root_DEGs.csv\n"
    "  ├── Seed_Silique/\n"
    "  │   └── ... (seed and silique sub-tissues)\n"
    "  ├── Seedling/\n"
    "  │   └── ...\n"
    "  ├── constitutive_DEGs.csv          (broadly expressed DEGs)\n"
    "  ├── unannotated_DEGs.csv           (organellar / not on ATH1 array)\n"
    "  └── all_degs_with_tissue_annotation.csv  (master file: all 23,573 genes)"
))

story.append(Paragraph(
    "Each CSV contains the original DESeq2 columns plus the tissue annotations:", styles["Body"]))

output_table = make_table(
    ["Added Column", "Description"],
    [
        ["regulation", "up, down, or non_DEG"],
        ["tau_subtissue", "Tau index at sub-tissue level (0-1)"],
        ["tau_broad", "Tau index at broad-organ level (0-1)"],
        ["specificity", "tissue_specific, constitutive, or unannotated"],
        ["predominant_broad_organ", "Organ with highest expression (e.g., Flower)"],
        ["predominant_subtissue", "Sub-tissue with highest expression (e.g., sepal)"],
    ],
    [160, 300]
)
story.append(output_table)
story.append(Spacer(1, 8))

story.append(Paragraph("5.2 Summary Table (tissue_deg_summary.csv)", styles["SubHead"]))
story.append(Paragraph(
    "This table provides a bird's-eye view of the tissue distribution of radiation-responsive DEGs. "
    "Here are the key results from this dataset:", styles["Body"]))

summary_table = make_table(
    ["Broad Organ", "Total DEGs", "Up", "Down", "Median log2FC"],
    [
        ["Leaf_Shoot", "1,399", "727", "672", "+0.22"],
        ["Flower", "1,192", "677", "515", "+0.22"],
        ["Seed_Silique", "609", "437", "172", "+0.24"],
        ["Root", "432", "154", "278", "-0.25"],
        ["Seedling", "72", "54", "18", "+0.69"],
        ["Constitutive", "2,115", "1,055", "1,060", "-0.06"],
        ["Unannotated", "1,123", "670", "453", "+0.21"],
        ["TOTAL", "6,942", "3,719", "3,223", "+0.12"],
    ],
    [100, 80, 60, 60, 100]
)
story.append(summary_table)
story.append(Spacer(1, 8))

story.append(Paragraph(
    "A striking pattern emerges: <b>root-specific DEGs are predominantly downregulated</b> "
    "(278 down vs 154 up, median log2FC = -0.25), while <b>seed/silique and seedling DEGs are "
    "predominantly upregulated</b>. This suggests radiation suppresses root gene expression more "
    "than it enhances it, while developmental and reproductive tissues show more activation.", styles["Body"]))

story.append(Paragraph("5.3 Figures", styles["SubHead"]))

story.append(Paragraph("<b>Figure 1: Bar Chart (fig1_deg_counts_by_organ.png)</b> — "
    "A stacked bar chart showing the number of upregulated (blue) and downregulated (orange) "
    "DEGs in each broad organ. This gives a quick visual overview of which tissues have the "
    "most radiation-responsive genes and whether the response is primarily activation or "
    "suppression.", styles["Body"]))

story.append(Paragraph("<b>Figure 2: Volcano Plots (fig2_volcano_plots_by_organ.png)</b> — "
    "Five panels (one per organ), each showing log2FoldChange on the x-axis and -log10(padj) "
    "on the y-axis. Points above the dashed line are statistically significant. Blue points "
    "are upregulated DEGs, orange are downregulated. Grey points are constitutive genes shown "
    "for context. Volcano plots help you see both the magnitude and significance of expression "
    "changes simultaneously.", styles["Body"]))

story.append(Paragraph("<b>Figure 3: Heatmap (fig3_heatmap_top_degs.png)</b> — "
    "Shows the tissue expression patterns of the 50 most significant DEGs across all 33 "
    "sub-tissues. Each row is a gene (z-scored expression), each column is a sub-tissue. "
    "The top annotation bar colors sub-tissues by broad organ. The right annotation shows "
    "whether each DEG is up- or downregulated in response to radiation. This reveals which "
    "tissues the most significant radiation genes normally operate in.", styles["Body"]))

story.append(Paragraph("<b>Figure 4: UpSet Diagram (fig4_upset_organ_overlaps.png)</b> — "
    "Shows how many DEGs are expressed above their median level in each combination of organs. "
    "The bar chart on the left shows the total number of DEGs expressed in each organ. The "
    "main bar chart shows intersection sizes — how many DEGs are co-expressed across multiple "
    "organs. Connected dots indicate which organs are in each intersection. This helps identify "
    "DEGs that are broadly active vs. organ-restricted.", styles["Body"]))

story.append(PageBreak())

# ── SECTION 6: How to Run the Code ─────────────────────────────────────────────
story.append(Paragraph("6. How to Run the Code: Prerequisites and Execution", styles["SectionHead"]))
story.append(divider())

story.append(Paragraph("6.1 Prerequisites", styles["SubHead"]))
story.append(Paragraph(
    "You need R (version 4.0 or later) with internet access (for downloading GEO data). "
    "The script will automatically install any missing packages.", styles["Body"]))

story.append(Paragraph("6.2 Step-by-Step Instructions", styles["SubHead"]))

story.append(Paragraph("<b>Step 1:</b> Place the DEG CSV file and the R script in the same directory.", styles["Body"]))
story.append(code_block(
    "# Your directory should look like:\n"
    "# /your_project/\n"
    "#   ├── DEG_OSD498_510_radiation_effect.csv\n"
    "#   └── tissue_specific_deg_analysis.R"
))

story.append(Paragraph("<b>Step 2:</b> Open R or RStudio and set the working directory to "
    "the folder containing your files.", styles["Body"]))
story.append(code_block(
    'setwd("/path/to/your_project")'
))

story.append(Paragraph("<b>Step 3:</b> Run the script.", styles["Body"]))
story.append(code_block(
    '# In RStudio: click "Source" or press Cmd/Ctrl + Shift + S\n'
    '# In terminal:\n'
    'Rscript tissue_specific_deg_analysis.R'
))

story.append(Paragraph("<b>Step 4:</b> Wait for the analysis to complete. The GEO download "
    "step may take 5-15 minutes depending on your internet speed. The entire analysis "
    "typically completes in 10-20 minutes.", styles["Body"]))

story.append(Paragraph("<b>Step 5:</b> Check the outputs. All files will be created in your "
    "working directory:", styles["Body"]))

story.append(code_block(
    "# Output files:\n"
    "#   tissue_specific_degs/          (folder with hierarchical CSVs)\n"
    "#   tissue_deg_summary.csv         (summary table)\n"
    "#   fig1_deg_counts_by_organ.png\n"
    "#   fig2_volcano_plots_by_organ.png\n"
    "#   fig3_heatmap_top_degs.png\n"
    "#   fig4_upset_organ_overlaps.png"
))

story.append(callout_box(
    "BEGINNER'S CORNER: What is RStudio?",
    "RStudio is a free, user-friendly interface for R. Instead of typing commands in a plain "
    "terminal, you get a code editor (with syntax highlighting), a console (where R runs), "
    "an environment panel (showing your variables), and a plots panel (showing your figures). "
    "Download it from posit.co. To run a script, open it in RStudio and click the 'Source' "
    "button in the top right of the code editor."
))

story.append(Paragraph("6.3 Customizing the Analysis", styles["SubHead"]))
story.append(Paragraph(
    "You can modify these parameters near the top of the script:", styles["Body"]))

params_table = make_table(
    ["Parameter", "Default", "Effect of Changing"],
    [
        ["deg_file", "DEG_OSD498_510_...", "Point to a different DEG file"],
        ["TAU_THRESHOLD", "0.6", "Lower (e.g., 0.5) = more genes called tissue-specific; Higher (e.g., 0.8) = stricter"],
        ["UP_COLOR / DOWN_COLOR", "#0072B2 / #D55E00", "Change plot colors"],
    ],
    [130, 130, 200]
)
story.append(params_table)

story.append(PageBreak())

# ── SECTION 7: Interpretation Guide ────────────────────────────────────────────
story.append(Paragraph("7. Interpretation Guide: What Tissue-Specific Radiation Responses Mean", styles["SectionHead"]))
story.append(divider())

story.append(Paragraph("7.1 Key Findings from This Dataset", styles["SubHead"]))

story.append(Paragraph(
    "<b>Finding 1: Leaf/shoot tissues harbor the most radiation-responsive DEGs (1,399).</b> "
    "This is consistent with leaves being the primary site of photosynthesis and oxidative "
    "metabolism. Radiation generates reactive oxygen species (ROS), and leaf tissues have "
    "extensive antioxidant systems that may be transcriptionally modulated.", styles["Body"]))

story.append(Paragraph(
    "<b>Finding 2: Root-specific DEGs are predominantly downregulated (64%).</b> "
    "Radiation appears to suppress root gene expression more than it activates it. This could "
    "reflect cell cycle arrest (roots are actively growing and dividing, making them vulnerable "
    "to DNA damage checkpoints) or a reallocation of resources away from root growth.", styles["Body"]))

story.append(Paragraph(
    "<b>Finding 3: Senescing leaf DEGs are almost entirely upregulated (97%, 426/436).</b> "
    "Radiation may accelerate senescence programs in leaves. Senescence involves the ordered "
    "breakdown and remobilization of cellular components, and radiation-induced DNA damage "
    "could trigger premature activation of these pathways.", styles["Body"]))

story.append(Paragraph(
    "<b>Finding 4: Seed/silique DEGs are predominantly upregulated (72%).</b> "
    "Reproductive tissues may activate protective pathways to shield developing embryos from "
    "radiation damage. This is consistent with the importance of protecting the germline in "
    "all organisms.", styles["Body"]))

story.append(Paragraph(
    "<b>Finding 5: 2,115 DEGs are constitutive (broadly expressed).</b> "
    "These are housekeeping-like genes that respond to radiation regardless of tissue context. "
    "They likely include core DNA damage response genes (e.g., PARP, BRCA homologs, DNA "
    "polymerases) that are needed in every cell type.", styles["Body"]))

story.append(Paragraph("7.2 Biological Interpretation Framework", styles["SubHead"]))
story.append(Paragraph(
    "When interpreting tissue-specific DEG results, consider these questions:", styles["Body"]))

story.append(Paragraph("&bull; <b>Is the direction consistent with known biology?</b> "
    "If root growth is known to be inhibited by radiation, downregulation of root-specific "
    "growth genes makes sense.", styles["BulletStyle"]))
story.append(Paragraph("&bull; <b>Are DNA repair genes tissue-specific or constitutive?</b> "
    "Core repair genes should be constitutive (every cell needs to repair DNA). Tissue-specific "
    "repair genes might reflect different damage types or repair strategies in different organs.", styles["BulletStyle"]))
story.append(Paragraph("&bull; <b>Do stress response genes show tissue specificity?</b> "
    "Leaf-specific stress genes might relate to oxidative stress (from photosynthesis + radiation), "
    "while root-specific stress genes might relate to different stress pathways.", styles["BulletStyle"]))
story.append(Paragraph("&bull; <b>Are developmental genes affected?</b> "
    "If flower or seed developmental genes are disrupted, this could have implications for "
    "plant reproduction in space environments.", styles["BulletStyle"]))

story.append(callout_box(
    "ADVANCED NOTE: Caveats and limitations",
    "1. The AtGenExpress atlas measures baseline expression in unstressed plants. Radiation "
    "may alter tissue-specificity. 2. The ATH1 array covers ~21,000 of ~27,000 genes; ~1,123 "
    "DEGs (mostly organellar) could not be annotated. 3. The Tau threshold of 0.6 is a "
    "convention, not a biological absolute — results may shift with different thresholds. "
    "4. The original experiment used whole seedlings; true tissue-level radiation responses "
    "require tissue-specific irradiation and harvest. 5. Cross-platform differences between "
    "microarray (atlas) and RNA-seq (DEG data) may introduce noise."
))

story.append(PageBreak())

# ── SECTION 8: Exercises and Extensions ────────────────────────────────────────
story.append(Paragraph("8. Exercises and Extensions for Further Exploration", styles["SectionHead"]))
story.append(divider())

story.append(Paragraph(
    "This section provides exercises for students to deepen their understanding and suggestions "
    "for extending the analysis.", styles["Body"]))

story.append(Paragraph("8.1 Beginner Exercises", styles["SubHead"]))

story.append(Paragraph("&bull; <b>Exercise 1:</b> Open <font face='Courier'>Root/root_DEGs.csv</font> "
    "in a spreadsheet. Sort by log2FoldChange. What are the top 5 upregulated and top 5 "
    "downregulated root-specific genes? Look up their functions on TAIR "
    "(www.arabidopsis.org).", styles["BulletStyle"]))

story.append(Paragraph("&bull; <b>Exercise 2:</b> Compare the number of DEGs in "
    "<font face='Courier'>Flower/mature_pollen_DEGs.csv</font> vs "
    "<font face='Courier'>Flower/sepal_DEGs.csv</font>. Why might pollen have more "
    "radiation-responsive genes than sepals?", styles["BulletStyle"]))

story.append(Paragraph("&bull; <b>Exercise 3:</b> Look at the bar chart (Figure 1). Which organ "
    "has the most balanced up/down ratio? Which is most skewed? What might this tell you "
    "about how that tissue responds to radiation?", styles["BulletStyle"]))

story.append(Paragraph("8.2 Intermediate Exercises", styles["SubHead"]))

story.append(Paragraph("&bull; <b>Exercise 4:</b> Modify the TAU_THRESHOLD to 0.5 and re-run "
    "the script. How does this change the number of tissue-specific vs constitutive DEGs? "
    "Is the change dramatic or modest? What does this tell you about the sensitivity of "
    "the results to the threshold?", styles["BulletStyle"]))

story.append(Paragraph("&bull; <b>Exercise 5:</b> Extract the constitutive DEGs and perform "
    "Gene Ontology enrichment analysis (using R packages like clusterProfiler or online tools "
    "like DAVID). Are DNA repair genes overrepresented in the constitutive set?", styles["BulletStyle"]))

story.append(Paragraph("&bull; <b>Exercise 6:</b> Create a Venn diagram comparing the DEG sets "
    "from two organs (e.g., Root vs Flower). How many DEGs are tissue-specific to both? "
    "(Hint: since each gene is assigned to one tissue, you'll need to use the expression "
    "threshold approach from the UpSet diagram.)", styles["BulletStyle"]))

story.append(Paragraph("8.3 Advanced Extensions", styles["SubHead"]))

story.append(Paragraph("&bull; <b>Extension 1:</b> Integrate Gene Ontology (GO) enrichment "
    "analysis for each tissue-specific DEG set. Use the <font face='Courier'>clusterProfiler</font> "
    "R package with the Arabidopsis org.At.tair.db annotation. This will tell you which "
    "biological processes are enriched in each tissue's radiation response.", styles["BulletStyle"]))

story.append(Paragraph("&bull; <b>Extension 2:</b> Compare these results with other NASA GeneLab "
    "radiation studies (OSD-502, OSD-508, OSD-658). Do the same tissue-specific patterns hold "
    "across different radiation types (gamma vs HZE vs simulated GCR)?", styles["BulletStyle"]))

story.append(Paragraph("&bull; <b>Extension 3:</b> Use the newer Arabidopsis single-cell atlas "
    "(GSE226097, published in Nature Plants 2025) instead of AtGenExpress for cell-type-level "
    "resolution. This would give you much finer granularity (individual cell types rather than "
    "bulk tissues).", styles["BulletStyle"]))

story.append(Paragraph("&bull; <b>Extension 4:</b> Build a gene regulatory network for each "
    "tissue-specific DEG set using the SCENIC or WGCNA approach. This would identify "
    "transcription factors driving the tissue-specific radiation responses.", styles["BulletStyle"]))

story.append(divider())
story.append(Spacer(1, 20))

story.append(Paragraph(
    "<b>References:</b>", styles["SubHead"]))
story.append(Paragraph(
    "1. Schmid M et al. (2005) A gene expression map of Arabidopsis thaliana development. "
    "Nature Genetics 37:501-506.", styles["Body"]))
story.append(Paragraph(
    "2. Yanai I et al. (2005) Genome-wide midrange transcription profiles reveal expression "
    "level relationships in human tissue specification. Bioinformatics 21:650-659. (Tau index)", styles["Body"]))
story.append(Paragraph(
    "3. Love MI, Huber W, Anders S (2014) Moderated estimation of fold change and dispersion "
    "for RNA-seq data with DESeq2. Genome Biology 15:550.", styles["Body"]))
story.append(Paragraph(
    "4. NASA GeneLab OSD-498 and OSD-510: https://osdr.nasa.gov/bio/repo/data/studies/OSD-498", styles["Body"]))

# ── Build the PDF ──
doc.build(story, onFirstPage=page_header_footer, onLaterPages=page_header_footer)

# Validate
from pypdf import PdfReader
reader = PdfReader(output_path)
page_count = len(reader.pages)
file_size = os.path.getsize(output_path)
first_page_text = reader.pages[0].extract_text()

print(f"PDF generated: {output_path}")
print(f"Pages: {page_count}")
print(f"File size: {file_size:,} bytes")
print(f"First page text length: {len(first_page_text.strip())} chars")
assert page_count >= 2, "Too few pages"
assert file_size > 5000, "File too small"
assert len(first_page_text.strip()) > 0, "No text on first page"
print("Validation passed!")


PDF generated: /mnt/results/instruction_manual_tissue_specific_deg.pdf
Pages: 17
File size: 40,355 bytes
First page text length: 1271 chars
Validation passed!


Load data for statistical analysis

In [51]:

# ── Statistical Analysis: Are tissue-specific radiation responses significant? ──
# 
# Framework:
#   For each broad organ, we have:
#     - N_total = number of genes tissue-specific to that organ (background, from atlas)
#     - N_deg   = number of those genes that are DEGs (observed)
#     - N_non   = N_total - N_deg (non-DEGs)
#
#   H0: DEG status is independent of tissue assignment
#       (i.e., each tissue has the same proportion of its genes becoming DEGs)
#   H1: DEG status depends on tissue (some tissues are enriched/depleted for DEGs)
#
# Tests:
#   1. Chi-square test of independence (overall: is there ANY tissue effect?)
#   2. Fisher's exact test per tissue (is each tissue individually enriched/depleted?)
#   3. Pairwise Fisher's exact tests (is root responding more than shoot? etc.)

suppressPackageStartupMessages({
  library(ggplot2)
})

# Load the annotated data
deg_annotated <- readRDS("/workspace/deg_annotated.rds")
atlas <- readRDS("/workspace/atgenexpress_atlas.rlas")


Warning message:
“package ‘ggplot2’ was built under R version 4.4.3”
Warning message in gzfile(file, "rb"):
“cannot open compressed file '/workspace/atgenexpress_atlas.rlas', probable reason 'No such file or directory'”
Error in gzfile(file, "rb"): cannot open the connection
Traceback:

1. gzfile(file, "rb")
2. .handleSimpleError(function (cnd) 
 . {
 .     watcher$capture_plot_and_output()
 .     cnd <- sanitize_call(cnd)
 .     watcher$push(cnd)
 .     switch(on_error, continue = invokeRestart("eval_continue"), 
 .         stop = invokeRestart("eval_stop"), error = NULL)
 . }, "cannot open the connection", base::quote(gzfile(file, "rb")))

Build contingency table for tissue enrichment test

In [53]:

# ── Statistical Analysis: Are tissue-specific radiation responses significant? ──
suppressPackageStartupMessages({
  library(ggplot2)
})

# Load the annotated data
deg_annotated <- readRDS("/workspace/deg_annotated.rds")
atlas <- readRDS("/workspace/atgenexpress_atlas.rds")
broad_organ_expr <- atlas$broad_organ_expr
sub_tissue_expr <- atlas$sub_tissue_expr

# ── Build the contingency table: tissue × DEG status ───────────────────────────
# Only consider genes that are in the atlas (exclude unannotated)
atlas_genes <- rownames(broad_organ_expr)
deg_in_atlas <- deg_annotated[deg_annotated$gene_id %in% atlas_genes, ]

# For tissue-specific genes only (Tau >= 0.6)
tissue_specific <- deg_in_atlas[deg_in_atlas$specificity == "tissue_specific", ]

cat("=== Tissue-specific genes in atlas ===\n")
cat("Total:", nrow(tissue_specific), "\n\n")

# Contingency table: broad_organ × deg_flag
contingency <- table(tissue_specific$predominant_broad_organ, tissue_specific$deg_flag)
cat("=== Contingency table: Broad Organ × DEG status ===\n")
print(contingency)

# Add row totals and DEG rates
contingency_df <- as.data.frame.matrix(contingency)
contingency_df$total <- rowSums(contingency_df)
contingency_df$deg_rate <- contingency_df$yes / contingency_df$total
contingency_df$organ <- rownames(contingency_df)

cat("\n=== DEG rates per organ ===\n")
print(contingency_df[, c("organ", "yes", "no", "total", "deg_rate")])

# Overall DEG rate (expected under H0)
overall_rate <- sum(contingency_df$yes) / sum(contingency_df$total)
cat("\nOverall DEG rate (expected under H0):", round(overall_rate, 4), "\n")


=== Tissue-specific genes in atlas ===
Total: 10622 

=== Contingency table: Broad Organ × DEG status ===
              
                 no  yes
  Flower       2585 1192
  Leaf_Shoot   1889 1399
  Root          763  432
  Seed_Silique 1611  609
  Seedling       70   72

=== DEG rates per organ ===
                    organ  yes   no total  deg_rate
Flower             Flower 1192 2585  3777 0.3155944
Leaf_Shoot     Leaf_Shoot 1399 1889  3288 0.4254866
Root                 Root  432  763  1195 0.3615063
Seed_Silique Seed_Silique  609 1611  2220 0.2743243
Seedling         Seedling   72   70   142 0.5070423

Overall DEG rate (expected under H0): 0.3487 


Run chi-square test of independence

In [55]:

# ── TEST 1: Chi-square test of independence ────────────────────────────────────
# H0: DEG status is independent of tissue assignment
# H1: DEG status depends on tissue (some tissues enriched/depleted for DEGs)

cat("=== TEST 1: Chi-square test of independence ===\n")
chi_result <- chisq.test(contingency)
print(chi_result)

cat("\nExpected counts under H0:\n")
print(round(chi_result$expected, 1))

cat("\nResiduals (observed - expected) / sqrt(expected):\n")
cat("Positive = more DEGs than expected (enriched)\n")
cat("Negative = fewer DEGs than expected (depleted)\n")
print(round(chi_result$residuals[, "yes"], 3))

cat("\n=== Interpretation ===\n")
if (chi_result$p.value < 0.05) {
  cat("p-value =", format(chi_result$p.value, digits=4), "< 0.05\n")
  cat("REJECT H0: Tissue assignment and DEG status are NOT independent.\n")
  cat("There IS a statistically significant tissue-specific radiation response.\n")
} else {
  cat("p-value =", format(chi_result$p.value, digits=4), ">= 0.05\n")
  cat("FAIL TO REJECT H0: No evidence of tissue-specific response.\n")
}


=== TEST 1: Chi-square test of independence ===

	Pearson's Chi-squared test

data:  contingency
X-squared = 174.2, df = 4, p-value < 2.2e-16


Expected counts under H0:
              
                   no    yes
  Flower       2459.9 1317.1
  Leaf_Shoot   2141.4 1146.6
  Root          778.3  416.7
  Seed_Silique 1445.9  774.1
  Seedling       92.5   49.5

Residuals (observed - expected) / sqrt(expected):
Positive = more DEGs than expected (enriched)
Negative = fewer DEGs than expected (depleted)
      Flower   Leaf_Shoot         Root Seed_Silique     Seedling 
      -3.446        7.455        0.749       -5.935        3.195 

=== Interpretation ===
p-value = 1.312e-36 < 0.05
REJECT H0: Tissue assignment and DEG status are NOT independent.
There IS a statistically significant tissue-specific radiation response.


Run per-tissue Fisher's exact tests

In [57]:

# ── TEST 2: Per-tissue Fisher's exact test (each tissue vs all others) ──────────
# For each organ, test whether its DEG rate differs from the rest.
# This gives a per-organ enrichment/depletion p-value.

cat("=== TEST 2: Per-tissue Fisher's exact test (organ vs rest) ===\n\n")

organs <- rownames(contingency)
fisher_results <- data.frame(
  organ = character(),
  deg_in_organ = integer(),
  total_in_organ = integer(),
  deg_rate_organ = numeric(),
  deg_rate_rest = numeric(),
  odds_ratio = numeric(),
  p_value = numeric(),
  direction = character(),
  stringsAsFactors = FALSE
)

for (org in organs) {
  # 2x2 table: this organ vs all others × DEG vs non-DEG
  org_yes <- contingency[org, "yes"]
  org_no  <- contingency[org, "no"]
  rest_yes <- sum(contingency[, "yes"]) - org_yes
  rest_no  <- sum(contingency[, "no"]) - org_no
  
  mat <- matrix(c(org_yes, org_no, rest_yes, rest_no), nrow = 2)
  ft <- fisher.test(mat, alternative = "two.sided")
  
  deg_rate_org <- org_yes / (org_yes + org_no)
  deg_rate_rest <- rest_yes / (rest_yes + rest_no)
  direction <- ifelse(deg_rate_org > deg_rate_rest, "enriched", "depleted")
  
  fisher_results <- rbind(fisher_results, data.frame(
    organ = org,
    deg_in_organ = org_yes,
    total_in_organ = org_yes + org_no,
    deg_rate_organ = round(deg_rate_org, 4),
    deg_rate_rest = round(deg_rate_rest, 4),
    odds_ratio = round(ft$estimate, 3),
    p_value = ft$p.value,
    direction = direction,
    stringsAsFactors = FALSE
  ))
}

# Bonferroni correction for 5 tests
fisher_results$padj <- p.adjust(fisher_results$p_value, method = "bonferroni")
fisher_results$significant <- ifelse(fisher_results$padj < 0.05, "YES", "no")

cat("Per-tissue Fisher's exact test results:\n")
print(fisher_results, row.names = FALSE)

cat("\n=== Interpretation ===\n")
for (i in 1:nrow(fisher_results)) {
  r <- fisher_results[i, ]
  sig <- if (r$padj < 0.05) "SIGNIFICANT" else "not significant"
  cat(sprintf("  %s: %s (OR=%.2f, padj=%.2e) — DEG rate %.1f%% vs %.1f%% in rest [%s]\n",
              r$organ, r$direction, r$odds_ratio, r$padj,
              r$deg_rate_organ * 100, r$deg_rate_rest * 100, sig))
}


=== TEST 2: Per-tissue Fisher's exact test (organ vs rest) ===

Per-tissue Fisher's exact test results:
        organ deg_in_organ total_in_organ deg_rate_organ deg_rate_rest
       Flower         1192           3777         0.3156        0.3670
   Leaf_Shoot         1399           3288         0.4255        0.3143
         Root          432           1195         0.3615        0.3471
 Seed_Silique          609           2220         0.2743        0.3684
     Seedling           72            142         0.5070        0.3466
 odds_ratio      p_value direction         padj significant
      0.795 1.045516e-07  depleted 5.227580e-07         YES
      1.616 2.708947e-28  enriched 1.354473e-27         YES
      1.065 3.337991e-01  enriched 1.000000e+00          no
      0.648 5.411587e-17  depleted 2.705794e-16         YES
      1.939 1.205900e-04  enriched 6.029502e-04         YES

=== Interpretation ===
  Flower: depleted (OR=0.80, padj=5.23e-07) — DEG rate 31.6% vs 36.7% in rest [SIGNIFI

Run pairwise Fisher's exact tests between organs

In [59]:

# ── TEST 3: Pairwise Fisher's exact tests (organ vs organ) ─────────────────────
# Directly answers: "Are roots responding more than shoots?"
# With Bonferroni correction for C(5,2)=10 pairwise comparisons

cat("=== TEST 3: Pairwise Fisher's exact tests (organ vs organ) ===\n\n")

organs <- rownames(contingency)
n_organs <- length(organs)
n_pairs <- choose(n_organs, 2)

pairwise_results <- data.frame(
  organ1 = character(),
  organ2 = character(),
  rate1 = numeric(),
  rate2 = numeric(),
  odds_ratio = numeric(),
  p_value = numeric(),
  stringsAsFactors = FALSE
)

pair_idx <- 1
for (i in 1:(n_organs - 1)) {
  for (j in (i + 1):n_organs) {
    org1 <- organs[i]
    org2 <- organs[j]
    
    # 2x2 table: org1 × org2, DEG × non-DEG
    mat <- matrix(c(
      contingency[org1, "yes"], contingency[org1, "no"],
      contingency[org2, "yes"], contingency[org2, "no"]
    ), nrow = 2)
    
    ft <- fisher.test(mat, alternative = "two.sided")
    
    rate1 <- contingency[org1, "yes"] / sum(contingency[org1, ])
    rate2 <- contingency[org2, "yes"] / sum(contingency[org2, ])
    
    pairwise_results <- rbind(pairwise_results, data.frame(
      organ1 = org1,
      organ2 = org2,
      rate1 = round(rate1, 4),
      rate2 = round(rate2, 4),
      odds_ratio = round(ft$estimate, 3),
      p_value = ft$p.value,
      stringsAsFactors = FALSE
    ))
    pair_idx <- pair_idx + 1
  }
}

# Bonferroni correction for 10 pairwise tests
pairwise_results$padj <- p.adjust(pairwise_results$p_value, method = "bonferroni")
pairwise_results$higher <- ifelse(pairwise_results$rate1 > pairwise_results$rate2,
                                   paste0(pairwise_results$organ1, " > ", pairwise_results$organ2),
                                   paste0(pairwise_results$organ2, " > ", pairwise_results$organ1))
pairwise_results$significant <- ifelse(pairwise_results$padj < 0.05, "YES", "no")

# Sort by p-value
pairwise_results <- pairwise_results[order(pairwise_results$p_value), ]

cat("Pairwise comparisons (sorted by p-value):\n")
print(pairwise_results, row.names = FALSE)

cat("\n=== Key pairwise findings ===\n")
sig_pairs <- pairwise_results[pairwise_results$padj < 0.05, ]
if (nrow(sig_pairs) > 0) {
  for (i in 1:nrow(sig_pairs)) {
    r <- sig_pairs[i, ]
    cat(sprintf("  %s: OR=%.2f, padj=%.2e [%s]\n",
                r$higher, r$odds_ratio, r$padj, r$significant))
  }
} else {
  cat("  No significant pairwise differences after Bonferroni correction.\n")
}


=== TEST 3: Pairwise Fisher's exact tests (organ vs organ) ===

Pairwise comparisons (sorted by p-value):
       organ1       organ2  rate1  rate2 odds_ratio      p_value         padj
   Leaf_Shoot Seed_Silique 0.4255 0.2743      1.959 1.232663e-30 1.232663e-29
       Flower   Leaf_Shoot 0.3156 0.4255      0.623 1.417021e-21 1.417021e-20
 Seed_Silique     Seedling 0.2743 0.5070      0.368 1.620881e-08 1.620881e-07
         Root Seed_Silique 0.3615 0.2743      1.497 1.710751e-07 1.710751e-06
       Flower     Seedling 0.3156 0.5070      0.448 3.771757e-06 3.771757e-05
   Leaf_Shoot         Root 0.4255 0.3615      1.308 1.181643e-04 1.181643e-03
       Flower Seed_Silique 0.3156 0.2743      1.220 7.922085e-04 7.922085e-03
         Root     Seedling 0.3615 0.5070      0.551 9.349684e-04 9.349684e-03
       Flower         Root 0.3156 0.3615      0.814 3.319972e-03 3.319972e-02
   Leaf_Shoot     Seedling 0.4255 0.5070      0.720 5.703960e-02 5.703960e-01
                    higher significa

Test direction-specific tissue enrichment

In [61]:

# ── TEST 4: Direction-specific enrichment (up vs down separately) ──────────────
# The above tests ask "are there more DEGs?" but not "are there more UP or DOWN?"
# We should also test whether the up/down ratio differs by tissue.

cat("=== TEST 4: Tissue-specific direction of response (up vs down) ===\n\n")

# For tissue-specific DEGs only, test if up/down ratio differs across organs
tissue_specific_degs <- deg_annotated[
  deg_annotated$specificity == "tissue_specific" & 
  deg_annotated$deg_flag == "yes", ]

direction_table <- table(tissue_specific_degs$predominant_broad_organ, 
                         tissue_specific_degs$regulation)
cat("Contingency table: Organ × Regulation direction (DEGs only)\n")
print(direction_table)

cat("\nChi-square test: Is up/down ratio independent of tissue?\n")
chi_dir <- chisq.test(direction_table)
print(chi_dir)

cat("\n=== Per-organ up/down ratios ===\n")
for (org in rownames(direction_table)) {
  n_up <- direction_table[org, "up"]
  n_down <- direction_table[org, "down"]
  ratio <- n_up / (n_up + n_down)
  cat(sprintf("  %s: %d up, %d down, %%up = %.1f%%\n", org, n_up, n_down, ratio * 100))
}

cat("\n=== Interpretation ===\n")
if (chi_dir$p.value < 0.05) {
  cat("p-value =", format(chi_dir$p.value, digits=4), "< 0.05\n")
  cat("REJECT H0: The up/down regulation ratio DIFFERS significantly across tissues.\n")
  cat("Different tissues show qualitatively different radiation responses.\n")
} else {
  cat("p-value =", format(chi_dir$p.value, digits=4), ">= 0.05\n")
  cat("No evidence that up/down ratio differs across tissues.\n")
}


=== TEST 4: Tissue-specific direction of response (up vs down) ===

Contingency table: Organ × Regulation direction (DEGs only)
              
               down  up
  Flower        511 681
  Leaf_Shoot    674 725
  Root          278 154
  Seed_Silique  239 370
  Seedling        8  64

Chi-square test: Is up/down ratio independent of tissue?

	Pearson's Chi-squared test

data:  direction_table
X-squared = 112.32, df = 4, p-value < 2.2e-16


=== Per-organ up/down ratios ===
  Flower: 681 up, 511 down, %up = 57.1%
  Leaf_Shoot: 725 up, 674 down, %up = 51.8%
  Root: 154 up, 278 down, %up = 35.6%
  Seed_Silique: 370 up, 239 down, %up = 60.8%
  Seedling: 64 up, 8 down, %up = 88.9%

=== Interpretation ===
p-value = 2.333e-23 < 0.05
REJECT H0: The up/down regulation ratio DIFFERS significantly across tissues.
Different tissues show qualitatively different radiation responses.


Generate tissue enrichment forest plot

In [63]:

# ── Visualization: DEG rate by organ with significance annotations ─────────────
# A forest-plot-style figure showing DEG rates with confidence intervals

suppressPackageStartupMessages({
  library(ggplot2)
})

# Build plot data from Fisher's test results
plot_data <- fisher_results
plot_data$organ <- factor(plot_data$organ, 
                           levels = plot_data$organ[order(plot_data$deg_rate_organ)])

# Calculate 95% CI for the DEG rate (Wilson interval)
plot_data$ci_lower <- sapply(1:nrow(plot_data), function(i) {
  binom.test(plot_data$deg_in_organ[i], plot_data$total_in_organ[i])$conf.int[1]
})
plot_data$ci_upper <- sapply(1:nrow(plot_data), function(i) {
  binom.test(plot_data$deg_in_organ[i], plot_data$total_in_organ[i])$conf.int[2]
})

# Significance labels
plot_data$sig_label <- ifelse(plot_data$padj < 0.001, "***",
                       ifelse(plot_data$padj < 0.01, "**",
                       ifelse(plot_data$padj < 0.05, "*", "ns")))

# Overall rate line
overall_rate <- sum(contingency_df$yes) / sum(contingency_df$total)

p <- ggplot(plot_data, aes(x = deg_rate_organ, y = organ, color = direction)) +
  geom_vline(xintercept = overall_rate, linetype = "dashed", color = "grey50", linewidth = 0.5) +
  annotate("text", x = overall_rate + 0.01, y = 5.5, label = sprintf("Overall rate: %.1f%%", overall_rate * 100),
           hjust = 0, size = 3, color = "grey40") +
  geom_errorbarh(aes(xmin = ci_lower, xmax = ci_upper), height = 0.2, linewidth = 0.8) +
  geom_point(size = 4) +
  geom_text(aes(label = sprintf("%.1f%% %s", deg_rate_organ * 100, sig_label)),
            hjust = -0.2, vjust = 0.5, size = 3.5, color = "black") +
  scale_color_manual(values = c("enriched" = "#0072B2", "depleted" = "#D55E00"),
                     labels = c("enriched" = "Enriched for DEGs", "depleted" = "Depleted for DEGs"),
                     name = "") +
  labs(
    title = "Tissue-Specific Radiation Response: DEG Rate by Organ",
    subtitle = "Fisher's exact test (organ vs rest), Bonferroni-corrected  |  *** p<0.001, ** p<0.01, * p<0.05",
    x = "Proportion of tissue-specific genes that are DEGs",
    y = "Broad Organ"
  ) +
  theme_minimal(base_size = 13) +
  theme(
    plot.title = element_text(face = "bold", size = 15),
    plot.subtitle = element_text(size = 10, color = "grey40"),
    legend.position = "top",
    panel.grid.minor = element_blank(),
    axis.text.y = element_text(face = "bold")
  ) +
  xlim(0.15, 0.65)

ggsave("/mnt/results/fig5_tissue_enrichment_test.png", p, width = 9, height = 5, dpi = 300, bg = "white")
cat("Saved fig5_tissue_enrichment_test.png\n")


Warning message:
“`geom_errorbarh()` was deprecated in ggplot2 4.0.0.
ℹ Please use the `orientation` argument of `geom_errorbar()` instead.”
Error in `scale_y_continuous()`:
! Discrete value supplied to a continuous scale.
ℹ Example values: Flower, Leaf_Shoot, Root, Seed_Silique, and Seedling.
Traceback:

1. lapply(plot, grid.draw)
2. FUN(X[[i]], ...)
3. `grid.draw.ggplot2::ggplot`(X[[i]], ...)
4. print(x)
5. `print.ggplot2::ggplot`(x)
6. ggplot_build(x)
7. `ggplot_build.ggplot2::ggplot`(x)
8. layout$train_position(data, scale_x(), scale_y())
9. train_position(..., self = self)
10. self$facet$train_scales(self$panel_scales_x, self$panel_scales_y, 
  .     layout, data, self$facet_params)
11. train_scales(...)
12. scale_apply(layer_data, y_vars, "train", SCALE_Y, y_scales)
13. lapply(vars, function(var) {
  .     pieces <- lapply(seq_along(scales), function(i) {
  .         scales[[i]][[method]](vec_slice(data[[var]], scale_index[[i]]))
  .     })
  .     pieces[lengths(pieces) == 0] <-

Fix and regenerate enrichment forest plot

In [65]:

# ── Fix: use scale_x_continuous instead of xlim ────────────────────────────────
p <- ggplot(plot_data, aes(x = deg_rate_organ, y = organ, color = direction)) +
  geom_vline(xintercept = overall_rate, linetype = "dashed", color = "grey50", linewidth = 0.5) +
  annotate("text", x = overall_rate + 0.01, y = 5.5, label = sprintf("Overall rate: %.1f%%", overall_rate * 100),
           hjust = 0, size = 3, color = "grey40") +
  geom_errorbarh(aes(xmin = ci_lower, xmax = ci_upper), height = 0.2, linewidth = 0.8) +
  geom_point(size = 4) +
  geom_text(aes(label = sprintf("%.1f%% %s", deg_rate_organ * 100, sig_label)),
            hjust = -0.2, vjust = 0.5, size = 3.5, color = "black") +
  scale_color_manual(values = c("enriched" = "#0072B2", "depleted" = "#D55E00"),
                     labels = c("enriched" = "Enriched for DEGs", "depleted" = "Depleted for DEGs"),
                     name = "") +
  scale_x_continuous(limits = c(0.15, 0.70)) +
  labs(
    title = "Tissue-Specific Radiation Response: DEG Rate by Organ",
    subtitle = "Fisher's exact test (organ vs rest), Bonferroni-corrected  |  *** p<0.001, ** p<0.01, * p<0.05",
    x = "Proportion of tissue-specific genes that are DEGs",
    y = "Broad Organ"
  ) +
  theme_minimal(base_size = 13) +
  theme(
    plot.title = element_text(face = "bold", size = 15),
    plot.subtitle = element_text(size = 10, color = "grey40"),
    legend.position = "top",
    panel.grid.minor = element_blank(),
    axis.text.y = element_text(face = "bold")
  )

ggsave("/mnt/results/fig5_tissue_enrichment_test.png", p, width = 9, height = 5, dpi = 300, bg = "white")
cat("Saved fig5_tissue_enrichment_test.png\n")


Error in `scale_y_continuous()`:
! Discrete value supplied to a continuous scale.
ℹ Example values: Flower, Leaf_Shoot, Root, Seed_Silique, and Seedling.
Traceback:

1. lapply(plot, grid.draw)
2. FUN(X[[i]], ...)
3. `grid.draw.ggplot2::ggplot`(X[[i]], ...)
4. print(x)
5. `print.ggplot2::ggplot`(x)
6. ggplot_build(x)
7. `ggplot_build.ggplot2::ggplot`(x)
8. layout$train_position(data, scale_x(), scale_y())
9. train_position(..., self = self)
10. self$facet$train_scales(self$panel_scales_x, self$panel_scales_y, 
  .     layout, data, self$facet_params)
11. train_scales(...)
12. scale_apply(layer_data, y_vars, "train", SCALE_Y, y_scales)
13. lapply(vars, function(var) {
  .     pieces <- lapply(seq_along(scales), function(i) {
  .         scales[[i]][[method]](vec_slice(data[[var]], scale_index[[i]]))
  .     })
  .     pieces[lengths(pieces) == 0] <- NULL
  .     o <- order(unlist(scale_index))[seq_len(sum(lengths(pieces)))]
  .     vec_c(!!!pieces)[o]
  . })
14. FUN(X[[i]], ...)
15. lapp

Rebuild and save enrichment forest plot

In [67]:

# ── Rebuild plot data cleanly ──────────────────────────────────────────────────
# The issue is likely that plot_data$organ lost its factor status

# Rebuild from scratch
organs_ordered <- fisher_results$organ[order(fisher_results$deg_rate_organ)]
plot_data2 <- data.frame(
  organ = factor(fisher_results$organ, levels = organs_ordered),
  deg_rate = fisher_results$deg_rate_organ,
  direction = fisher_results$direction,
  ci_lower = sapply(1:nrow(fisher_results), function(i) {
    binom.test(fisher_results$deg_in_organ[i], fisher_results$total_in_organ[i])$conf.int[1]
  }),
  ci_upper = sapply(1:nrow(fisher_results), function(i) {
    binom.test(fisher_results$deg_in_organ[i], fisher_results$total_in_organ[i])$conf.int[2]
  }),
  sig_label = ifelse(fisher_results$padj < 0.001, "***",
              ifelse(fisher_results$padj < 0.01, "**",
              ifelse(fisher_results$padj < 0.05, "*", "ns"))),
  stringsAsFactors = FALSE
)

cat("plot_data2 structure:\n")
str(plot_data2)
cat("\norgan levels:", levels(plot_data2$organ), "\n")

p2 <- ggplot(plot_data2, aes(x = deg_rate, y = organ, color = direction)) +
  geom_vline(xintercept = overall_rate, linetype = "dashed", color = "grey50", linewidth = 0.5) +
  geom_errorbarh(aes(xmin = ci_lower, xmax = ci_upper), height = 0.2, linewidth = 0.8) +
  geom_point(size = 4) +
  geom_text(aes(label = sprintf("%.1f%% %s", deg_rate * 100, sig_label)),
            hjust = -0.2, vjust = 0.5, size = 3.5, color = "black", inherit.aes = FALSE) +
  scale_color_manual(values = c("enriched" = "#0072B2", "depleted" = "#D55E00"),
                     labels = c("enriched" = "Enriched for DEGs", "depleted" = "Depleted for DEGs"),
                     name = "") +
  scale_x_continuous(limits = c(0.15, 0.70)) +
  labs(
    title = "Tissue-Specific Radiation Response: DEG Rate by Organ",
    subtitle = "Fisher's exact test (organ vs rest), Bonferroni-corrected  |  *** p<0.001, ** p<0.01, * p<0.05",
    x = "Proportion of tissue-specific genes that are DEGs",
    y = "Broad Organ"
  ) +
  theme_minimal(base_size = 13) +
  theme(
    plot.title = element_text(face = "bold", size = 15),
    plot.subtitle = element_text(size = 10, color = "grey40"),
    legend.position = "top",
    panel.grid.minor = element_blank(),
    axis.text.y = element_text(face = "bold")
  )

ggsave("/mnt/results/fig5_tissue_enrichment_test.png", p2, width = 9, height = 5, dpi = 300, bg = "white")
cat("Saved fig5_tissue_enrichment_test.png\n")


plot_data2 structure:
'data.frame':	5 obs. of  6 variables:
 $ organ    : Factor w/ 5 levels "Seed_Silique",..: 2 4 3 1 5
 $ deg_rate : num  0.316 0.425 0.361 0.274 0.507
 $ direction: chr  "depleted" "enriched" "enriched" "depleted" ...
 $ ci_lower : num  0.301 0.409 0.334 0.256 0.422
 $ ci_upper : num  0.331 0.443 0.389 0.293 0.592
 $ sig_label: chr  "***" "***" "ns" "***" ...

organ levels: Seed_Silique Flower Root Leaf_Shoot Seedling 
`height` was translated to `width`.
Error in `geom_text()`:
! Problem while setting up geom.
ℹ Error occurred in the 4th layer.
Caused by error in `compute_geom_1()`:
! `geom_text()` requires the following missing aesthetics: x and y.
Traceback:

1. lapply(plot, grid.draw)
2. FUN(X[[i]], ...)
3. `grid.draw.ggplot2::ggplot`(X[[i]], ...)
4. print(x)
5. `print.ggplot2::ggplot`(x)
6. ggplot_build(x)
7. `ggplot_build.ggplot2::ggplot`(x)
8. by_layer(function(l, d) l$compute_geom_1(d), layers, data, "setting up geom")
9. try_fetch(for (i in seq_along(data)) 

Fix geom_text and save enrichment plot

In [69]:

# ── Fix geom_text: keep inherited aesthetics, just override color ──────────────
p2 <- ggplot(plot_data2, aes(x = deg_rate, y = organ, color = direction)) +
  geom_vline(xintercept = overall_rate, linetype = "dashed", color = "grey50", linewidth = 0.5) +
  geom_errorbarh(aes(xmin = ci_lower, xmax = ci_upper), height = 0.2, linewidth = 0.8) +
  geom_point(size = 4) +
  geom_text(aes(x = deg_rate, y = organ, label = sprintf("%.1f%% %s", deg_rate * 100, sig_label)),
            hjust = -0.2, vjust = 0.5, size = 3.5, color = "black") +
  scale_color_manual(values = c("enriched" = "#0072B2", "depleted" = "#D55E00"),
                     labels = c("enriched" = "Enriched for DEGs", "depleted" = "Depleted for DEGs"),
                     name = "") +
  scale_x_continuous(limits = c(0.15, 0.70)) +
  labs(
    title = "Tissue-Specific Radiation Response: DEG Rate by Organ",
    subtitle = "Fisher's exact test (organ vs rest), Bonferroni-corrected  |  *** p<0.001, ** p<0.01, * p<0.05",
    x = "Proportion of tissue-specific genes that are DEGs",
    y = "Broad Organ"
  ) +
  theme_minimal(base_size = 13) +
  theme(
    plot.title = element_text(face = "bold", size = 15),
    plot.subtitle = element_text(size = 10, color = "grey40"),
    legend.position = "top",
    panel.grid.minor = element_blank(),
    axis.text.y = element_text(face = "bold")
  )

ggsave("/mnt/results/fig5_tissue_enrichment_test.png", p2, width = 9, height = 5, dpi = 300, bg = "white")
cat("Saved fig5_tissue_enrichment_test.png\n")


`height` was translated to `width`.
Saved fig5_tissue_enrichment_test.png


Save statistical results and print summary

In [71]:

# ── Save statistical test results to CSV ───────────────────────────────────────

# 1. Per-tissue Fisher's test results
fisher_out <- fisher_results
colnames(fisher_out) <- c("Organ", "DEGs_in_organ", "Total_genes_in_organ", 
                           "DEG_rate_organ", "DEG_rate_rest", "Odds_ratio", 
                           "P_value", "Direction", "P_adj_bonferroni", "Significant")
write.csv(fisher_out, "/mnt/results/tissue_enrichment_statistics.csv", row.names = FALSE)
cat("Saved tissue_enrichment_statistics.csv\n")

# 2. Pairwise comparison results
pairwise_out <- pairwise_results[, c("organ1", "organ2", "rate1", "rate2", 
                                      "odds_ratio", "p_value", "padj", "higher", "significant")]
colnames(pairwise_out) <- c("Organ1", "Organ2", "DEG_rate_1", "DEG_rate_2",
                             "Odds_ratio", "P_value", "P_adj_bonferroni", 
                             "Higher_response", "Significant")
write.csv(pairwise_out, "/mnt/results/pairwise_tissue_comparisons.csv", row.names = FALSE)
cat("Saved pairwise_tissue_comparisons.csv\n")

# 3. Print a clean summary
cat("\n")
cat("============================================================\n")
cat("  STATISTICAL ANALYSIS SUMMARY: Tissue-Specific Response\n")
cat("============================================================\n")
cat("\n")
cat("TEST 1: Chi-square test of independence\n")
cat(sprintf("  X-squared = %.1f, df = 4, p-value = %.2e\n", chi_result$statistic, chi_result$p.value))
cat("  Result: SIGNIFICANT — tissue assignment affects DEG status\n")
cat("\n")
cat("TEST 2: Per-tissue Fisher's exact test (organ vs rest)\n")
for (i in 1:nrow(fisher_results)) {
  r <- fisher_results[i, ]
  cat(sprintf("  %-14s: %s (OR=%.2f, padj=%.1e) — %.1f%% vs %.1f%%\n",
              r$organ, r$direction, r$odds_ratio, r$padj,
              r$deg_rate_organ * 100, r$deg_rate_rest * 100))
}
cat("\n")
cat("TEST 3: Pairwise Fisher's exact tests (Bonferroni-corrected)\n")
sig <- pairwise_results[pairwise_results$padj < 0.05, ]
cat(sprintf("  %d of 10 pairwise comparisons significant (padj < 0.05)\n", nrow(sig)))
cat("  Ranking (highest to lowest DEG rate):\n")
cat("    Seedling (50.7%) > Leaf_Shoot (42.5%) > Root (36.2%) > Flower (31.6%) > Seed_Silique (27.4%)\n")
cat("\n")
cat("TEST 4: Direction of response (up/down ratio across tissues)\n")
cat(sprintf("  X-squared = %.1f, df = 4, p-value = %.2e\n", chi_dir$statistic, chi_dir$p.value))
cat("  Result: SIGNIFICANT — up/down ratio differs by tissue\n")
cat("  Root: 35.6%% up (predominantly DOWNREGULATED)\n")
cat("  Seedling: 88.9%% up (predominantly UPREGULATED)\n")
cat("  Seed_Silique: 60.8%% up\n")
cat("  Flower: 57.1%% up\n")
cat("  Leaf_Shoot: 51.8%% up (roughly balanced)\n")
cat("\n")
cat("============================================================\n")


Saved tissue_enrichment_statistics.csv
Saved pairwise_tissue_comparisons.csv

  STATISTICAL ANALYSIS SUMMARY: Tissue-Specific Response

TEST 1: Chi-square test of independence
  X-squared = 174.2, df = 4, p-value = 1.31e-36
  Result: SIGNIFICANT — tissue assignment affects DEG status

TEST 2: Per-tissue Fisher's exact test (organ vs rest)
  Flower        : depleted (OR=0.80, padj=5.2e-07) — 31.6% vs 36.7%
  Leaf_Shoot    : enriched (OR=1.62, padj=1.4e-27) — 42.5% vs 31.4%
  Root          : enriched (OR=1.06, padj=1.0e+00) — 36.1% vs 34.7%
  Seed_Silique  : depleted (OR=0.65, padj=2.7e-16) — 27.4% vs 36.8%
  Seedling      : enriched (OR=1.94, padj=6.0e-04) — 50.7% vs 34.7%

TEST 3: Pairwise Fisher's exact tests (Bonferroni-corrected)
  9 of 10 pairwise comparisons significant (padj < 0.05)
  Ranking (highest to lowest DEG rate):
    Seedling (50.7%) > Leaf_Shoot (42.5%) > Root (36.2%) > Flower (31.6%) > Seed_Silique (27.4%)

TEST 4: Direction of response (up/down ratio across tissues)
 

Set up PDF infrastructure for statistical walkthrough

In [73]:

# ── Generate the statistical walkthrough PDF ───────────────────────────────────
import os
from reportlab.lib.pagesizes import letter
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
                                 PageBreak, HRFlowable, KeepTogether, Image)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.colors import HexColor
from reportlab.lib.enums import TA_JUSTIFY, TA_CENTER, TA_LEFT
from reportlab.lib.units import inch

# ── Phylo Brand Colors ──
PHYLO_GOLD      = HexColor("#D4A04A")
HEADING_COLOR   = HexColor("#111111")
BODY_TEXT       = HexColor("#2C2A26")
MUTED_TEXT      = HexColor("#8A8378")
TABLE_HEADER_BG = PHYLO_GOLD
TABLE_HEADER_FG = HexColor("#FFFFFF")
TABLE_ALT_ROW   = HexColor("#F9F7F3")
TABLE_BORDER    = HexColor("#D5CFC5")
CALLOUT_BG      = HexColor("#FAF9F3")
PHYLO_BLUE      = HexColor("#0279EE")

FONT_HEADING = "Helvetica-Bold"
FONT_BODY    = "Helvetica"
FONT_ITALIC  = "Helvetica-Oblique"
FONT_MONO    = "Courier"

styles = getSampleStyleSheet()

styles.add(ParagraphStyle(name="ReportTitle", fontName=FONT_HEADING, fontSize=24,
    textColor=HEADING_COLOR, spaceBefore=0, spaceAfter=6, leading=30))
styles.add(ParagraphStyle(name="Subtitle", fontName=FONT_BODY, fontSize=12,
    textColor=PHYLO_GOLD, spaceAfter=4))
styles.add(ParagraphStyle(name="Attribution", fontName=FONT_ITALIC, fontSize=10,
    textColor=MUTED_TEXT, spaceAfter=8))
styles.add(ParagraphStyle(name="SectionHead", fontName=FONT_HEADING, fontSize=17,
    textColor=HEADING_COLOR, spaceBefore=22, spaceAfter=8))
styles.add(ParagraphStyle(name="SubHead", fontName=FONT_HEADING, fontSize=13,
    textColor=HEADING_COLOR, spaceBefore=14, spaceAfter=5))
styles.add(ParagraphStyle(name="SubSubHead", fontName=FONT_HEADING, fontSize=11,
    textColor=PHYLO_GOLD, spaceBefore=10, spaceAfter=4))
styles.add(ParagraphStyle(name="Body", fontName=FONT_BODY, fontSize=10.5,
    textColor=BODY_TEXT, alignment=TA_JUSTIFY, spaceAfter=7, leading=15))
styles.add(ParagraphStyle(name="Caption", fontName=FONT_ITALIC, fontSize=9,
    textColor=MUTED_TEXT, alignment=TA_CENTER, spaceAfter=12))
styles.add(ParagraphStyle(name="CalloutText", fontName=FONT_BODY, fontSize=10,
    textColor=BODY_TEXT, alignment=TA_LEFT, spaceAfter=4, leading=14))
styles.add(ParagraphStyle(name="CalloutTitle", fontName=FONT_HEADING, fontSize=10.5,
    textColor=PHYLO_GOLD, spaceAfter=4))
styles.add(ParagraphStyle(name="CodeStyle", fontName=FONT_MONO, fontSize=8.5,
    textColor=BODY_TEXT, alignment=TA_LEFT, spaceAfter=4, leading=11,
    leftIndent=12, rightIndent=12))
styles.add(ParagraphStyle(name="BulletStyle", fontName=FONT_BODY, fontSize=10.5,
    textColor=BODY_TEXT, alignment=TA_LEFT, spaceAfter=4, leading=15,
    leftIndent=20, bulletIndent=8))
styles.add(ParagraphStyle(name="StatBox", fontName=FONT_BODY, fontSize=10,
    textColor=BODY_TEXT, alignment=TA_LEFT, spaceAfter=3, leading=14))
styles.add(ParagraphStyle(name="TableCell", fontName=FONT_BODY, fontSize=9,
    textColor=BODY_TEXT, alignment=TA_LEFT, leading=12))
styles.add(ParagraphStyle(name="TableHeader", fontName=FONT_HEADING, fontSize=9,
    textColor=TABLE_HEADER_FG, alignment=TA_LEFT, leading=12))

def divider(width=480):
    return HRFlowable(width=width, thickness=1, color=PHYLO_GOLD,
                      spaceAfter=10, spaceBefore=4)

def callout_box(title, text, width=460):
    content = []
    if title:
        content.append(Paragraph(title, styles["CalloutTitle"]))
    content.append(Paragraph(text, styles["CalloutText"]))
    t = Table([[content]], colWidths=[width])
    t.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, -1), CALLOUT_BG),
        ("BOX", (0, 0), (-1, -1), 0.5, TABLE_BORDER),
        ("LINEBEFORE", (0, 0), (0, -1), 3, PHYLO_GOLD),
        ("TOPPADDING", (0, 0), (-1, -1), 10),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 10),
        ("LEFTPADDING", (0, 0), (-1, -1), 14),
        ("RIGHTPADDING", (0, 0), (-1, -1), 14),
    ]))
    t.hAlign = "CENTER"
    return t

def stat_box(title, items, width=460):
    """Box for displaying statistical results."""
    content = [Paragraph(f"<b>{title}</b>", styles["CalloutTitle"])]
    for item in items:
        content.append(Paragraph(item, styles["StatBox"]))
    t = Table([[content]], colWidths=[width])
    t.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, -1), HexColor("#F0F4FA")),
        ("BOX", (0, 0), (-1, -1), 0.5, TABLE_BORDER),
        ("LINEBEFORE", (0, 0), (0, -1), 3, PHYLO_BLUE),
        ("TOPPADDING", (0, 0), (-1, -1), 10),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 10),
        ("LEFTPADDING", (0, 0), (-1, -1), 14),
        ("RIGHTPADDING", (0, 0), (-1, -1), 14),
    ]))
    t.hAlign = "CENTER"
    return t

def code_block(code_text, width=460):
    escaped = code_text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    escaped = escaped.replace("\n", "<br/>")
    p = Paragraph(escaped, styles["CodeStyle"])
    t = Table([[p]], colWidths=[width])
    t.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, -1), HexColor("#F5F3EE")),
        ("BOX", (0, 0), (-1, -1), 0.5, TABLE_BORDER),
        ("TOPPADDING", (0, 0), (-1, -1), 8),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 8),
        ("LEFTPADDING", (0, 0), (-1, -1), 10),
        ("RIGHTPADDING", (0, 0), (-1, -1), 10),
    ]))
    t.hAlign = "CENTER"
    return t

def make_table(headers, rows, col_widths):
    data = [[Paragraph(f'<b>{h}</b>', styles["TableHeader"]) for h in headers]]
    for row in rows:
        data.append([Paragraph(str(c), styles["TableCell"]) for c in row])
    t = Table(data, colWidths=col_widths, repeatRows=1)
    t.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), TABLE_HEADER_BG),
        ("TEXTCOLOR", (0, 0), (-1, 0), TABLE_HEADER_FG),
        *[("BACKGROUND", (0, i), (-1, i), TABLE_ALT_ROW) for i in range(2, len(data), 2)],
        ("GRID", (0, 0), (-1, -1), 0.5, TABLE_BORDER),
        ("BOX", (0, 0), (-1, -1), 0.75, TABLE_BORDER),
        ("TOPPADDING", (0, 0), (-1, -1), 5),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
        ("LEFTPADDING", (0, 0), (-1, -1), 6),
        ("RIGHTPADDING", (0, 0), (-1, -1), 6),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
    ]))
    t.hAlign = "CENTER"
    return t

def page_header_footer(canvas, doc):
    canvas.saveState()
    w, h = letter
    canvas.setFont("Helvetica", 9)
    canvas.setFillColor(MUTED_TEXT)
    canvas.drawString(60, h - 40, "Statistical Walkthrough: Figures and Code")
    canvas.setStrokeColor(PHYLO_GOLD)
    canvas.setLineWidth(1)
    canvas.line(60, h - 48, w - 60, h - 48)
    canvas.setStrokeColor(TABLE_BORDER)
    canvas.setLineWidth(0.75)
    canvas.line(60, 40, w - 60, 40)
    canvas.setFont("Helvetica", 8)
    canvas.setFillColor(MUTED_TEXT)
    canvas.drawCentredString(w / 2, 26, f"Page {doc.page}")
    canvas.restoreState()

output_path = "/mnt/results/statistical_walkthrough_figures_and_code.pdf"
doc = SimpleDocTemplate(output_path, pagesize=letter,
                        topMargin=52, bottomMargin=52,
                        leftMargin=60, rightMargin=60)
story = []

print("Infrastructure ready. Building PDF content...")


Infrastructure ready. Building PDF content...


Add Part I: all 5 figures with statistical explanations

In [75]:

# ── TITLE PAGE ─────────────────────────────────────────────────────────────────
story.append(Spacer(1, 40))
story.append(Paragraph("Statistical Walkthrough", styles["ReportTitle"]))
story.append(Paragraph("Figures and Code Explained with Basic + Intermediate Statistical Terminology", styles["Subtitle"]))
story.append(Spacer(1, 8))
story.append(Paragraph("<i>Companion to the Tissue-Specific DEG Analysis  |  Generated by Biomni (Phylo)  |  2026-07-09</i>", styles["Attribution"]))
story.append(Spacer(1, 20))

story.append(Paragraph(
    "This document walks through each figure and key code block from the tissue-specific "
    "DEG analysis, explaining the statistical concepts in basic and intermediate terms. "
    "Each section covers one figure or code block, describing what it shows, what statistical "
    "test or concept underlies it, how to read the results, and what the numbers mean in "
    "plain language.", styles["Body"]))

story.append(divider())

story.append(Paragraph("Contents", styles["SubHead"]))
toc = [
    "Part I: Figures Explained",
    "  Figure 1 — Stacked Bar Chart: DEG Counts by Organ",
    "  Figure 2 — Volcano Plots: Magnitude vs. Significance",
    "  Figure 3 — Heatmap: Expression Specificity of Top DEGs",
    "  Figure 4 — UpSet Diagram: Cross-Organ DEG Overlaps",
    "  Figure 5 — Forest Plot: Tissue Enrichment Statistics",
    "",
    "Part II: Key Code Blocks Explained",
    "  Code Block 1 — The Tau Tissue-Specificity Index",
    "  Code Block 2 — Chi-Square Test of Independence",
    "  Code Block 3 — Fisher's Exact Test (Per-Tissue)",
    "  Code Block 4 — Pairwise Fisher's Exact Tests",
    "  Code Block 5 — Direction-of-Response Chi-Square Test",
    "",
    "Part III: Statistical Glossary",
]
for item in toc:
    story.append(Paragraph(item, styles["BulletStyle"]))

story.append(PageBreak())

# ── PART I HEADER ──────────────────────────────────────────────────────────────
story.append(Paragraph("Part I: Figures Explained", styles["SectionHead"]))
story.append(divider())
story.append(Paragraph(
    "Each figure below is reproduced with a full statistical explanation. We cover what the "
    "plot displays, what statistical concept it illustrates, how to interpret the visual "
    "elements, and what conclusions we can draw.", styles["Body"]))

story.append(PageBreak())

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 1: STACKED BAR CHART
# ═══════════════════════════════════════════════════════════════════════════════
story.append(Paragraph("Figure 1 — Stacked Bar Chart: DEG Counts by Organ", styles["SectionHead"]))
story.append(divider())

# Embed the figure
img1 = Image("/mnt/results/fig1_deg_counts_by_organ.png", width=380, height=285)
img1.hAlign = "CENTER"
story.append(img1)
story.append(Paragraph("Figure 1: Number of upregulated (blue) and downregulated (orange) DEGs per broad organ.", styles["Caption"]))

story.append(Paragraph("What This Graph Shows", styles["SubHead"]))
story.append(Paragraph(
    "A stacked bar chart displays the count of differentially expressed genes (DEGs) for each "
    "of the five broad organ categories. Each bar is split into two colors: blue for genes that "
    "increased in expression after radiation (upregulated) and orange for genes that decreased "
    "(downregulated). The total height of each bar represents the total number of tissue-specific "
    "DEGs assigned to that organ.", styles["Body"]))

story.append(Paragraph("Statistical Concept: Frequency Distribution", styles["SubHead"]))
story.append(Paragraph(
    "This is a <b>frequency distribution</b> — a count of how many observations (genes) fall into "
    "each category (organ). In statistics, a frequency distribution is the most basic way to "
    "summarize categorical data. The bars show the <b>absolute frequency</b> (raw count) for each "
    "category. The blue/orange split shows a <b>conditional frequency</b> — the count of up vs. "
    "down within each organ.", styles["Body"]))

story.append(callout_box(
    "BASIC TERM: What is a frequency?",
    "A frequency is simply how many times something occurs. If 1,399 genes in the Leaf_Shoot "
    "category are DEGs, then the frequency of Leaf_Shoot DEGs is 1,399. A frequency distribution "
    "shows these counts for all categories at once, making it easy to compare groups visually."
))

story.append(Paragraph("How to Read It", styles["SubHead"]))
story.append(Paragraph(
    "&bull; <b>Bar height</b> = total DEGs for that organ. Taller bars mean more radiation-responsive "
    "genes were assigned to that tissue.", styles["BulletStyle"]))
story.append(Paragraph(
    "&bull; <b>Blue vs. orange proportion</b> = direction of response. If blue dominates, that "
    "tissue's response is primarily activation (upregulation). If orange dominates, the response "
    "is primarily suppression (downregulation).", styles["BulletStyle"]))
story.append(Paragraph(
    "&bull; <b>Numbers inside bars</b> = exact counts for each segment.", styles["BulletStyle"]))

story.append(Paragraph("What the Data Tell Us", styles["SubHead"]))
story.append(Paragraph(
    "Leaf_Shoot has the tallest bar (1,399 DEGs), meaning more radiation-responsive genes are "
    "normally active in leaf and shoot tissues than anywhere else. The blue and orange segments "
    "are roughly equal in Leaf_Shoot, indicating a balanced mix of up- and downregulation. "
    "In contrast, Root shows more orange than blue (278 down vs. 154 up), suggesting radiation "
    "predominantly suppresses root gene expression. Seedling is almost entirely blue (64 up vs. "
    "8 down), indicating a strong activation response.", styles["Body"]))

story.append(callout_box(
    "INTERMEDIATE TERM: Why counts alone can be misleading",
    "Raw counts don't account for the fact that some organs have more tissue-specific genes to "
    "begin with. Leaf_Shoot has 3,288 tissue-specific genes in the atlas; Root has only 1,195. "
    "So Leaf_Shoot having more DEGs could simply be because it has more genes total. To determine "
    "whether the <i>rate</i> of differential expression differs (not just the count), we need the "
    "statistical tests in Figure 5. This is the difference between <b>absolute frequency</b> "
    "(raw counts) and <b>relative frequency</b> (proportions or rates)."
))

story.append(PageBreak())

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 2: VOLCANO PLOTS
# ═══════════════════════════════════════════════════════════════════════════════
story.append(Paragraph("Figure 2 — Volcano Plots: Magnitude vs. Significance", styles["SectionHead"]))
story.append(divider())

img2 = Image("/mnt/results/fig2_volcano_plots_by_organ.png", width=420, height=300)
img2.hAlign = "CENTER"
story.append(img2)
story.append(Paragraph("Figure 2: Five volcano plots (one per organ). Blue = upregulated DEGs, orange = downregulated, grey = non-DEGs or constitutive genes.", styles["Caption"]))

story.append(Paragraph("What This Graph Shows", styles["SubHead"]))
story.append(Paragraph(
    "A volcano plot is a scatter plot where each point is one gene. The x-axis shows the "
    "<b>log2 fold change</b> (magnitude and direction of expression change), and the y-axis shows "
    "the <b>-log10 adjusted p-value</b> (statistical significance). Five panels show the same "
    "plot separately for each broad organ, with constitutive genes (broadly expressed) shown as "
    "grey background points for context.", styles["Body"]))

story.append(Paragraph("Statistical Concept: Effect Size vs. Significance", styles["SubHead"]))
story.append(Paragraph(
    "Volcano plots display two fundamental statistical quantities simultaneously:", styles["Body"]))

story.append(Paragraph(
    "&bull; <b>Effect size</b> (x-axis): The log2 fold change measures how much the gene's "
    "expression changed. A value of +2 means the expression quadrupled; -1 means it halved. "
    "This is the <i>magnitude</i> of the biological effect.", styles["BulletStyle"]))
story.append(Paragraph(
    "&bull; <b>Statistical significance</b> (y-axis): The -log10(padj) measures how confident "
    "we are that the change is real (not due to chance). Higher values = more confident. The "
    "dashed horizontal line marks padj = 0.05, the conventional significance threshold.", styles["BulletStyle"]))

story.append(callout_box(
    "BASIC TERM: What is a p-value?",
    "A p-value answers the question: 'If there were truly no difference between irradiated and "
    "control plants, how likely is it that we would observe a difference this large or larger "
    "just by random chance?' A small p-value (typically < 0.05) means the observed difference "
    "is unlikely to be due to chance, so we conclude the difference is real. The adjusted p-value "
    "(padj) corrects for the fact that we are testing thousands of genes at once."
))

story.append(callout_box(
    "INTERMEDIATE TERM: Why -log10(padj) instead of padj directly?",
    "P-values for significant genes can be extremely small (e.g., 5.34 x 10<super>-38</super>). "
    "On a linear scale, all these tiny values would cluster near zero and be indistinguishable. "
    "Taking -log10 transforms them: -log10(0.05) = 1.3, -log10(0.001) = 3, -log10(10<super>-38</super>) = 38. "
    "Now the most significant genes are at the top of the plot, spread out and visible. This is "
    "called a <b>logarithmic transformation</b>, commonly used to spread out values that span "
    "many orders of magnitude."
))

story.append(Paragraph("How to Read It", styles["SubHead"]))
story.append(Paragraph(
    "&bull; <b>Points above the dashed line</b> = statistically significant (padj < 0.05).", styles["BulletStyle"]))
story.append(Paragraph(
    "&bull; <b>Points to the right of x=0</b> = upregulated (expression increased). Points to "
    "the left = downregulated (expression decreased).", styles["BulletStyle"]))
story.append(Paragraph(
    "&bull; <b>Points in the top-right corner</b> = genes that are both highly significant AND "
    "strongly upregulated — the most biologically interesting genes.", styles["BulletStyle"]))
story.append(Paragraph(
    "&bull; <b>Points in the top-left corner</b> = highly significant AND strongly downregulated.", styles["BulletStyle"]))

story.append(Paragraph("What the Data Tell Us", styles["SubHead"]))
story.append(Paragraph(
    "Each organ panel shows a different pattern. Leaf_Shoot and Flower have many points spread "
    "across both sides of x=0, indicating both up- and downregulation. Root has more points on "
    "the left (downregulation), consistent with the bar chart. Seed_Silique shows a cluster of "
    "blue points on the right (upregulation). The vertical spread (significance) is similar across "
    "panels, but the horizontal spread (effect size) and direction differ — this is the tissue "
    "specificity we are quantifying.", styles["Body"]))

story.append(PageBreak())

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 3: HEATMAP
# ═══════════════════════════════════════════════════════════════════════════════
story.append(Paragraph("Figure 3 — Heatmap: Expression Specificity of Top DEGs", styles["SectionHead"]))
story.append(divider())

img3 = Image("/mnt/results/fig3_heatmap_top_degs.png", width=420, height=280)
img3.hAlign = "CENTER"
story.append(img3)
story.append(Paragraph("Figure 3: Z-scored expression of the 50 most significant DEGs across 33 sub-tissues. Top bar = broad organ; right bar = regulation direction.", styles["Caption"]))

story.append(Paragraph("What This Graph Shows", styles["SubHead"]))
story.append(Paragraph(
    "A heatmap displays expression levels as colors. Each row is one of the 50 most significant "
    "DEGs (sorted by padj); each column is one of 33 sub-tissues. The color intensity shows how "
    "highly that gene is expressed in that tissue, relative to its own average across all tissues. "
    "The top annotation bar groups columns by broad organ; the right annotation bar shows whether "
    "each gene was up- or downregulated in response to radiation.", styles["Body"]))

story.append(Paragraph("Statistical Concept: Z-Score Normalization", styles["SubHead"]))
story.append(Paragraph(
    "The colors represent <b>z-scores</b>, not raw expression values. A z-score measures how many "
    "standard deviations a value is above or below the gene's mean expression across all tissues.", styles["Body"]))

story.append(code_block(
    "z = (expression_in_tissue - mean_expression) / standard_deviation\n\n"
    "z = 0  → expression is exactly at the gene's average\n"
    "z = +2 → expression is 2 standard deviations above average\n"
    "z = -2 → expression is 2 standard deviations below average"
))

story.append(callout_box(
    "BASIC TERM: Why use z-scores instead of raw values?",
    "Different genes have very different baseline expression levels. Gene A might be expressed "
    "at 1,000 units everywhere; gene B at 10 units everywhere. If we plotted raw values, gene A "
    "would always be darker than gene B, making it impossible to see tissue-specific patterns. "
    "Z-scoring rescales each gene so its average is 0 and its variation is comparable to other "
    "genes. Now we can see patterns: a red block means 'this gene is expressed much higher here "
    "than its own average,' regardless of its absolute level."
))

story.append(callout_box(
    "INTERMEDIATE TERM: Standard deviation as a spread measure",
    "The standard deviation (SD) measures how spread out a gene's expression is across tissues. "
    "If a gene is expressed at similar levels everywhere, its SD is small, and even a modest "
    "difference will produce a large z-score. If a gene varies wildly across tissues, its SD is "
    "large, and only big differences produce notable z-scores. This means z-scores account for "
    "each gene's natural variability — a form of <b>standardization</b> that makes genes comparable."
))

story.append(Paragraph("How to Read It", styles["SubHead"]))
story.append(Paragraph(
    "&bull; <b>Red blocks</b> = the gene is expressed above its personal average in that tissue.", styles["BulletStyle"]))
story.append(Paragraph(
    "&bull; <b>Blue blocks</b> = the gene is expressed below its personal average.", styles["BulletStyle"]))
story.append(Paragraph(
    "&bull; <b>Rows that are red in only one organ block</b> = tissue-specific genes (high Tau).", styles["BulletStyle"]))
story.append(Paragraph(
    "&bull; <b>Rows that are uniformly colored</b> = broadly expressed genes (low Tau).", styles["BulletStyle"]))
story.append(Paragraph(
    "&bull; <b>The right-side annotation</b> lets you check whether upregulated vs. downregulated "
    "DEGs tend to come from specific tissues.", styles["BulletStyle"]))

story.append(Paragraph("What the Data Tell Us", styles["SubHead"]))
story.append(Paragraph(
    "The heatmap reveals that the most significant radiation-responsive genes come from diverse "
    "tissue backgrounds. Some rows show sharp red blocks in only one organ (e.g., a row that is "
    "red only in the Root columns), confirming these are tissue-specific genes. Others show "
    "patchy patterns across multiple organs, suggesting broader expression. The clustering of "
    "rows (dendrogram on the left) groups genes with similar tissue expression profiles, revealing "
    "that upregulated and downregulated DEGs are interspersed — radiation affects genes from "
    "many tissue contexts.", styles["Body"]))

story.append(PageBreak())

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 4: UPSET DIAGRAM
# ═══════════════════════════════════════════════════════════════════════════════
story.append(Paragraph("Figure 4 — UpSet Diagram: Cross-Organ DEG Overlaps", styles["SectionHead"]))
story.append(divider())

img4 = Image("/mnt/results/fig4_upset_organ_overlaps.png", width=400, height=240)
img4.hAlign = "CENTER"
story.append(img4)
story.append(Paragraph("Figure 4: UpSet diagram showing how many DEGs are expressed above median in each combination of organs.", styles["Caption"]))

story.append(Paragraph("What This Graph Shows", styles["SubHead"]))
story.append(Paragraph(
    "An UpSet diagram is an alternative to a Venn diagram for showing <b>set intersections</b>. "
    "Each DEG can be 'expressed above its own median' in multiple organs. The left bar chart shows "
    "the total number of DEGs expressed above median in each organ. The main bar chart shows how "
    "many DEGs are shared across each combination of organs. The matrix of connected dots below "
    "indicates which organs are included in each intersection.", styles["Body"]))

story.append(Paragraph("Statistical Concept: Set Intersection Analysis", styles["SubHead"]))
story.append(Paragraph(
    "A <b>set</b> is a collection of items (here, DEGs expressed in a given organ). The "
    "<b>intersection</b> of two sets is the collection of items that belong to both. For example, "
    "the intersection of 'DEGs expressed in Root' and 'DEGs expressed in Flower' is the set of "
    "DEGs that are active in both organs. UpSet diagrams scale better than Venn diagrams when "
    "there are more than 3-4 sets.", styles["Body"]))

story.append(callout_box(
    "BASIC TERM: What is a median?",
    "The median is the middle value when you sort a set of numbers. If a gene's expression across "
    "5 organs is [10, 20, 30, 40, 50], the median is 30. We use 'above median' as a threshold: "
    "an organ counts as 'expressing' the gene if its expression value is above the gene's median. "
    "This is a <b>non-parametric</b> threshold — it doesn't assume any particular distribution, "
    "just that values above the midpoint are 'relatively high' for that gene."
))

story.append(callout_box(
    "INTERMEDIATE TERM: Why not a Venn diagram?",
    "Venn diagrams work for 2-3 sets but become unreadable with 5 sets (the circles overlap in "
    "ways that make regions too small to label). The UpSet diagram solves this by using a matrix "
    "of dots to encode set membership: a filled dot means 'included in this set,' and connected "
    "dots define an intersection. The bar height shows the size of that intersection. This is a "
    "<b>scalable</b> visualization for set intersection analysis."
))

story.append(Paragraph("How to Read It", styles["SubHead"]))
story.append(Paragraph(
    "&bull; <b>Left bars</b> = total DEGs expressed above median in each organ (set sizes).", styles["BulletStyle"]))
story.append(Paragraph(
    "&bull; <b>Main bars</b> = number of DEGs in each specific intersection (combination of organs).", styles["BulletStyle"]))
story.append(Paragraph(
    "&bull; <b>Dot matrix</b> = which organs are included in each intersection. Filled dots connected "
    "by a line = the organs whose overlap is being counted in that bar.", styles["BulletStyle"]))

story.append(Paragraph("What the Data Tell Us", styles["SubHead"]))
story.append(Paragraph(
    "The largest intersections typically involve 2-3 organs, meaning many DEGs are co-expressed "
    "across multiple tissues. This is expected: genes involved in core stress responses (like DNA "
    "repair) are active in many tissues. The single-organ bars (only one dot filled) represent "
    "DEGs that are expressed above median in only one organ — these are the most tissue-restricted. "
    "The distribution of intersection sizes tells us whether radiation-responsive genes tend to "
    "be organ-specific or broadly active.", styles["Body"]))

story.append(PageBreak())

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 5: FOREST PLOT
# ═══════════════════════════════════════════════════════════════════════════════
story.append(Paragraph("Figure 5 — Forest Plot: Tissue Enrichment Statistics", styles["SectionHead"]))
story.append(divider())

img5 = Image("/mnt/results/fig5_tissue_enrichment_test.png", width=420, height=233)
img5.hAlign = "CENTER"
story.append(img5)
story.append(Paragraph("Figure 5: DEG rate per organ with 95% confidence intervals. Dashed line = overall rate. Significance: *** p<0.001, ** p<0.01, * p<0.05.", styles["Caption"]))

story.append(Paragraph("What This Graph Shows", styles["SubHead"]))
story.append(Paragraph(
    "A forest plot displays the <b>proportion</b> of tissue-specific genes that became DEGs for "
    "each organ, along with <b>95% confidence intervals</b> (horizontal lines) and significance "
    "markers. The dashed vertical line shows the overall DEG rate across all organs (the rate "
    "expected if tissue had no effect). Points to the right of the dashed line have higher-than-"
    "expected DEG rates (enriched); points to the left have lower-than-expected rates (depleted).", styles["Body"]))

story.append(Paragraph("Statistical Concept: Proportion, Confidence Interval, and Hypothesis Testing", styles["SubHead"]))
story.append(Paragraph(
    "This figure brings together three core statistical concepts:", styles["Body"]))

story.append(Paragraph(
    "&bull; <b>Proportion</b> (the dots): The DEG rate is a proportion — the fraction of "
    "tissue-specific genes that are DEGs. For example, Seedling's rate of 50.7% means 72 out of "
    "142 seedling-specific genes are DEGs. Proportions range from 0 to 1 (or 0% to 100%).", styles["BulletStyle"]))

story.append(Paragraph(
    "&bull; <b>95% confidence interval</b> (the horizontal lines): The CI tells us the range of "
    "values within which we are 95% confident the true DEG rate lies. Narrow lines = precise "
    "estimate (large sample); wide lines = uncertain estimate (small sample). Seedling has the "
    "widest CI because it has the fewest tissue-specific genes (142).", styles["BulletStyle"]))

story.append(Paragraph(
    "&bull; <b>Hypothesis testing</b> (the significance stars): Each organ is tested against the "
    "null hypothesis that its DEG rate equals the overall rate. *** means p < 0.001 (very strong "
    "evidence against the null), ** means p < 0.01, * means p < 0.05, and 'ns' means not "
    "significant.", styles["BulletStyle"]))

story.append(callout_box(
    "BASIC TERM: What is a confidence interval?",
    "When we calculate that 42.5% of Leaf_Shoot genes are DEGs, that's our best estimate from "
    "the sample we have. But the true rate in the entire population of Leaf_Shoot genes might be "
    "slightly different. A 95% confidence interval says: 'If we repeated this experiment many "
    "times, 95% of the time the true rate would fall within this range.' For Leaf_Shoot, the CI "
    "is 40.9%-44.3%, meaning we're fairly confident the true rate is close to our estimate."
))

story.append(callout_box(
    "INTERMEDIATE TERM: What is the odds ratio?",
    "The odds ratio (OR) compares the odds of being a DEG in one group vs. another. OR = 1 means "
    "no difference; OR > 1 means the first group has higher odds (enriched); OR < 1 means lower "
    "odds (depleted). For Leaf_Shoot, OR = 1.62 means a tissue-specific Leaf_Shoot gene has 1.62 "
    "times the odds of being a DEG compared to a gene from any other organ. For Seed_Silique, "
    "OR = 0.65 means its genes have 35% lower odds of being DEGs. The OR is the test statistic "
    "used in Fisher's exact test (see Code Block 3)."
))

story.append(Paragraph("How to Read It", styles["SubHead"]))
story.append(Paragraph(
    "&bull; <b>Dots</b> = estimated DEG rate for each organ.", styles["BulletStyle"]))
story.append(Paragraph(
    "&bull; <b>Horizontal lines</b> = 95% confidence interval. If the line does not cross the "
    "dashed vertical line, the organ's rate is significantly different from the overall rate.", styles["BulletStyle"]))
story.append(Paragraph(
    "&bull; <b>Color</b> = blue = enriched (more DEGs than expected), orange = depleted (fewer).", styles["BulletStyle"]))
story.append(Paragraph(
    "&bull; <b>Stars</b> = statistical significance after Bonferroni correction for multiple testing.", styles["BulletStyle"]))

story.append(Paragraph("What the Data Tell Us", styles["SubHead"]))
story.append(Paragraph(
    "Seedling (50.7%) and Leaf_Shoot (42.5%) are significantly enriched for DEGs — their "
    "confidence intervals are entirely to the right of the dashed line. Seed_Silique (27.4%) and "
    "Flower (31.6%) are significantly depleted — their CIs are entirely to the left. Root (36.2%) "
    "is not significantly different from the overall rate (34.9%), as its CI overlaps the dashed "
    "line. This means leaf/shoot and seedling tissues show a stronger transcriptional response to "
    "radiation than expected, while reproductive tissues (seed/silique) and flowers show a weaker "
    "response.", styles["Body"]))

story.append(PageBreak())

print("Part I (all 5 figures) added. Continuing to Part II...")


Part I (all 5 figures) added. Continuing to Part II...


Add Part II: code block explanations

In [77]:

# ── PART II: KEY CODE BLOCKS EXPLAINED ─────────────────────────────────────────
story.append(Paragraph("Part II: Key Code Blocks Explained", styles["SectionHead"]))
story.append(divider())
story.append(Paragraph(
    "This section explains the statistical reasoning behind the five most important code blocks "
    "in the analysis. For each block, we show the code, explain what statistical test or concept "
    "it implements, and walk through the output.", styles["Body"]))

story.append(PageBreak())

# ═══════════════════════════════════════════════════════════════════════════════
# CODE BLOCK 1: TAU INDEX
# ═══════════════════════════════════════════════════════════════════════════════
story.append(Paragraph("Code Block 1 — The Tau Tissue-Specificity Index", styles["SectionHead"]))
story.append(divider())

story.append(Paragraph("The Code", styles["SubHead"]))
story.append(code_block(
    'compute_tau <- function(expr_matrix) {\n'
    '  expr_matrix[expr_matrix < 0] <- 0\n'
    '  apply(expr_matrix, 1, function(x) {\n'
    '    mx <- max(x, na.rm = TRUE)\n'
    '    if (mx == 0) return(NA)\n'
    '    n <- sum(!is.na(x))\n'
    '    sum(1 - (x / mx), na.rm = TRUE) / (n - 1)\n'
    '  })\n'
    '}\n\n'
    'tau_subtissue <- compute_tau(sub_tissue_expr)\n'
    'specificity <- ifelse(tau_subtissue >= 0.6, "tissue_specific", "constitutive")'
))

story.append(Paragraph("What This Code Does", styles["SubHead"]))
story.append(Paragraph(
    "This function computes the <b>Tau index</b> for every gene, measuring how tissue-specific "
    "its expression pattern is. The <font face='Courier'>apply()</font> function with "
    "<font face='Courier'>MARGIN = 1</font> iterates over each row (gene) of the expression matrix. "
    "For each gene, it normalizes all tissue expression values by dividing by the maximum, "
    "subtracts each from 1, sums these, and divides by (n - 1) where n is the number of tissues.", styles["Body"]))

story.append(Paragraph("Statistical Concept: Tissue-Specificity Metric", styles["SubHead"]))
story.append(Paragraph(
    "The Tau index is a <b>specificity metric</b> — a single number that summarizes how unevenly "
    "a gene's expression is distributed across tissues. It is based on the idea of normalizing "
    "each tissue's expression by the maximum expression observed for that gene:", styles["Body"]))

story.append(code_block(
    "Tau = sum(1 - x_i / max(x)) / (n - 1)\n\n"
    "  x_i     = expression of gene in tissue i\n"
    "  max(x)  = highest expression across all tissues\n"
    "  n       = number of tissues\n\n"
    "Range: 0 (ubiquitous) to 1 (perfectly tissue-specific)"
))

story.append(callout_box(
    "BASIC TERM: What does 'normalizing by the max' mean?",
    "Imagine a gene expressed at levels [100, 10, 5, 2, 1] across 5 tissues. The max is 100. "
    "Dividing each by 100 gives [1.0, 0.1, 0.05, 0.02, 0.01]. Now subtract each from 1: "
    "[0, 0.9, 0.95, 0.98, 0.99]. Sum = 3.82. Divide by (5-1) = 4. Tau = 0.955. This gene is "
    "almost exclusively expressed in tissue 1 — it's highly tissue-specific. Now imagine a gene "
    "expressed at [100, 98, 95, 92, 90]. Same max (100), but the ratios are [1.0, 0.98, 0.95, "
    "0.92, 0.90]. Subtract from 1: [0, 0.02, 0.05, 0.08, 0.10]. Sum = 0.25. Divide by 4. "
    "Tau = 0.063. This gene is expressed nearly everywhere — it's constitutive."
))

story.append(callout_box(
    "INTERMEDIATE TERM: Why Tau instead of other specificity metrics?",
    "Several metrics exist for tissue-specificity: Tau, the Gini coefficient, the entropy-based "
    "specificity score (H), and the tissue specificity index (TSI). Tau is preferred because it "
    "is (1) robust to noise — small fluctuations in low-expression tissues have minimal impact, "
    "(2) bounded between 0 and 1, making interpretation straightforward, and (3) does not require "
    "any distributional assumptions about the expression data. The threshold of 0.6 is a widely "
    "used convention in the plant genomics literature, balancing sensitivity (catching real "
    "tissue-specific genes) with specificity (not misclassifying broadly expressed genes)."
))

story.append(Paragraph("The Output", styles["SubHead"]))
story.append(Paragraph(
    "After computing Tau, each gene is classified as either <b>tissue-specific</b> (Tau >= 0.6) "
    "or <b>constitutive</b> (Tau < 0.6). Genes not present in the AtGenExpress atlas are labeled "
    "<b>unannotated</b>. For this dataset:", styles["Body"]))

story.append(stat_box("Classification Results", [
    "Tissue-specific DEGs: 3,704 (53.4% of all DEGs)",
    "Constitutive DEGs: 2,115 (30.5%)",
    "Unannotated DEGs: 1,123 (16.2%)",
    "Mean Tau (all atlas genes): 0.662",
    "Median Tau: 0.661",
]))

story.append(PageBreak())

# ═══════════════════════════════════════════════════════════════════════════════
# CODE BLOCK 2: CHI-SQUARE TEST
# ═══════════════════════════════════════════════════════════════════════════════
story.append(Paragraph("Code Block 2 — Chi-Square Test of Independence", styles["SectionHead"]))
story.append(divider())

story.append(Paragraph("The Code", styles["SubHead"]))
story.append(code_block(
    '# Build contingency table: organ x DEG status\n'
    'contingency <- table(tissue_specific$predominant_broad_organ,\n'
    '                     tissue_specific$deg_flag)\n\n'
    '# Chi-square test of independence\n'
    'chi_result <- chisq.test(contingency)\n'
    'print(chi_result)'
))

story.append(Paragraph("What This Code Does", styles["SubHead"]))
story.append(Paragraph(
    "First, <font face='Courier'>table()</font> builds a <b>contingency table</b> — a matrix that "
    "counts how many genes fall into each combination of organ (rows) and DEG status (columns: "
    "yes/no). Then <font face='Courier'>chisq.test()</font> performs a <b>chi-square test of "
    "independence</b>, which asks: 'Is DEG status independent of tissue assignment, or does the "
    "probability of being a DEG depend on which tissue a gene belongs to?'", styles["Body"]))

story.append(Paragraph("Statistical Concept: Chi-Square Test of Independence", styles["SubHead"]))
story.append(Paragraph(
    "The chi-square test compares the <b>observed</b> counts in each cell of the contingency table "
    "to the <b>expected</b> counts we would see if tissue and DEG status were truly independent "
    "(the null hypothesis). If the observed counts deviate substantially from expected, we reject "
    "the null hypothesis and conclude that tissue matters.", styles["Body"]))

story.append(code_block(
    "Chi-square statistic: X^2 = sum((observed - expected)^2 / expected)\n\n"
    "  observed  = actual count in each cell\n"
    "  expected  = count expected if tissue and DEG status were independent\n"
    "             = (row total x column total) / grand total\n\n"
    "Degrees of freedom: df = (rows - 1) x (cols - 1) = (5 - 1) x (2 - 1) = 4"
))

story.append(callout_box(
    "BASIC TERM: What is a null hypothesis?",
    "The null hypothesis (H0) is the 'nothing is going on' assumption. Here, H0 says: 'The "
    "probability of a gene being a DEG is the same regardless of which tissue it belongs to.' "
    "The test calculates how likely the observed data would be if this boring scenario were true. "
    "If the data would be very unlikely under H0 (small p-value), we reject H0 and conclude that "
    "tissue does matter. If the data are plausible under H0 (large p-value), we fail to reject "
    "H0 — we don't have evidence that tissue matters."
))

story.append(callout_box(
    "INTERMEDIATE TERM: What are residuals?",
    "The chi-square test also produces <b>residuals</b> — the standardized difference between "
    "observed and expected counts for each cell: (observed - expected) / sqrt(expected). A "
    "positive residual means more DEGs than expected (enriched); negative means fewer (depleted). "
    "Residuals above +2 or below -2 are conventionally considered noteworthy. For this data, "
    "Leaf_Shoot has a residual of +7.5 (strongly enriched) and Seed_Silique has -5.9 (strongly "
    "depleted), immediately revealing which tissues drive the overall significant result."
))

story.append(Paragraph("The Output", styles["SubHead"]))
story.append(stat_box("Chi-Square Test Results", [
    "X-squared = 174.2",
    "Degrees of freedom = 4",
    "p-value = 1.31 x 10^-36",
    "",
    "Conclusion: REJECT H0 (p << 0.05)",
    "Tissue assignment and DEG status are NOT independent.",
    "There IS a statistically significant tissue-specific radiation response.",
    "",
    "Key residuals (DEG column):",
    "  Leaf_Shoot: +7.5 (strongly enriched)",
    "  Seed_Silique: -5.9 (strongly depleted)",
    "  Flower: -3.4 (depleted)",
    "  Seedling: +3.2 (enriched)",
    "  Root: +0.7 (near expected)",
]))

story.append(PageBreak())

# ═══════════════════════════════════════════════════════════════════════════════
# CODE BLOCK 3: FISHER'S EXACT TEST
# ═══════════════════════════════════════════════════════════════════════════════
story.append(Paragraph("Code Block 3 — Fisher's Exact Test (Per-Tissue)", styles["SectionHead"]))
story.append(divider())

story.append(Paragraph("The Code", styles["SubHead"]))
story.append(code_block(
    'for (org in organs) {\n'
    '  # 2x2 table: this organ vs all others x DEG vs non-DEG\n'
    '  org_yes <- contingency[org, "yes"]\n'
    '  org_no  <- contingency[org, "no"]\n'
    '  rest_yes <- sum(contingency[, "yes"]) - org_yes\n'
    '  rest_no  <- sum(contingency[, "no"]) - org_no\n'
    '\n'
    '  mat <- matrix(c(org_yes, org_no, rest_yes, rest_no), nrow = 2)\n'
    '  ft <- fisher.test(mat, alternative = "two.sided")\n'
    '}\n'
    '\n'
    '# Bonferroni correction for 5 tests\n'
    'fisher_results$padj <- p.adjust(fisher_results$p_value, method = "bonferroni")'
))

story.append(Paragraph("What This Code Does", styles["SubHead"]))
story.append(Paragraph(
    "For each organ, the code builds a 2x2 table comparing that organ to all other organs combined, "
    "crossed with DEG status (yes/no). It then runs <font face='Courier'>fisher.test()</font> to "
    "test whether that organ's DEG rate differs significantly from the rest. Finally, "
    "<font face='Courier'>p.adjust()</font> applies the <b>Bonferroni correction</b> for the fact "
    "that we are running 5 tests (one per organ).", styles["Body"]))

story.append(Paragraph("Statistical Concept: Fisher's Exact Test", styles["SubHead"]))
story.append(Paragraph(
    "Fisher's exact test is an alternative to the chi-square test for 2x2 tables. While the "
    "chi-square test approximates the p-value using a theoretical distribution, Fisher's test "
    "calculates the <b>exact</b> p-value by enumerating all possible tables that could arise under "
    "the null hypothesis. This makes it more accurate for small sample sizes, where the chi-square "
    "approximation may be unreliable.", styles["Body"]))

story.append(callout_box(
    "BASIC TERM: What is the odds ratio?",
    "The odds ratio (OR) is the test statistic from Fisher's test. 'Odds' is different from "
    "probability: if 72 out of 142 seedling genes are DEGs, the probability is 72/142 = 0.507, "
    "but the odds are 72/(142-72) = 72/70 = 1.03 (ratio of 'yes' to 'no'). The odds ratio compares "
    "the odds in two groups: OR = odds(seedling) / odds(rest). OR = 1 means no difference; "
    "OR > 1 means the first group has higher odds; OR < 1 means lower odds. For Seedling, "
    "OR = 1.94, meaning seedling-specific genes have nearly double the odds of being DEGs."
))

story.append(callout_box(
    "INTERMEDIATE TERM: Why Bonferroni correction?",
    "When we run multiple statistical tests, the probability of getting at least one false "
    "positive increases. If we run 5 tests at alpha = 0.05, the chance of at least one false "
    "positive is 1 - (0.95)^5 = 23%. The <b>Bonferroni correction</b> controls this by dividing "
    "the significance threshold by the number of tests: 0.05 / 5 = 0.01. A result is significant "
    "only if its adjusted p-value (padj) is below 0.05. This is the simplest and most conservative "
    "multiple testing correction. Alternatives include Benjamini-Hochberg (FDR control), which is "
    "less conservative and more commonly used in genomics, but Bonferroni is appropriate here "
    "because we have only 5 tests."
))

story.append(Paragraph("The Output", styles["SubHead"]))
story.append(make_table(
    ["Organ", "DEG Rate", "vs Rest", "Odds Ratio", "padj", "Direction"],
    [
        ["Seedling", "50.7%", "34.7%", "1.94", "6.0e-04", "Enriched ***"],
        ["Leaf_Shoot", "42.5%", "31.4%", "1.62", "1.4e-27", "Enriched ***"],
        ["Root", "36.2%", "34.7%", "1.07", "1.00", "ns"],
        ["Flower", "31.6%", "36.7%", "0.80", "5.2e-07", "Depleted ***"],
        ["Seed_Silique", "27.4%", "36.8%", "0.65", "2.7e-16", "Depleted ***"],
    ],
    [80, 60, 55, 65, 65, 100]
))

story.append(Spacer(1, 6))
story.append(Paragraph(
    "Root is the only organ that is not significantly different from the rest (padj = 1.00). "
    "Its DEG rate (36.2%) is close to the overall rate (34.9%), so the data are consistent with "
    "the null hypothesis for root specifically — even though the overall chi-square test (Code "
    "Block 2) showed that tissue matters in general.", styles["Body"]))

story.append(PageBreak())

# ═══════════════════════════════════════════════════════════════════════════════
# CODE BLOCK 4: PAIRWISE FISHER'S TESTS
# ═══════════════════════════════════════════════════════════════════════════════
story.append(Paragraph("Code Block 4 — Pairwise Fisher's Exact Tests", styles["SectionHead"]))
story.append(divider())

story.append(Paragraph("The Code", styles["SubHead"]))
story.append(code_block(
    'for (i in 1:(length(organs) - 1)) {\n'
    '  for (j in (i + 1):length(organs)) {\n'
    '    mat <- matrix(c(\n'
    '      contingency[organs[i], "yes"], contingency[organs[i], "no"],\n'
    '      contingency[organs[j], "yes"], contingency[organs[j], "no"]\n'
    '    ), nrow = 2)\n'
    '    ft <- fisher.test(mat, alternative = "two.sided")\n'
    '  }\n'
    '}\n'
    '\n'
    '# Bonferroni correction for C(5,2) = 10 pairwise tests\n'
    'pairwise_results$padj <- p.adjust(pairwise_results$p_value, method = "bonferroni")'
))

story.append(Paragraph("What This Code Does", styles["SubHead"]))
story.append(Paragraph(
    "This nested loop compares every pair of organs directly. With 5 organs, there are "
    "C(5,2) = 10 possible pairs. For each pair, it builds a 2x2 table (organ1 x organ2, crossed "
    "with DEG status) and runs Fisher's exact test. The Bonferroni correction now divides by 10 "
    "instead of 5, because we are running 10 tests.", styles["Body"]))

story.append(Paragraph("Statistical Concept: Pairwise Comparisons and Multiple Testing", styles["SubHead"]))
story.append(Paragraph(
    "The per-tissue test (Code Block 3) compares each organ to 'all others combined.' But the "
    "user's question — 'are roots responding more than shoots?' — requires a <b>direct pairwise "
    "comparison</b>. This code block answers that by testing each organ against each other organ "
    "individually.", styles["Body"]))

story.append(callout_box(
    "BASIC TERM: What is a combination C(n,2)?",
    "If you have 5 organs and want to compare every pair, the number of unique pairs is "
    "'5 choose 2' = C(5,2) = 5! / (2! x 3!) = 10. The nested loop generates these 10 pairs: "
    "(Flower, Leaf_Shoot), (Flower, Root), (Flower, Seed_Silique), (Flower, Seedling), "
    "(Leaf_Shoot, Root), and so on. Each pair gets its own Fisher's exact test."
))

story.append(callout_box(
    "INTERMEDIATE TERM: The multiple testing problem escalates",
    "With 10 pairwise tests at alpha = 0.05, the probability of at least one false positive under "
    "the global null is 1 - (0.95)^10 = 40%. The Bonferroni correction becomes more stringent: "
    "0.05 / 10 = 0.005. Despite this stricter threshold, 9 of 10 comparisons remain significant, "
    "indicating robust differences between organs. The only non-significant pair is "
    "Seedling vs. Leaf_Shoot (padj = 0.57), likely because both are enriched and their confidence "
    "intervals overlap."
))

story.append(Paragraph("The Output", styles["SubHead"]))
story.append(Paragraph(
    "The pairwise results establish a clear ranking of radiation responsiveness:", styles["Body"]))

story.append(stat_box("Pairwise Ranking (DEG rate, highest to lowest)", [
    "1. Seedling: 50.7%",
    "2. Leaf_Shoot: 42.5%",
    "3. Root: 36.2%",
    "4. Flower: 31.6%",
    "5. Seed_Silique: 27.4%",
    "",
    "9 of 10 pairwise comparisons significant (padj < 0.05)",
    "Only non-significant: Seedling vs. Leaf_Shoot (padj = 0.57)",
    "",
    "Direct answer to 'are roots responding more than shoots?':",
    "  Leaf_Shoot > Root: OR = 1.31, padj = 0.001",
    "  Yes — leaf/shoot genes are significantly more likely to be DEGs.",
]))

story.append(PageBreak())

# ═══════════════════════════════════════════════════════════════════════════════
# CODE BLOCK 5: DIRECTION-OF-RESPONSE TEST
# ═══════════════════════════════════════════════════════════════════════════════
story.append(Paragraph("Code Block 5 — Direction-of-Response Chi-Square Test", styles["SectionHead"]))
story.append(divider())

story.append(Paragraph("The Code", styles["SubHead"]))
story.append(code_block(
    '# For tissue-specific DEGs only, test if up/down ratio\n'
    '# differs across organs\n'
    'direction_table <- table(ts_degs$predominant_broad_organ,\n'
    '                         ts_degs$regulation)\n'
    '\n'
    '# Chi-square test: is up/down ratio independent of tissue?\n'
    'chi_dir <- chisq.test(direction_table)'
))

story.append(Paragraph("What This Code Does", styles["SubHead"]))
story.append(Paragraph(
    "This builds a different contingency table: organ (rows) x regulation direction (columns: "
    "up/down), using only the DEGs (not all genes). The chi-square test asks whether the "
    "<b>proportion of up vs. down regulation</b> is the same across all organs, or whether some "
    "tissues are predominantly upregulated while others are predominantly downregulated.", styles["Body"]))

story.append(Paragraph("Statistical Concept: Testing for Qualitative Differences", styles["SubHead"]))
story.append(Paragraph(
    "The previous tests asked 'how many genes respond?' (quantity). This test asks 'in which "
    "direction do they respond?' (quality). Even if two organs have the same number of DEGs, they "
    "could differ in whether those DEGs are mostly up or mostly down. This is a fundamentally "
    "different biological question: upregulation suggests activation of pathways, while "
    "downregulation suggests suppression.", styles["Body"]))

story.append(callout_box(
    "BASIC TERM: What does 'up' vs. 'down' mean biologically?",
    "When a gene is 'upregulated' after radiation, it means the plant is producing more of that "
    "gene's protein — typically because the pathway is being activated (e.g., a DNA repair gene "
    "is upregulated because the cell needs more repair protein). 'Downregulated' means less "
    "protein is being made — the pathway is being suppressed (e.g., a cell cycle gene is "
    "downregulated because the cell pauses division to repair damage first). The balance of up "
    "vs. down tells us whether a tissue is primarily 'turning things on' or 'shutting things down' "
    "in response to radiation."
))

story.append(callout_box(
    "INTERMEDIATE TERM: Why a separate test for direction?",
    "The first chi-square test (Code Block 2) used a 5x2 table (organ x DEG yes/no). It could "
    "detect that organs differ in DEG rate, but it treated all DEGs the same regardless of "
    "direction. This second test uses a 5x2 table (organ x up/down) among DEGs only. It can "
    "detect that organs differ in response direction even if they have similar DEG counts. "
    "For example, Root and Flower have similar DEG rates (36.2% vs. 31.6%), but Root is 64% "
    "downregulated while Flower is 57% upregulated — a qualitative difference that the first "
    "test would miss."
))

story.append(Paragraph("The Output", styles["SubHead"]))
story.append(stat_box("Direction-of-Response Test Results", [
    "X-squared = 112.3, df = 4, p-value = 2.33 x 10^-23",
    "",
    "Conclusion: REJECT H0 (p << 0.05)",
    "The up/down regulation ratio DIFFERS significantly across tissues.",
    "",
    "Per-organ up/down ratios:",
    "  Seedling:      88.9% up  (predominantly UPREGULATED)",
    "  Seed_Silique:  60.8% up",
    "  Flower:        57.1% up",
    "  Leaf_Shoot:    51.8% up  (roughly balanced)",
    "  Root:          35.6% up  (predominantly DOWNREGULATED)",
    "",
    "Biological interpretation:",
    "  Root: radiation suppresses gene expression (cell cycle arrest)",
    "  Seedling: radiation activates gene expression (stress response)",
]))

story.append(PageBreak())

print("Part II (all 5 code blocks) added. Continuing to Part III...")


Part II (all 5 code blocks) added. Continuing to Part III...


Add Part III glossary and build the PDF

In [79]:

# ── PART III: STATISTICAL GLOSSARY ─────────────────────────────────────────────
story.append(Paragraph("Part III: Statistical Glossary", styles["SectionHead"]))
story.append(divider())
story.append(Paragraph(
    "Quick reference for the statistical terms used throughout this document, organized from "
    "basic to intermediate.", styles["Body"]))

# Glossary table
glossary_data = [
    ["Term", "Level", "Definition"],
    ["Frequency", "Basic", "How many times something occurs. A frequency distribution shows counts for all categories."],
    ["Proportion", "Basic", "A fraction or percentage: the count in one category divided by the total count."],
    ["Median", "Basic", "The middle value when data are sorted. Half the values are above it, half below."],
    ["Null hypothesis (H0)", "Basic", "The 'nothing is going on' assumption. The test asks: could the data plausibly arise if H0 were true?"],
    ["P-value", "Basic", "The probability of observing data this extreme (or more) if H0 were true. Small p-value = evidence against H0."],
    ["Statistical significance", "Basic", "When p < 0.05 (conventional threshold), we reject H0 and call the result 'significant.'"],
    ["Confidence interval (CI)", "Basic", "A range of values that likely contains the true parameter. A 95% CI means 95% of repeated experiments would capture the true value."],
    ["Odds ratio (OR)", "Intermediate", "Ratio of odds between two groups. OR = 1: no difference; OR > 1: group 1 has higher odds; OR < 1: lower odds."],
    ["Effect size", "Intermediate", "The magnitude of a difference (e.g., log2 fold change). Distinct from significance, which measures confidence."],
    ["Contingency table", "Intermediate", "A matrix counting observations across two categorical variables (e.g., organ x DEG status)."],
    ["Chi-square test", "Intermediate", "Tests whether two categorical variables are independent by comparing observed vs. expected counts."],
    ["Fisher's exact test", "Intermediate", "An exact test for 2x2 tables; calculates the precise p-value rather than approximating it. Preferred for small samples."],
    ["Degrees of freedom (df)", "Intermediate", "The number of independent pieces of information. For a contingency table: (rows-1) x (cols-1)."],
    ["Residuals", "Intermediate", "Standardized difference between observed and expected counts. |residual| > 2 indicates that cell deviates from expectation."],
    ["Bonferroni correction", "Intermediate", "Divides the significance threshold by the number of tests to control the family-wise error rate. Conservative but simple."],
    ["Z-score", "Intermediate", "Number of standard deviations a value is from the mean. Used to standardize variables on different scales."],
    ["Logarithmic transformation", "Intermediate", "Applying log (e.g., log10 or log2) to spread out values spanning many orders of magnitude. Used in volcano plots."],
    ["Tau index", "Intermediate", "A specificity metric from 0 (ubiquitous) to 1 (tissue-specific). Threshold 0.6 separates tissue-specific from constitutive genes."],
    ["Multiple testing problem", "Intermediate", "Running many tests increases false positives. Corrections (Bonferroni, BH-FDR) adjust p-values to control this."],
]

glossary_table = make_table(
    glossary_data[0],
    glossary_data[1:],
    [110, 70, 280]
)
story.append(glossary_table)

story.append(Spacer(1, 20))
story.append(divider())
story.append(Spacer(1, 10))

story.append(Paragraph(
    "<b>Key Takeaways from the Statistical Analysis</b>", styles["SubHead"]))
story.append(Paragraph(
    "1. <b>Tissue matters</b> (chi-square p = 1.3 x 10<super>-36</super>): the probability of a "
    "gene being a DEG depends significantly on which tissue it belongs to.", styles["Body"]))
story.append(Paragraph(
    "2. <b>Leaf/Shoot and Seedling are enriched</b> for DEGs (higher response rate than expected); "
    "<b>Flower and Seed/Silique are depleted</b> (lower rate). Root is not significantly different "
    "from the overall rate.", styles["Body"]))
story.append(Paragraph(
    "3. <b>9 of 10 pairwise comparisons are significant</b>, establishing a clear ranking: "
    "Seedling > Leaf_Shoot > Root > Flower > Seed_Silique.", styles["Body"]))
story.append(Paragraph(
    "4. <b>Direction differs by tissue</b> (chi-square p = 2.3 x 10<super>-23</super>): roots are "
    "predominantly downregulated (suppression), while seedlings are predominantly upregulated "
    "(activation). This is a qualitative difference, not just a quantitative one.", styles["Body"]))

# ── Build the PDF ──
doc.build(story, onFirstPage=page_header_footer, onLaterPages=page_header_footer)

# Validate
from pypdf import PdfReader
reader = PdfReader(output_path)
page_count = len(reader.pages)
file_size = os.path.getsize(output_path)
first_page_text = reader.pages[0].extract_text()

print(f"\nPDF generated: {output_path}")
print(f"Pages: {page_count}")
print(f"File size: {file_size:,} bytes")
print(f"First page text length: {len(first_page_text.strip())} chars")
assert page_count >= 2, "Too few pages"
assert file_size > 5000, "File too small"
assert len(first_page_text.strip()) > 0, "No text on first page"
print("Validation passed!")



PDF generated: /mnt/results/statistical_walkthrough_figures_and_code.pdf
Pages: 25
File size: 1,997,788 bytes
First page text length: 1198 chars
Validation passed!


Load and validate all 6 DESeq2 CSVs with new threshold

In [81]:

# ============================================================
# Step 1: Load & validate all 6 DESeq2 CSVs
# Apply threshold: padj < 0.05 AND |log2FC| >= 1
# ============================================================

library(data.table)

# Define the 6 comparison files with labels
comparison_files <- list(
  list(label = "radiation_effect",       file = "/mnt/user-uploads/DEG_unnorm_OSD498_510_radiation_effect.csv"),
  list(label = "genotype_interaction",   file = "/mnt/user-uploads/DEG_unnorm_OSD508_510_genotype_x_radiation_interaction.csv"),
  list(label = "GCR40",                  file = "/mnt/user-uploads/DEG_unnorm_OSD658_GCR40_vs_none.csv"),
  list(label = "GCR80",                  file = "/mnt/user-uploads/DEG_unnorm_OSD658_GCR80_vs_none.csv"),
  list(label = "Cs137_100cGy",           file = "/mnt/user-uploads/DEG_unnorm_OSD782_100cGy_vs_none.csv"),
  list(label = "Cs137_10cGy",            file = "/mnt/user-uploads/DEG_unnorm_OSD782_10cGy_vs_none.csv")
)

# Load all 6 files
deg_list <- list()
overview_rows <- list()

for (i in seq_along(comparison_files)) {
  cf <- comparison_files[[i]]
  dt <- fread(cf$file)
  # First column is gene_id (unnamed in file)
  colnames(dt)[1] <- "gene_id"
  
  # Apply DEG threshold: padj < 0.05 AND |log2FC| >= 1
  dt[, deg_flag := ifelse(!is.na(padj) & padj < 0.05 & abs(log2FoldChange) >= 1, "yes", "no")]
  dt[, direction := ifelse(deg_flag == "yes" & log2FoldChange > 0, "up",
                    ifelse(deg_flag == "yes" & log2FoldChange < 0, "down", "none"))]
  
  # Validate AGI IDs
  dt[, is_nuclear := grepl("^AT[1-5]G[0-9]{5}$", gene_id)]
  dt[, is_organellar := grepl("^AT[CM]G[0-9]{5}$", gene_id)]
  
  deg_list[[cf$label]] <- dt
  
  # Overview stats
  n_total <- nrow(dt)
  n_deg <- sum(dt$deg_flag == "yes", na.rm = TRUE)
  n_up <- sum(dt$direction == "up", na.rm = TRUE)
  n_down <- sum(dt$direction == "down", na.rm = TRUE)
  n_nuclear <- sum(dt$is_nuclear)
  n_organellar <- sum(dt$is_organellar)
  lfc_range <- range(dt$log2FoldChange, na.rm = TRUE)
  padj_range <- range(dt$padj[!is.na(dt$padj)], na.rm = TRUE)
  
  overview_rows[[cf$label]] <- data.table(
    comparison = cf$label,
    total_genes = n_total,
    DEGs = n_deg,
    upregulated = n_up,
    downregulated = n_down,
    pct_DEG = round(n_deg / n_total * 100, 2),
    nuclear_genes = n_nuclear,
    organellar_genes = n_organellar,
    log2FC_min = round(lfc_range[1], 3),
    log2FC_max = round(lfc_range[2], 3),
    padj_min = format(padj_range[1], scientific = TRUE, digits = 3),
    padj_max = format(padj_range[2], scientific = TRUE, digits = 3)
  )
  
  cat(sprintf("[%d/6] %s: %d total genes, %d DEGs (%d up, %d down, %.1f%%)\n",
              i, cf$label, n_total, n_deg, n_up, n_down, n_deg/n_total*100))
}

overview <- rbindlist(overview_rows)
cat("\n=== DEG Overview (all 6 comparisons) ===\n")
print(overview)

# Save overview
fwrite(overview, "/mnt/results/deg_overview_all_comparisons.csv")
cat("\nSaved: deg_overview_all_comparisons.csv\n")


Warning message:
“package ‘data.table’ was built under R version 4.4.3”
[1/6] radiation_effect: 25519 total genes, 822 DEGs (764 up, 58 down, 3.2%)
[2/6] genotype_interaction: 26072 total genes, 433 DEGs (76 up, 357 down, 1.7%)
[3/6] GCR40: 22178 total genes, 2 DEGs (1 up, 1 down, 0.0%)
[4/6] GCR80: 22178 total genes, 11 DEGs (11 up, 0 down, 0.0%)
[5/6] Cs137_100cGy: 23667 total genes, 229 DEGs (220 up, 9 down, 1.0%)
[6/6] Cs137_10cGy: 23667 total genes, 87 DEGs (79 up, 8 down, 0.4%)

=== DEG Overview (all 6 comparisons) ===
             comparison total_genes  DEGs upregulated downregulated pct_DEG
                 <char>       <int> <int>       <int>         <int>   <num>
1:     radiation_effect       25519   822         764            58    3.22
2: genotype_interaction       26072   433          76           357    1.66
3:                GCR40       22178     2           1             1    0.01
4:                GCR80       22178    11          11             0    0.05
5:         Cs

Load cached atlas and compute Tau tissue-specificity index

In [83]:

# ============================================================
# Step 2: Load AtGenExpress atlas & compute Tau index
# Reuse cached atlas from /workspace/atgenexpress_atlas.rds
# ============================================================

# Load cached atlas
atlas <- readRDS("/workspace/atgenexpress_atlas.rds")
cat("Loaded atlas. Components:\n")
cat("  sub_tissue_expr:", dim(atlas$sub_tissue_expr), "\n")
cat("  broad_organ_expr:", dim(atlas$broad_organ_expr), "\n")
cat("  agi_map_clean:", nrow(atlas$agi_map_clean), "probes\n")

# Tau function
compute_tau <- function(expr_matrix) {
  # expr_matrix: genes x tissues, values are mean expression
  tau <- apply(expr_matrix, 1, function(x) {
    x <- as.numeric(x)
    if (max(x, na.rm = TRUE) == 0) return(NA)
    sum(1 - x / max(x, na.rm = TRUE), na.rm = TRUE) / (sum(!is.na(x)) - 1)
  })
  return(tau)
}

# Compute Tau at both levels
tau_subtissue <- compute_tau(atlas$sub_tissue_expr)
tau_broad <- compute_tau(atlas$broad_organ_expr)

cat("\n=== Tau Index Summary ===\n")
cat("Sub-tissue Tau:  mean =", round(mean(tau_subtissue, na.rm=TRUE), 4),
    " median =", round(median(tau_subtissue, na.rm=TRUE), 4), "\n")
cat("Broad-organ Tau: mean =", round(mean(tau_broad, na.rm=TRUE), 4),
    " median =", round(median(tau_broad, na.rm=TRUE), 4), "\n")

# Classify: tissue-specific (Tau >= 0.6) vs constitutive (Tau < 0.6)
TAU_THRESHOLD <- 0.6

# Build tissue annotation table
gene_ids <- rownames(atlas$sub_tissue_expr)
tissue_ann <- data.table(
  gene_id = gene_ids,
  tau_subtissue = tau_subtissue[gene_ids],
  tau_broad = tau_broad[gene_ids],
  is_tissue_specific = tau_subtissue[gene_ids] >= TAU_THRESHOLD
)

# Assign predominant sub-tissue and broad organ for tissue-specific genes
# (tissue with highest expression)
for (g in gene_ids) {
  if (tissue_ann[gene_id == g, is_tissue_specific]) {
    sub_vals <- atlas$sub_tissue_expr[g, ]
    broad_vals <- atlas$broad_organ_expr[g, ]
    tissue_ann[gene_id == g, predominant_subtissue := colnames(atlas$sub_tissue_expr)[which.max(sub_vals)]]
    tissue_ann[gene_id == g, predominant_broad_organ := colnames(atlas$broad_organ_expr)[which.max(broad_vals)]]
  } else {
    tissue_ann[gene_id == g, predominant_subtissue := NA_character_]
    tissue_ann[gene_id == g, predominant_broad_organ := NA_character_]
  }
}

cat("\n=== Tissue Classification ===\n")
cat("Tissue-specific (Tau >= 0.6):", sum(tissue_ann$is_tissue_specific, na.rm=TRUE), "\n")
cat("Constitutive (Tau < 0.6):", sum(!tissue_ann$is_tissue_specific, na.rm=TRUE), "\n")

cat("\n=== DEG counts by broad organ (tissue-specific genes only) ===\n")
print(tissue_ann[is_tissue_specific == TRUE, .N, by = predominant_broad_organ][order(-N)])

cat("\n=== Sub-tissue distribution (top 10) ===\n")
print(tissue_ann[is_tissue_specific == TRUE, .N, by = predominant_subtissue][order(-N)][1:10])

# Save tissue annotation for reuse
saveRDS(tissue_ann, "/workspace/tissue_annotation.rds")
cat("\nSaved tissue_annotation.rds\n")


Loaded atlas. Components:
  sub_tissue_expr: 20833 33 
  broad_organ_expr: 20833 5 
  agi_map_clean: 21045 probes

=== Tau Index Summary ===
Sub-tissue Tau:  mean = 0.6622  median = 0.6611 
Broad-organ Tau: mean = 0.4597  median = 0.4163 

=== Tissue Classification ===
Tissue-specific (Tau >= 0.6): 12772 
Constitutive (Tau < 0.6): 8061 

=== DEG counts by broad organ (tissue-specific genes only) ===
   predominant_broad_organ     N
                    <char> <int>
1:            Seed_Silique  3735
2:                  Flower  3398
3:                    Root  3261
4:                Seedling  1237
5:              Leaf_Shoot  1141

=== Sub-tissue distribution (top 10) ===
    predominant_subtissue     N
                   <char> <int>
 1:         mature_pollen  2966
 2:                  root  1216
 3:         seed_stage_10   952
 4:        senescing_leaf   855
 5:          seed_stage_9   635
 6: shoot_apex_transition   471
 7:                  stem   433
 8:    inflorescence_apex   393
 9: 

Merge all 6 DEG tables with tissue annotations

In [85]:

# ============================================================
# Step 3: Merge DEGs with tissue annotations (all 6 comparisons)
# ============================================================

tissue_ann <- readRDS("/workspace/tissue_annotation.rds")

# Merge each comparison's DEG table with tissue annotations
deg_annotated_list <- list()

for (label in names(deg_list)) {
  dt <- deg_list[[label]]
  
  # Merge with tissue annotation by gene_id
  dt_ann <- merge(dt, tissue_ann, by = "gene_id", all.x = TRUE)
  
  # Classify: tissue-specific, constitutive, or unannotated
  dt_ann[, tissue_class := fifelse(
    is.na(tau_subtissue), "unannotated",
    fifelse(is_tissue_specific == TRUE, "tissue_specific", "constitutive")
  )]
  
  # For unannotated genes (not in atlas), check if organellar
  dt_ann[is.na(tau_subtissue), tissue_class := fifelse(
    grepl("^AT[CM]G[0-9]{5}$", gene_id), "unannotated_organellar", "unannotated"
  )]
  
  deg_annotated_list[[label]] <- dt_ann
  
  # Summary
  ts_degs <- dt_ann[deg_flag == "yes" & tissue_class == "tissue_specific", .N]
  const_degs <- dt_ann[deg_flag == "yes" & tissue_class == "constitutive", .N]
  unann_degs <- dt_ann[deg_flag == "yes" & tissue_class %like% "unannotated", .N]
  total_degs <- dt_ann[deg_flag == "yes", .N]
  
  cat(sprintf("%s: %d DEGs → %d tissue-specific, %d constitutive, %d unannotated\n",
              label, total_degs, ts_degs, const_degs, unann_degs))
}

# Save annotated DEG objects
saveRDS(deg_annotated_list, "/workspace/deg_annotated_list_v2.rds")
cat("\nSaved deg_annotated_list_v2.rds\n")

# Show tissue-specific DEG distribution by broad organ for each comparison
cat("\n=== Tissue-specific DEGs by broad organ ===\n")
for (label in names(deg_annotated_list)) {
  dt <- deg_annotated_list[[label]]
  cat(sprintf("\n--- %s ---\n", label))
  organ_dist <- dt[deg_flag == "yes" & tissue_class == "tissue_specific",
                   .N, by = predominant_broad_organ][order(-N)]
  if (nrow(organ_dist) > 0) {
    print(organ_dist)
  } else {
    cat("  (no tissue-specific DEGs)\n")
  }
}


radiation_effect: 822 DEGs → 528 tissue-specific, 63 constitutive, 231 unannotated
genotype_interaction: 433 DEGs → 286 tissue-specific, 61 constitutive, 86 unannotated
GCR40: 2 DEGs → 2 tissue-specific, 0 constitutive, 0 unannotated
GCR80: 11 DEGs → 4 tissue-specific, 5 constitutive, 2 unannotated
Cs137_100cGy: 229 DEGs → 171 tissue-specific, 7 constitutive, 51 unannotated
Cs137_10cGy: 87 DEGs → 61 tissue-specific, 3 constitutive, 23 unannotated

Saved deg_annotated_list_v2.rds

=== Tissue-specific DEGs by broad organ ===

--- radiation_effect ---
   predominant_broad_organ     N
                    <char> <int>
1:                Seedling   148
2:                    Root    99
3:                  Flower    97
4:            Seed_Silique    96
5:              Leaf_Shoot    88

--- genotype_interaction ---
   predominant_broad_organ     N
                    <char> <int>
1:                  Flower    90
2:            Seed_Silique    85
3:                    Root    75
4:                S

Check atlas structure for sub-tissue to organ mapping

In [87]:

# ============================================================
# Step 4: Export hierarchical CSVs (per comparison)
# Structure: tissue_specific_degs_v2/<label>/{organ_folders + constitutive + unannotated + master}
# ============================================================

broad_organs <- c("Root", "Seedling", "Leaf_Shoot", "Flower", "Seed_Silique")
base_dir <- "/mnt/results/tissue_specific_degs_v2"

# Get all sub-tissues per broad organ from the atlas
subtissue_to_organ <- list()
for (bo in broad_organs) {
  # Find sub-tissues belonging to this broad organ
  st_cols <- colnames(atlas$sub_tissue_expr)
  # Use the tissue annotation to find which sub-tissues map to each organ
  # We need the mapping from the atlas build - reconstruct from column names
  subtissue_to_organ[[bo]] <- st_cols
}

# Actually, let's reconstruct the sub-tissue to broad-organ mapping properly
# from the previous session's build_tissue_map logic
# The broad_organ_expr columns are the 5 organs, sub_tissue_expr columns are 33 sub-tissues
# We need to know which sub-tissues belong to which broad organ

# Let's derive this from the atlas data: for each sub-tissue, find which broad organ
# has the most tissue-specific genes assigned to it
# Better: use the sample mapping from the atlas

# The atlas has all_samples with tissue labels. Let's check what's available
cat("Atlas components:", names(atlas), "\n")
if ("all_samples" %in% names(atlas)) {
  cat("all_samples columns:", colnames(atlas$all_samples), "\n")
  cat("all_samples head:\n")
  print(head(atlas$all_samples, 3))
}


Atlas components: sub_tissue_expr broad_organ_expr all_samples agi_map_clean 
all_samples columns: sample_id series tissue_raw broad_organ sub_tissue 
all_samples head:
  sample_id  series            tissue_raw broad_organ           sub_tissue
1 ATGE_7_A2 GSE5629 seedling, green parts    Seedling seedling_green_parts
2 ATGE_7_B2 GSE5629 seedling, green parts    Seedling seedling_green_parts
3 ATGE_7_C2 GSE5629 seedling, green parts    Seedling seedling_green_parts


Build sub-tissue to broad-organ mapping

In [89]:

# Build sub-tissue to broad-organ mapping from atlas sample metadata
st_bo_map <- unique(atlas$all_samples[, c("sub_tissue", "broad_organ")])
setorder(st_bo_map, broad_organ, sub_tissue)
cat("=== Sub-tissue to Broad Organ Mapping ===\n")
print(st_bo_map)

# Create a named vector for quick lookup
st_to_bo <- setNames(st_bo_map$broad_organ, st_bo_map$sub_tissue)
cat("\nTotal sub-tissues:", length(st_to_bo), "\n")


=== Sub-tissue to Broad Organ Mapping ===
                           sub_tissue  broad_organ
124                            carpel       Flower
109                flower_stage_10_11       Flower
112                   flower_stage_12       Flower
127                   flower_stage_15       Flower
106                    flower_stage_9       Flower
145                      flower_whole       Flower
190                inflorescence_apex       Flower
166                     mature_pollen       Flower
130                           pedicel       Flower
118                             petal       Flower
115                             sepal       Flower
121                            stamen       Flower
70                       cauline_leaf   Leaf_Shoot
25                          cotyledon   Leaf_Shoot
172                         hypocotyl   Leaf_Shoot
28                       rosette_leaf   Leaf_Shoot
73                 rosette_vegetative   Leaf_Shoot
67                     senescing_leaf   

Export hierarchical CSVs for all 6 comparisons

In [91]:

# ============================================================
# Step 4 (continued): Export hierarchical CSVs for all 6 comparisons
# ============================================================

# Columns to export in DEG CSVs
export_cols <- c("gene_id", "baseMean", "log2FoldChange", "lfcSE", "stat", 
                 "pvalue", "padj", "deg_flag", "direction",
                 "tau_subtissue", "tau_broad", "predominant_subtissue", 
                 "predominant_broad_organ", "tissue_class")

total_files <- 0

for (label in names(deg_annotated_list)) {
  dt <- deg_annotated_list[[label]]
  comp_dir <- file.path(base_dir, label)
  dir.create(comp_dir, showWarnings = FALSE, recursive = TRUE)
  
  degs_only <- dt[deg_flag == "yes"]
  
  # 1. Master file: all genes with tissue annotation
  master <- dt[, ..export_cols]
  fwrite(master, file.path(comp_dir, "all_degs_with_tissue_annotation.csv"))
  total_files <- total_files + 1
  
  # 2. Constitutive DEGs
  const <- degs_only[tissue_class == "constitutive", ..export_cols]
  fwrite(const, file.path(comp_dir, "constitutive_DEGs.csv"))
  total_files <- total_files + 1
  
  # 3. Unannotated DEGs
  unann <- degs_only[tissue_class %like% "unannotated", ..export_cols]
  fwrite(unann, file.path(comp_dir, "unannotated_DEGs.csv"))
  total_files <- total_files + 1
  
  # 4. Per broad organ: folder with all_DEGs + per-sub-tissue CSVs
  for (bo in broad_organs) {
    organ_dir <- file.path(comp_dir, bo)
    dir.create(organ_dir, showWarnings = FALSE, recursive = TRUE)
    
    # All DEGs for this organ (tissue-specific genes assigned to this organ)
    organ_degs <- degs_only[tissue_class == "tissue_specific" & predominant_broad_organ == bo, ..export_cols]
    fwrite(organ_degs, file.path(organ_dir, "all_DEGs.csv"))
    total_files <- total_files + 1
    
    # Per sub-tissue within this organ
    organ_subtissues <- st_bo_map[broad_organ == bo, sub_tissue]
    for (st in organ_subtissues) {
      st_degs <- organ_degs[predominant_subtissue == st, ..export_cols]
      fwrite(st_degs, file.path(organ_dir, paste0(st, ".csv")))
      total_files <- total_files + 1
    }
  }
  
  cat(sprintf("[%s] Exported: master + constitutive + unannotated + 5 organ folders with sub-tissue CSVs\n", label))
}

cat(sprintf("\nTotal CSV files exported: %d\n", total_files))
cat(sprintf("Base directory: %s\n", base_dir))


Error: object 'sub_tissue' not found
Traceback:

1. `[.data.frame`(st_bo_map, broad_organ == bo, sub_tissue)
2. .handleSimpleError(function (cnd) 
 . {
 .     watcher$capture_plot_and_output()
 .     cnd <- sanitize_call(cnd)
 .     watcher$push(cnd)
 .     switch(on_error, continue = invokeRestart("eval_continue"), 
 .         stop = invokeRestart("eval_stop"), error = NULL)
 . }, "object 'sub_tissue' not found", base::quote(eval(expr, envir)))

Debug st_bo_map subsetting

In [93]:

# Fix: st_bo_map is a data.frame, need to use proper subsetting
# Let's check the class
class(st_bo_map)
head(st_bo_map)

# Use base R subsetting
organ_subtissues <- st_bo_map[st_bo_map$broad_organ == bo, "sub_tissue"]
print(organ_subtissues)


[1] "root"


Export hierarchical CSVs for all 6 comparisons (fixed)

In [95]:

# st_bo_map is a data.frame (not data.table). Fix the subsetting.
# Redo the full export with base R subsetting for st_bo_map

broad_organs <- c("Root", "Seedling", "Leaf_Shoot", "Flower", "Seed_Silique")
base_dir <- "/mnt/results/tissue_specific_degs_v2"

export_cols <- c("gene_id", "baseMean", "log2FoldChange", "lfcSE", "stat", 
                 "pvalue", "padj", "deg_flag", "direction",
                 "tau_subtissue", "tau_broad", "predominant_subtissue", 
                 "predominant_broad_organ", "tissue_class")

total_files <- 0

for (label in names(deg_annotated_list)) {
  dt <- deg_annotated_list[[label]]
  comp_dir <- file.path(base_dir, label)
  dir.create(comp_dir, showWarnings = FALSE, recursive = TRUE)
  
  degs_only <- dt[deg_flag == "yes"]
  
  # 1. Master file: all genes with tissue annotation
  master <- dt[, ..export_cols]
  fwrite(master, file.path(comp_dir, "all_degs_with_tissue_annotation.csv"))
  total_files <- total_files + 1
  
  # 2. Constitutive DEGs
  const <- degs_only[tissue_class == "constitutive", ..export_cols]
  fwrite(const, file.path(comp_dir, "constitutive_DEGs.csv"))
  total_files <- total_files + 1
  
  # 3. Unannotated DEGs
  unann <- degs_only[tissue_class %like% "unannotated", ..export_cols]
  fwrite(unann, file.path(comp_dir, "unannotated_DEGs.csv"))
  total_files <- total_files + 1
  
  # 4. Per broad organ
  for (bo in broad_organs) {
    organ_dir <- file.path(comp_dir, bo)
    dir.create(organ_dir, showWarnings = FALSE, recursive = TRUE)
    
    organ_degs <- degs_only[tissue_class == "tissue_specific" & predominant_broad_organ == bo, ..export_cols]
    fwrite(organ_degs, file.path(organ_dir, "all_DEGs.csv"))
    total_files <- total_files + 1
    
    # Per sub-tissue within this organ (use base R for st_bo_map)
    organ_subtissues <- as.character(st_bo_map[st_bo_map$broad_organ == bo, "sub_tissue"])
    for (st in organ_subtissues) {
      st_degs <- organ_degs[predominant_subtissue == st, ..export_cols]
      fwrite(st_degs, file.path(organ_dir, paste0(st, ".csv")))
      total_files <- total_files + 1
    }
  }
  
  cat(sprintf("[%s] Exported: master + constitutive + unannotated + 5 organ folders with sub-tissue CSVs\n", label))
}

cat(sprintf("\nTotal CSV files exported: %d\n", total_files))
cat(sprintf("Base directory: %s\n", base_dir))

# Verify directory structure
cat("\n=== Directory structure (radiation_effect example) ===\n")
cat(system("find /mnt/results/tissue_specific_degs_v2/radiation_effect -type f | sort", intern = TRUE), sep = "\n")




[Execution timed out after 5 minutes. Your code took too long to complete.

Tips:
- Break down long-running operations into smaller steps
- Increase the timeout parameter if the operation genuinely needs more time]

Check if CSV export completed

In [97]:

# The timeout was likely due to the system() call at the end. Let's check if the export completed.
cat("Checking if files were exported...\n")
cat("Total files in tissue_specific_degs_v2:\n")
n <- length(list.files("/mnt/results/tissue_specific_degs_v2", recursive = TRUE))
cat(n, "files\n")

# Check one comparison
cat("\nFiles in radiation_effect:\n")
rf_files <- list.files("/mnt/results/tissue_specific_degs_v2/radiation_effect", recursive = TRUE)
cat(length(rf_files), "files\n")
cat(head(rf_files, 10), sep = "\n")


Checking if files were exported...
Total files in tissue_specific_degs_v2:
246 files

Files in radiation_effect:
41 files
all_degs_with_tissue_annotation.csv
constitutive_DEGs.csv
Flower/all_DEGs.csv
Flower/carpel.csv
Flower/flower_stage_10_11.csv
Flower/flower_stage_12.csv
Flower/flower_stage_15.csv
Flower/flower_stage_9.csv
Flower/flower_whole.csv
Flower/inflorescence_apex.csv


Generate per-comparison and cross-comparison summary tables

In [99]:

# ============================================================
# Step 5: Summary tables (per comparison + cross-comparison)
# ============================================================

broad_organs <- c("Root", "Seedling", "Leaf_Shoot", "Flower", "Seed_Silique")

# --- Per-comparison summary tables ---
for (label in names(deg_annotated_list)) {
  dt <- deg_annotated_list[[label]]
  degs <- dt[deg_flag == "yes" & tissue_class == "tissue_specific"]
  
  if (nrow(degs) == 0) {
    cat(sprintf("[%s] No tissue-specific DEGs - writing empty summary\n", label))
    empty_dt <- data.table(broad_organ = character(), sub_tissue = character(),
                           total_DEGs = integer(), upregulated = integer(),
                           downregulated = integer(), median_log2FC = numeric(),
                           mean_log2FC = numeric(), mean_tau_subtissue = numeric())
    fwrite(empty_dt, file.path("/mnt/results", paste0("tissue_deg_summary_", label, ".csv")))
    next
  }
  
  # Build hierarchical summary
  summary_rows <- list()
  
  for (bo in broad_organs) {
    organ_degs <- degs[predominant_broad_organ == bo]
    organ_subtissues <- as.character(st_bo_map[st_bo_map$broad_organ == bo, "sub_tissue"])
    
    for (st in organ_subtissues) {
      st_degs <- organ_degs[predominant_subtissue == st]
      if (nrow(st_degs) > 0) {
        summary_rows[[length(summary_rows) + 1]] <- data.table(
          broad_organ = bo,
          sub_tissue = st,
          total_DEGs = nrow(st_degs),
          upregulated = sum(st_degs$direction == "up"),
          downregulated = sum(st_degs$direction == "down"),
          median_log2FC = round(median(st_degs$log2FoldChange, na.rm = TRUE), 3),
          mean_log2FC = round(mean(st_degs$log2FoldChange, na.rm = TRUE), 3),
          mean_tau_subtissue = round(mean(st_degs$tau_subtissue, na.rm = TRUE), 4)
        )
      }
    }
  }
  
  summary_dt <- rbindlist(summary_rows)
  fwrite(summary_dt, file.path("/mnt/results", paste0("tissue_deg_summary_", label, ".csv")))
  cat(sprintf("[%s] %d sub-tissue rows in summary\n", label, nrow(summary_dt)))
}

# --- Cross-comparison tissue summary (DEG counts: tissue x comparison) ---
cat("\n=== Building cross-comparison summary ===\n")

cross_rows <- list()
for (label in names(deg_annotated_list)) {
  dt <- deg_annotated_list[[label]]
  degs <- dt[deg_flag == "yes" & tissue_class == "tissue_specific"]
  
  for (bo in broad_organs) {
    n <- degs[predominant_broad_organ == bo, .N]
    n_up <- degs[predominant_broad_organ == bo & direction == "up", .N]
    n_down <- degs[predominant_broad_organ == bo & direction == "down", .N]
    
    # Total tissue-specific genes in atlas for this organ (for rate calculation)
    total_in_organ <- tissue_ann[is_tissue_specific == TRUE & predominant_broad_organ == bo, .N]
    
    cross_rows[[length(cross_rows) + 1]] <- data.table(
      broad_organ = bo,
      comparison = label,
      DEGs = n,
      upregulated = n_up,
      downregulated = n_down,
      total_atlas_genes = total_in_organ,
      deg_rate = round(n / total_in_organ * 100, 2)
    )
  }
}

cross_summary <- rbindlist(cross_rows)
# Reshape to wide format: organ x comparison
cross_wide <- dcast(cross_summary, broad_organ ~ comparison, value.var = "DEGs")
fwrite(cross_summary, "/mnt/results/cross_comparison_tissue_summary.csv")
cat("Saved cross_comparison_tissue_summary.csv\n")

# --- Cross-comparison direction (up/down ratios) ---
cross_dir <- dcast(cross_summary, broad_organ ~ comparison, value.var = "deg_rate")
fwrite(cross_dir, "/mnt/results/cross_comparison_direction.csv")
cat("Saved cross_comparison_direction.csv\n")

cat("\n=== Cross-comparison DEG counts (organ x comparison) ===\n")
print(cross_wide)

cat("\n=== Cross-comparison DEG rates % (organ x comparison) ===\n")
print(cross_dir)


[radiation_effect] 28 sub-tissue rows in summary
[genotype_interaction] 26 sub-tissue rows in summary
[GCR40] 1 sub-tissue rows in summary
[GCR80] 2 sub-tissue rows in summary
[Cs137_100cGy] 15 sub-tissue rows in summary
[Cs137_10cGy] 11 sub-tissue rows in summary

=== Building cross-comparison summary ===
Saved cross_comparison_tissue_summary.csv
Saved cross_comparison_direction.csv

=== Cross-comparison DEG counts (organ x comparison) ===
Key: <broad_organ>
    broad_organ Cs137_100cGy Cs137_10cGy GCR40 GCR80 genotype_interaction
         <char>        <int>       <int> <int> <int>                <int>
1:       Flower           14           2     1     3                   90
2:   Leaf_Shoot            7           2     1     1                   14
3:         Root           69          26     0     0                   75
4: Seed_Silique           46          12     0     0                   85
5:     Seedling           35          19     0     0                   22
   radiation_effec

Run 4-test statistical battery for all 6 comparisons

In [101]:

# ============================================================
# Step 6: Statistical tests (per comparison)
# 4-test battery: chi-square, per-tissue Fisher, pairwise Fisher, direction chi-square
# ============================================================

broad_organs <- c("Root", "Seedling", "Leaf_Shoot", "Flower", "Seed_Silique")

all_enrichment_stats <- list()
all_pairwise_stats <- list()
all_direction_stats <- list()

for (label in names(deg_annotated_list)) {
  dt <- deg_annotated_list[[label]]
  
  # Build contingency table: organ x DEG status (tissue-specific genes only)
  ts_genes <- dt[tissue_class == "tissue_specific"]
  if (nrow(ts_genes) == 0) {
    cat(sprintf("[%s] No tissue-specific genes - skipping stats\n", label))
    next
  }
  
  # Contingency table: broad_organ x deg_flag
  contig <- table(ts_genes$predominant_broad_organ, ts_genes$deg_flag)
  # Ensure all organs present
  for (bo in broad_organs) {
    if (!(bo %in% rownames(contig))) {
      contig <- rbind(contig, setNames(c(0, 0), colnames(contig)))
      rownames(contig)[nrow(contig)] <- bo
    }
  }
  contig <- contig[broad_organs, ]  # reorder
  
  cat(sprintf("\n=== %s ===\n", label))
  cat("Contingency table (organ x DEG):\n")
  print(contig)
  
  # --- Test 1: Chi-square test of independence ---
  deg_count <- sum(contig[, "yes"])
  if (deg_count < 5) {
    cat(sprintf("  [SKIP] Only %d DEGs - insufficient for chi-square\n", deg_count))
    chi_result <- list(statistic = NA, p.value = NA, df = NA, note = "Insufficient DEGs")
  } else {
    chi_test <- chisq.test(contig)
    cat(sprintf("  Chi-square: X2=%.2f, df=%d, p=%.2e\n", 
                chi_test$statistic, chi_test$parameter, chi_test$p.value))
    chi_result <- list(statistic = chi_test$statistic, p.value = chi_test$p.value, 
                       df = chi_test$parameter, residuals = chi_test$residuals)
  }
  
  # --- Test 2: Per-tissue Fisher's exact test (organ vs rest) ---
  fisher_rows <- list()
  total_degs <- sum(contig[, "yes"])
  total_non <- sum(contig[, "no"])
  
  for (bo in broad_organs) {
    organ_deg <- contig[bo, "yes"]
    organ_non <- contig[bo, "no"]
    rest_deg <- total_degs - organ_deg
    rest_non <- total_non - organ_non
    
    fisher_mat <- matrix(c(organ_deg, organ_non, rest_deg, rest_non), nrow = 2)
    ft <- fisher.test(fisher_mat)
    
    # Odds ratio: OR > 1 means enriched
    or <- unname(ft$estimate)
    ci_low <- ft$conf.int[1]
    ci_high <- ft$conf.int[2]
    
    # DEG rate
    organ_rate <- organ_deg / (organ_deg + organ_non)
    rest_rate <- rest_deg / (rest_deg + rest_non)
    
    fisher_rows[[bo]] <- data.table(
      comparison = label,
      broad_organ = bo,
      organ_DEGs = organ_deg,
      organ_total = organ_deg + organ_non,
      organ_rate = round(organ_rate * 100, 2),
      rest_DEGs = rest_deg,
      rest_total = rest_deg + rest_non,
      rest_rate = round(rest_rate * 100, 2),
      odds_ratio = round(or, 3),
      ci95_low = round(ci_low, 3),
      ci95_high = round(ci_high, 3),
      pvalue = ft$p.value,
      padj_bonf = p.adjust(ft$p.value, method = "bonferroni", n = 5),
      enrichment = ifelse(or > 1 & ft$p.value < 0.05, "enriched", 
                   ifelse(or < 1 & ft$p.value < 0.05, "depleted", "ns"))
    )
  }
  fisher_dt <- rbindlist(fisher_rows)
  all_enrichment_stats[[label]] <- fisher_dt
  
  cat("  Fisher's exact (organ vs rest):\n")
  print(fisher_dt[, .(broad_organ, organ_DEGs, organ_rate, odds_ratio, padj_bonf, enrichment)])
  
  # --- Test 3: Pairwise Fisher's exact tests (10 pairs) ---
  if (deg_count >= 5) {
    pairwise_rows <- list()
    pairs <- combn(broad_organs, 2, simplify = FALSE)
    
    for (pr in pairs) {
      bo1 <- pr[1]
      bo2 <- pr[2]
      mat <- matrix(c(contig[bo1, "yes"], contig[bo1, "no"],
                      contig[bo2, "yes"], contig[bo2, "no"]), nrow = 2)
      ft_pair <- fisher.test(mat)
      
      rate1 <- contig[bo1, "yes"] / (contig[bo1, "yes"] + contig[bo1, "no"])
      rate2 <- contig[bo2, "yes"] / (contig[bo2, "yes"] + contig[bo2, "no"])
      
      pairwise_rows[[paste(bo1, bo2, sep = "_vs_")]] <- data.table(
        comparison = label,
        organ_1 = bo1,
        organ_2 = bo2,
        rate_1 = round(rate1 * 100, 2),
        rate_2 = round(rate2 * 100, 2),
        odds_ratio = round(unname(ft_pair$estimate), 3),
        pvalue = ft_pair$p.value,
        padj_bonf = p.adjust(ft_pair$p.value, method = "bonferroni", n = 10),
        higher = ifelse(rate1 > rate2, bo1, bo2)
      )
    }
    pairwise_dt <- rbindlist(pairwise_rows)
    all_pairwise_stats[[label]] <- pairwise_dt
    
    n_sig <- sum(pairwise_dt$padj_bonf < 0.05)
    cat(sprintf("  Pairwise Fisher: %d/%d significant after Bonferroni\n", n_sig, nrow(pairwise_dt)))
  } else {
    all_pairwise_stats[[label]] <- NULL
  }
  
  # --- Test 4: Direction-of-response chi-square ---
  degs_only <- ts_genes[deg_flag == "yes"]
  if (nrow(degs_only) >= 5 && sum(degs_only$direction == "up") > 0 && sum(degs_only$direction == "down") > 0) {
    dir_contig <- table(degs_only$predominant_broad_organ, degs_only$direction)
    # Ensure all organs present
    for (bo in broad_organs) {
      if (!(bo %in% rownames(dir_contig))) {
        dir_contig <- rbind(dir_contig, setNames(c(0, 0), colnames(dir_contig)))
        rownames(dir_contig)[nrow(dir_contig)] <- bo
      }
    }
    dir_contig <- dir_contig[broad_organs, ]
    
    # Remove organs with 0 total
    dir_contig <- dir_contig[rowSums(dir_contig) > 0, , drop = FALSE]
    
    if (nrow(dir_contig) >= 2) {
      dir_chi <- chisq.test(dir_contig)
      cat(sprintf("  Direction chi-square: X2=%.2f, df=%d, p=%.2e\n",
                  dir_chi$statistic, dir_chi$parameter, dir_chi$p.value))
      
      # Per-organ direction percentages
      dir_pcts <- data.table(
        comparison = label,
        broad_organ = rownames(dir_contig),
        n_DEGs = rowSums(dir_contig),
        pct_up = round(dir_contig[, "up"] / rowSums(dir_contig) * 100, 1),
        pct_down = round(dir_contig[, "down"] / rowSums(dir_contig) * 100, 1)
      )
      all_direction_stats[[label]] <- list(chi = dir_chi, pcts = dir_pcts)
      print(dir_pcts)
    }
  } else {
    cat(sprintf("  [SKIP] Direction test - insufficient up/down DEGs\n"))
    all_direction_stats[[label]] <- NULL
  }
}

# Save all stats
enrichment_all <- rbindlist(all_enrichment_stats)
fwrite(enrichment_all, "/mnt/results/cross_comparison_statistics.csv")
cat("\nSaved cross_comparison_statistics.csv\n")

# Save per-comparison files
for (label in names(all_enrichment_stats)) {
  fwrite(all_enrichment_stats[[label]], 
         file.path("/mnt/results", paste0("tissue_enrichment_stats_", label, ".csv")))
  if (!is.null(all_pairwise_stats[[label]])) {
    fwrite(all_pairwise_stats[[label]], 
           file.path("/mnt/results", paste0("pairwise_comparisons_", label, ".csv")))
  }
}

cat("Saved per-comparison enrichment and pairwise stats\n")



=== radiation_effect ===
Contingency table (organ x DEG):
              
                 no  yes
  Root         3094   99
  Seedling     1019  148
  Leaf_Shoot    995   88
  Flower       2775   97
  Seed_Silique 2752   96
  Chi-square: X2=233.63, df=4, p=2.19e-49
  Fisher's exact (organ vs rest):
    broad_organ organ_DEGs organ_rate odds_ratio    padj_bonf enrichment
         <char>      <int>      <num>      <num>        <num>     <char>
1:         Root         99       3.10      0.562 5.759019e-07   depleted
2:     Seedling        148      12.68      3.674 1.545737e-30   enriched
3:   Leaf_Shoot         88       8.13      1.938 1.686661e-06   enriched
4:       Flower         97       3.38      0.637 2.669143e-04   depleted
5: Seed_Silique         96       3.37      0.637 2.557128e-04   depleted
  Pairwise Fisher: 7/10 significant after Bonferroni
  Direction chi-square: X2=62.67, df=4, p=7.97e-13
         comparison  broad_organ n_DEGs pct_up pct_down
             <char>       <ch

Run cross-comparison statistical tests (radiation type, dose, genotype)

In [103]:

# ============================================================
# Step 7: Cross-comparison statistical tests
# Test whether tissue response profiles differ across:
#   1. Radiation types (gamma Co-60 vs gamma Cs-137 vs simulated GCR)
#   2. Dose-response (OSD782: 10cGy vs 100cGy; OSD658: 40cGy vs 80cGy)
#   3. Genotype effect (radiation_effect vs genotype_interaction)
# ============================================================

cross_test_results <- list()

# --- Test 1: Radiation type comparison ---
# Compare tissue-specific DEG distribution across radiation types
# Use comparisons with sufficient DEGs: radiation_effect (Co-60), Cs137_100cGy (Cs-137), GCR80 (GCR)
# GCR40/GCR80 have too few DEGs, so focus on Co-60 vs Cs-137

cat("=== Test 1: Radiation type comparison (Co-60 vs Cs-137) ===\n")
# Build 5 organs x 2 conditions x 2 (DEG/non-DEG) table
# radiation_effect = Co-60 100Gy, Cs137_100cGy = Cs-137 100cGy
co60 <- deg_annotated_list[["radiation_effect"]][tissue_class == "tissue_specific"]
cs137 <- deg_annotated_list[["Cs137_100cGy"]][tissue_class == "tissue_specific"]

# Build organ x condition x DEG status array
rad_type_array <- array(0, dim = c(5, 2, 2),
                        dimnames = list(broad_organs, c("Co60", "Cs137"), c("no", "yes")))
for (bo in broad_organs) {
  rad_type_array[bo, "Co60", "yes"] <- sum(co60$predominant_broad_organ == bo & co60$deg_flag == "yes")
  rad_type_array[bo, "Co60", "no"]  <- sum(co60$predominant_broad_organ == bo & co60$deg_flag == "no")
  rad_type_array[bo, "Cs137", "yes"] <- sum(cs137$predominant_broad_organ == bo & cs137$deg_flag == "yes")
  rad_type_array[bo, "Cs137", "no"]  <- sum(cs137$predominant_broad_organ == bo & cs137$deg_flag == "no")
}
print(rad_type_array)

# Cochran-Mantel-Haenszel test: stratify by organ, test condition x DEG status
cmh_rad <- mantelhaen.test(rad_type_array)
cat(sprintf("CMH (radiation type): CMH=%.2f, df=%d, p=%.4e\n", 
            cmh_rad$statistic, cmh_rad$parameter, cmh_rad$p.value))
cat(sprintf("Common OR=%.3f, 95%% CI [%.3f, %.3f]\n", 
            cmh_rad$estimate, cmh_rad$conf.int[1], cmh_rad$conf.int[2]))

cross_test_results[["radiation_type_CMH"]] <- data.table(
  test = "Cochran-Mantel-Haenszel (Co-60 vs Cs-137, stratified by organ)",
  statistic = round(unname(cmh_rad$statistic), 2),
  df = unname(cmh_rad$parameter),
  pvalue = cmh_rad$p.value,
  common_OR = round(unname(cmh_rad$estimate), 3),
  ci95_low = round(cmh_rad$conf.int[1], 3),
  ci95_high = round(cmh_rad$conf.int[2], 3),
  interpretation = ifelse(cmh_rad$p.value < 0.05, 
    "Significant: tissue-specific DEG rates differ between Co-60 and Cs-137 gamma radiation",
    "Not significant: no difference in tissue-specific DEG rates between radiation types")
)

# --- Test 2: Dose-response (Cs-137: 10cGy vs 100cGy) ---
cat("\n=== Test 2: Dose-response (Cs-137: 10cGy vs 100cGy) ===\n")
cs10 <- deg_annotated_list[["Cs137_10cGy"]][tissue_class == "tissue_specific"]
cs100 <- deg_annotated_list[["Cs137_100cGy"]][tissue_class == "tissue_specific"]

dose_cs_array <- array(0, dim = c(5, 2, 2),
                       dimnames = list(broad_organs, c("10cGy", "100cGy"), c("no", "yes")))
for (bo in broad_organs) {
  dose_cs_array[bo, "10cGy", "yes"] <- sum(cs10$predominant_broad_organ == bo & cs10$deg_flag == "yes")
  dose_cs_array[bo, "10cGy", "no"]  <- sum(cs10$predominant_broad_organ == bo & cs10$deg_flag == "no")
  dose_cs_array[bo, "100cGy", "yes"] <- sum(cs100$predominant_broad_organ == bo & cs100$deg_flag == "yes")
  dose_cs_array[bo, "100cGy", "no"]  <- sum(cs100$predominant_broad_organ == bo & cs100$deg_flag == "no")
}
print(dose_cs_array)

cmh_dose_cs <- mantelhaen.test(dose_cs_array)
cat(sprintf("CMH (Cs-137 dose): CMH=%.2f, df=%d, p=%.4e\n", 
            cmh_dose_cs$statistic, cmh_dose_cs$parameter, cmh_dose_cs$p.value))
cat(sprintf("Common OR=%.3f, 95%% CI [%.3f, %.3f]\n", 
            cmh_dose_cs$estimate, cmh_dose_cs$conf.int[1], cmh_dose_cs$conf.int[2]))

cross_test_results[["dose_response_cs137_CMH"]] <- data.table(
  test = "Cochran-Mantel-Haenszel (Cs-137 10cGy vs 100cGy, stratified by organ)",
  statistic = round(unname(cmh_dose_cs$statistic), 2),
  df = unname(cmh_dose_cs$parameter),
  pvalue = cmh_dose_cs$p.value,
  common_OR = round(unname(cmh_dose_cs$estimate), 3),
  ci95_low = round(cmh_dose_cs$conf.int[1], 3),
  ci95_high = round(cmh_dose_cs$conf.int[2], 3),
  interpretation = ifelse(cmh_dose_cs$p.value < 0.05,
    "Significant: higher dose produces more tissue-specific DEGs",
    "Not significant: no dose-dependent change in tissue-specific DEG rates")
)

# --- Test 3: Dose-response (GCR: 40cGy vs 80cGy) ---
cat("\n=== Test 3: Dose-response (GCR: 40cGy vs 80cGy) ===\n")
gcr40 <- deg_annotated_list[["GCR40"]][tissue_class == "tissue_specific"]
gcr80 <- deg_annotated_list[["GCR80"]][tissue_class == "tissue_specific"]

dose_gcr_array <- array(0, dim = c(5, 2, 2),
                        dimnames = list(broad_organs, c("40cGy", "80cGy"), c("no", "yes")))
for (bo in broad_organs) {
  dose_gcr_array[bo, "40cGy", "yes"] <- sum(gcr40$predominant_broad_organ == bo & gcr40$deg_flag == "yes")
  dose_gcr_array[bo, "40cGy", "no"]  <- sum(gcr40$predominant_broad_organ == bo & gcr40$deg_flag == "no")
  dose_gcr_array[bo, "80cGy", "yes"] <- sum(gcr80$predominant_broad_organ == bo & gcr80$deg_flag == "yes")
  dose_gcr_array[bo, "80cGy", "no"]  <- sum(gcr80$predominant_broad_organ == bo & gcr80$deg_flag == "no")
}
print(dose_gcr_array)

# Too few DEGs for CMH - report descriptively
total_gcr40 <- sum(dose_gcr_array[, "40cGy", "yes"])
total_gcr80 <- sum(dose_gcr_array[, "80cGy", "yes"])
cat(sprintf("GCR 40cGy: %d DEGs, GCR 80cGy: %d DEGs - too few for CMH test\n", total_gcr40, total_gcr80))

cross_test_results[["dose_response_gCR_CMH"]] <- data.table(
  test = "Descriptive (GCR 40cGy vs 80cGy - too few DEGs for CMH)",
  statistic = NA_real_,
  df = NA_integer_,
  pvalue = NA_real_,
  common_OR = NA_real_,
  ci95_low = NA_real_,
  ci95_high = NA_real_,
  interpretation = sprintf("GCR 40cGy: %d tissue-specific DEGs, GCR 80cGy: %d - insufficient for statistical test", total_gcr40, total_gcr80)
)

# --- Test 4: Genotype effect (radiation main effect vs genotype x radiation interaction) ---
cat("\n=== Test 4: Genotype effect (radiation_effect vs genotype_interaction) ===\n")
# Compare tissue distribution of DEGs: radiation main effect vs interaction effect
rad_eff <- deg_annotated_list[["radiation_effect"]][tissue_class == "tissue_specific" & deg_flag == "yes"]
gen_int <- deg_annotated_list[["genotype_interaction"]][tissue_class == "tissue_specific" & deg_flag == "yes"]

# Chi-square: do the organ distributions differ between the two comparisons?
organ_dist_rad <- table(factor(rad_eff$predominant_broad_organ, levels = broad_organs))
organ_dist_int <- table(factor(gen_int$predominant_broad_organ, levels = broad_organs))
genotype_mat <- rbind(organ_dist_rad, organ_dist_int)
rownames(genotype_mat) <- c("radiation_effect", "genotype_interaction")
print(genotype_mat)

gen_chi <- chisq.test(genotype_mat)
cat(sprintf("Chi-square (organ distribution): X2=%.2f, df=%d, p=%.4e\n",
            gen_chi$statistic, gen_chi$parameter, gen_chi$p.value))

cross_test_results[["genotype_effect_chisq"]] <- data.table(
  test = "Chi-square (organ distribution: radiation_effect vs genotype_interaction)",
  statistic = round(unname(gen_chi$statistic), 2),
  df = unname(gen_chi$parameter),
  pvalue = gen_chi$p.value,
  common_OR = NA_real_,
  ci95_low = NA_real_,
  ci95_high = NA_real_,
  interpretation = ifelse(gen_chi$p.value < 0.05,
    "Significant: tissue distribution of DEGs differs between radiation main effect and genotype interaction",
    "Not significant: tissue distribution similar between the two comparisons")
)

# --- Test 5: Direction comparison (radiation_effect vs genotype_interaction) ---
cat("\n=== Test 5: Direction comparison (radiation_effect vs genotype_interaction) ===\n")
# Are the up/down proportions different?
dir_rad <- table(factor(rad_eff$direction, levels = c("up", "down")))
dir_int <- table(factor(gen_int$direction, levels = c("up", "down")))
dir_mat <- rbind(dir_rad, dir_int)
rownames(dir_mat) <- c("radiation_effect", "genotype_interaction")
print(dir_mat)

dir_chi <- chisq.test(dir_mat)
cat(sprintf("Chi-square (direction): X2=%.2f, df=%d, p=%.4e\n",
            dir_chi$statistic, dir_chi$parameter, dir_chi$p.value))

cross_test_results[["direction_comparison_chisq"]] <- data.table(
  test = "Chi-square (up/down direction: radiation_effect vs genotype_interaction)",
  statistic = round(unname(dir_chi$statistic), 2),
  df = unname(dir_chi$parameter),
  pvalue = dir_chi$p.value,
  common_OR = NA_real_,
  ci95_low = NA_real_,
  ci95_high = NA_real_,
  interpretation = ifelse(dir_chi$p.value < 0.05,
    "Significant: direction of DEGs differs (radiation effect is up-dominant, interaction is down-dominant)",
    "Not significant: similar direction patterns")
)

# --- Test 6: Overall tissue profile comparison across all 4 testable comparisons ---
cat("\n=== Test 6: Overall tissue profile (4 testable comparisons) ===\n")
# Compare organ x DEG-rate across radiation_effect, genotype_interaction, Cs137_100cGy, Cs137_10cGy
# Build organ x comparison contingency (DEG counts)
all4_mat <- rbind(
  organ_dist_rad,
  organ_dist_int,
  table(factor(cs137[tissue_class == "tissue_specific" & deg_flag == "yes"]$predominant_broad_organ, levels = broad_organs)),
  table(factor(cs10[tissue_class == "tissue_specific" & deg_flag == "yes"]$predominant_broad_organ, levels = broad_organs))
)
rownames(all4_mat) <- c("radiation_effect", "genotype_interaction", "Cs137_100cGy", "Cs137_10cGy")
print(all4_mat)

# Test if organ distribution is homogeneous across comparisons
all4_chi <- chisq.test(all4_mat)
cat(sprintf("Chi-square (organ x comparison): X2=%.2f, df=%d, p=%.4e\n",
            all4_chi$statistic, all4_chi$parameter, all4_chi$p.value))

cross_test_results[["overall_profile_chisq"]] <- data.table(
  test = "Chi-square (organ distribution across 4 testable comparisons)",
  statistic = round(unname(all4_chi$statistic), 2),
  df = unname(all4_chi$parameter),
  pvalue = all4_chi$p.value,
  common_OR = NA_real_,
  ci95_low = NA_real_,
  ci95_high = NA_real_,
  interpretation = ifelse(all4_chi$p.value < 0.05,
    "Significant: tissue response profiles differ across radiation conditions",
    "Not significant: tissue response profiles are similar across conditions")
)

# Save all cross-comparison tests
cross_tests_dt <- rbindlist(cross_test_results, fill = TRUE)
fwrite(cross_tests_dt, "/mnt/results/cross_comparison_statistical_tests.csv")
cat("\nSaved cross_comparison_statistical_tests.csv\n")
cat("\n=== Summary of cross-comparison tests ===\n")
print(cross_tests_dt[, .(test, statistic, pvalue, interpretation)])


=== Test 1: Radiation type comparison (Co-60 vs Cs-137) ===
, , no

             Co60 Cs137
Root         3094  2828
Seedling     1019  1122
Leaf_Shoot    995  1069
Flower       2775  2795
Seed_Silique 2752  2629

, , yes

             Co60 Cs137
Root           99    69
Seedling      148    35
Leaf_Shoot     88     7
Flower         97    14
Seed_Silique   96    46

CMH (radiation type): CMH=11.27, df=4, p=2.3653e-02
Error in round(unname(cmh_rad$estimate), 3): non-numeric argument to mathematical function
Traceback:

1. .handleSimpleError(function (cnd) 
 . {
 .     watcher$capture_plot_and_output()
 .     cnd <- sanitize_call(cnd)
 .     watcher$push(cnd)
 .     switch(on_error, continue = invokeRestart("eval_continue"), 
 .         stop = invokeRestart("eval_stop"), error = NULL)
 . }, "non-numeric argument to mathematical function", base::quote(round(unname(cmh_rad$estimate), 
 .     3)))

Debug CMH test output structure

In [105]:

# Fix: mantelhaen.test with 3D array returns a list estimate, not a simple numeric
# Let's check the structure
cat("Class of cmh_rad$estimate:", class(cmh_rad$estimate), "\n")
cat("cmh_rad$estimate:\n")
print(cmh_rad$estimate)
cat("\nClass of cmh_rad$conf.int:", class(cmh_rad$conf.int), "\n")
print(cmh_rad$conf.int)


Class of cmh_rad$estimate: NULL 
cmh_rad$estimate:
NULL

Class of cmh_rad$conf.int: NULL 
NULL


Fix CMH test with correct 2x2xK array format

In [107]:

# CMH with 3D array (5 organs x 2 conditions x 2 status) doesn't return common OR
# because it's a general CMH test (not the common OR version).
# The 3D array with df=4 is the "general association" CMH test.
# For common OR, we'd need a 2x2xK array. Let's use the correct approach.

# For radiation type comparison: use 2x2xK (condition x DEG x organ strata)
# Build as 2 (condition) x 2 (DEG) x 5 (organ strata)
rad_type_array_2x2xK <- array(0, dim = c(2, 2, 5),
                               dimnames = list(c("Co60", "Cs137"), c("no", "yes"), broad_organs))
for (bo in broad_organs) {
  rad_type_array_2x2xK["Co60", "yes", bo]  <- sum(co60$predominant_broad_organ == bo & co60$deg_flag == "yes")
  rad_type_array_2x2xK["Co60", "no", bo]   <- sum(co60$predominant_broad_organ == bo & co60$deg_flag == "no")
  rad_type_array_2x2xK["Cs137", "yes", bo] <- sum(cs137$predominant_broad_organ == bo & cs137$deg_flag == "yes")
  rad_type_array_2x2xK["Cs137", "no", bo]  <- sum(cs137$predominant_broad_organ == bo & cs137$deg_flag == "no")
}

cmh_rad_or <- mantelhaen.test(rad_type_array_2x2xK)
cat("CMH (Co-60 vs Cs-137, common OR):\n")
print(cmh_rad_or)
cat(sprintf("Common OR=%.3f, 95%% CI [%.3f, %.3f], p=%.4e\n",
            cmh_rad_or$estimate, cmh_rad_or$conf.int[1], cmh_rad_or$conf.int[2], cmh_rad_or$p.value))

# Same for Cs-137 dose-response
dose_cs_2x2xK <- array(0, dim = c(2, 2, 5),
                       dimnames = list(c("10cGy", "100cGy"), c("no", "yes"), broad_organs))
for (bo in broad_organs) {
  dose_cs_2x2xK["10cGy", "yes", bo]  <- sum(cs10$predominant_broad_organ == bo & cs10$deg_flag == "yes")
  dose_cs_2x2xK["10cGy", "no", bo]   <- sum(cs10$predominant_broad_organ == bo & cs10$deg_flag == "no")
  dose_cs_2x2xK["100cGy", "yes", bo] <- sum(cs100$predominant_broad_organ == bo & cs100$deg_flag == "yes")
  dose_cs_2x2xK["100cGy", "no", bo]  <- sum(cs100$predominant_broad_organ == bo & cs100$deg_flag == "no")
}

cmh_dose_cs_or <- mantelhaen.test(dose_cs_2x2xK)
cat("\nCMH (Cs-137 10cGy vs 100cGy, common OR):\n")
print(cmh_dose_cs_or)
cat(sprintf("Common OR=%.3f, 95%% CI [%.3f, %.3f], p=%.4e\n",
            cmh_dose_cs_or$estimate, cmh_dose_cs_or$conf.int[1], cmh_dose_cs_or$conf.int[2], cmh_dose_cs_or$p.value))


CMH (Co-60 vs Cs-137, common OR):

	Mantel-Haenszel chi-squared test with continuity correction

data:  rad_type_array_2x2xK
Mantel-Haenszel X-squared = 173.72, df = 1, p-value < 2.2e-16
alternative hypothesis: true common odds ratio is not equal to 1
95 percent confidence interval:
 0.2753776 0.3901000
sample estimates:
common odds ratio 
        0.3277572 

Common OR=0.328, 95% CI [0.275, 0.390], p=1.1406e-39
Error: object 'cs10' not found
Traceback:


Complete cross-comparison statistical tests with correct CMH format

In [109]:

# cs10 was defined in the previous kernel state but may have been lost. Redefine.
cs10 <- deg_annotated_list[["Cs137_10cGy"]][tissue_class == "tissue_specific"]
cs100 <- deg_annotated_list[["Cs137_100cGy"]][tissue_class == "tissue_specific"]

dose_cs_2x2xK <- array(0, dim = c(2, 2, 5),
                       dimnames = list(c("10cGy", "100cGy"), c("no", "yes"), broad_organs))
for (bo in broad_organs) {
  dose_cs_2x2xK["10cGy", "yes", bo]  <- sum(cs10$predominant_broad_organ == bo & cs10$deg_flag == "yes")
  dose_cs_2x2xK["10cGy", "no", bo]   <- sum(cs10$predominant_broad_organ == bo & cs10$deg_flag == "no")
  dose_cs_2x2xK["100cGy", "yes", bo] <- sum(cs100$predominant_broad_organ == bo & cs100$deg_flag == "yes")
  dose_cs_2x2xK["100cGy", "no", bo]  <- sum(cs100$predominant_broad_organ == bo & cs100$deg_flag == "no")
}

cmh_dose_cs_or <- mantelhaen.test(dose_cs_2x2xK)
cat("CMH (Cs-137 10cGy vs 100cGy, common OR):\n")
cat(sprintf("Common OR=%.3f, 95%% CI [%.3f, %.3f], p=%.4e\n",
            cmh_dose_cs_or$estimate, cmh_dose_cs_or$conf.int[1], cmh_dose_cs_or$conf.int[2], cmh_dose_cs_or$p.value))

# Now rebuild the full cross_test_results with correct values
# Redefine all needed objects
rad_eff <- deg_annotated_list[["radiation_effect"]][tissue_class == "tissue_specific" & deg_flag == "yes"]
gen_int <- deg_annotated_list[["genotype_interaction"]][tissue_class == "tissue_specific" & deg_flag == "yes"]
organ_dist_rad <- table(factor(rad_eff$predominant_broad_organ, levels = broad_organs))
organ_dist_int <- table(factor(gen_int$predominant_broad_organ, levels = broad_organs))
genotype_mat <- rbind(organ_dist_rad, organ_dist_int)
rownames(genotype_mat) <- c("radiation_effect", "genotype_interaction")
gen_chi <- chisq.test(genotype_mat)

dir_rad <- table(factor(rad_eff$direction, levels = c("up", "down")))
dir_int <- table(factor(gen_int$direction, levels = c("up", "down")))
dir_mat <- rbind(dir_rad, dir_int)
rownames(dir_mat) <- c("radiation_effect", "genotype_interaction")
dir_chi <- chisq.test(dir_mat)

all4_mat <- rbind(
  organ_dist_rad,
  organ_dist_int,
  table(factor(cs100$predominant_broad_organ, levels = broad_organs)),
  table(factor(cs10$predominant_broad_organ, levels = broad_organs))
)
rownames(all4_mat) <- c("radiation_effect", "genotype_interaction", "Cs137_100cGy", "Cs137_10cGy")
all4_chi <- chisq.test(all4_mat)

# GCR counts
gcr40 <- deg_annotated_list[["GCR40"]][tissue_class == "tissue_specific"]
gcr80 <- deg_annotated_list[["GCR80"]][tissue_class == "tissue_specific"]
total_gcr40 <- sum(gcr40$deg_flag == "yes")
total_gcr80 <- sum(gcr80$deg_flag == "yes")

# Build complete results table
cross_test_results <- list()

cross_test_results[["radiation_type_CMH"]] <- data.table(
  test = "CMH common OR (Co-60 vs Cs-137, stratified by organ)",
  statistic = round(unname(cmh_rad_or$statistic), 2),
  df = unname(cmh_rad_or$parameter),
  pvalue = cmh_rad_or$p.value,
  common_OR = round(as.numeric(cmh_rad_or$estimate), 3),
  ci95_low = round(cmh_rad_or$conf.int[1], 3),
  ci95_high = round(cmh_rad_or$conf.int[2], 3),
  interpretation = sprintf("Co-60 produces %.1fx more tissue-specific DEGs than Cs-137 (OR=%.3f, p=%.2e). Significant difference in radiation type response.",
    1/as.numeric(cmh_rad_or$estimate), as.numeric(cmh_rad_or$estimate), cmh_rad_or$p.value)
)

cross_test_results[["dose_response_cs137_CMH"]] <- data.table(
  test = "CMH common OR (Cs-137 10cGy vs 100cGy, stratified by organ)",
  statistic = round(unname(cmh_dose_cs_or$statistic), 2),
  df = unname(cmh_dose_cs_or$parameter),
  pvalue = cmh_dose_cs_or$p.value,
  common_OR = round(as.numeric(cmh_dose_cs_or$estimate), 3),
  ci95_low = round(cmh_dose_cs_or$conf.int[1], 3),
  ci95_high = round(cmh_dose_cs_or$conf.int[2], 3),
  interpretation = sprintf("100cGy produces %.1fx more tissue-specific DEGs than 10cGy (OR=%.3f, p=%.2e). Significant dose-response.",
    as.numeric(cmh_dose_cs_or$estimate), as.numeric(cmh_dose_cs_or$estimate), cmh_dose_cs_or$p.value)
)

cross_test_results[["dose_response_gCR"]] <- data.table(
  test = "Descriptive (GCR 40cGy vs 80cGy - too few DEGs for CMH)",
  statistic = NA_real_, df = NA_integer_, pvalue = NA_real_,
  common_OR = NA_real_, ci95_low = NA_real_, ci95_high = NA_real_,
  interpretation = sprintf("GCR 40cGy: %d tissue-specific DEGs, GCR 80cGy: %d - insufficient for statistical test", total_gcr40, total_gcr80)
)

cross_test_results[["genotype_effect_chisq"]] <- data.table(
  test = "Chi-square (organ distribution: radiation_effect vs genotype_interaction)",
  statistic = round(unname(gen_chi$statistic), 2),
  df = unname(gen_chi$parameter),
  pvalue = gen_chi$p.value,
  common_OR = NA_real_, ci95_low = NA_real_, ci95_high = NA_real_,
  interpretation = ifelse(gen_chi$p.value < 0.05,
    "Significant: tissue distribution of DEGs differs between radiation main effect and sog1-1 interaction",
    "Not significant: tissue distribution similar")
)

cross_test_results[["direction_comparison_chisq"]] <- data.table(
  test = "Chi-square (up/down: radiation_effect vs genotype_interaction)",
  statistic = round(unname(dir_chi$statistic), 2),
  df = unname(dir_chi$parameter),
  pvalue = dir_chi$p.value,
  common_OR = NA_real_, ci95_low = NA_real_, ci95_high = NA_real_,
  interpretation = ifelse(dir_chi$p.value < 0.05,
    "Significant: radiation effect is up-dominant, sog1-1 interaction is down-dominant",
    "Not significant: similar direction patterns")
)

cross_test_results[["overall_profile_chisq"]] <- data.table(
  test = "Chi-square (organ distribution across 4 testable comparisons)",
  statistic = round(unname(all4_chi$statistic), 2),
  df = unname(all4_chi$parameter),
  pvalue = all4_chi$p.value,
  common_OR = NA_real_, ci95_low = NA_real_, ci95_high = NA_real_,
  interpretation = ifelse(all4_chi$p.value < 0.05,
    "Significant: tissue response profiles differ across radiation conditions",
    "Not significant: tissue response profiles similar")
)

cross_tests_dt <- rbindlist(cross_test_results, fill = TRUE)
fwrite(cross_tests_dt, "/mnt/results/cross_comparison_statistical_tests.csv")

cat("\n=== Cross-Comparison Statistical Tests Summary ===\n")
print(cross_tests_dt[, .(test, statistic, pvalue, interpretation)])
cat("\nSaved cross_comparison_statistical_tests.csv\n")


CMH (Cs-137 10cGy vs 100cGy, common OR):
Common OR=2.846, 95% CI [2.121, 3.820], p=5.5798e-13

=== Cross-Comparison Statistical Tests Summary ===
                                                                        test
                                                                      <char>
1:                      CMH common OR (Co-60 vs Cs-137, stratified by organ)
2:               CMH common OR (Cs-137 10cGy vs 100cGy, stratified by organ)
3:                   Descriptive (GCR 40cGy vs 80cGy - too few DEGs for CMH)
4: Chi-square (organ distribution: radiation_effect vs genotype_interaction)
5:            Chi-square (up/down: radiation_effect vs genotype_interaction)
6:             Chi-square (organ distribution across 4 testable comparisons)
   statistic        pvalue
       <num>         <num>
1:    173.72  1.140609e-39
2:     51.99  5.579793e-13
3:        NA            NA
4:     87.06  5.531095e-18
5:    456.51 2.766838e-101
6:    209.23  4.031521e-38
                      

Generate 30 per-comparison figures (5 per comparison)

In [111]:

# ============================================================
# Step 8: Per-comparison figures (6 sets x 5 figures = 30 PNGs)
# Figure 1: Stacked bar chart (up/down by organ)
# Figure 2: 5-panel volcano plots by organ
# Figure 3: Top 50 DEGs heatmap x 33 sub-tissues
# Figure 4: UpSet diagram of organ overlaps
# Figure 5: Forest plot (DEG rates, 95% CIs, significance)
# ============================================================

library(ggplot2)
library(ComplexHeatmap)
library(circlize)
library(UpSetR)
library(grid)
library(gridExtra)

# Color palette (from previous session)
UP_COLOR <- "#0072B2"
DOWN_COLOR <- "#D55E00"
ORGAN_COLORS <- c(
  Root = "#0072B2", Seedling = "#009E73", Leaf_Shoot = "#E69F00",
  Flower = "#CC79A7", Seed_Silique = "#D55E00"
)

# Helper: safe figure generation (skip if too few DEGs)
safe_plot <- function(label, fig_num, plot_func, min_degs = 5) {
  dt <- deg_annotated_list[[label]]
  n_degs <- sum(dt$deg_flag == "yes" & dt$tissue_class == "tissue_specific")
  if (n_degs < min_degs) {
    cat(sprintf("  [SKIP] %s fig%d: only %d tissue-specific DEGs (need >= %d)\n", 
                label, fig_num, n_degs, min_degs))
    return(FALSE)
  }
  outfile <- file.path("/mnt/results", sprintf("fig%d_%s_%s.png", fig_num, 
                        c("deg_counts_by_organ","volcano_plots_by_organ","heatmap_top_degs",
                          "upset_organ_overlaps","tissue_enrichment_test")[fig_num], label))
  plot_func(outfile)
  cat(sprintf("  [OK] %s fig%d: %s\n", label, fig_num, basename(outfile)))
  return(TRUE)
}

# --- Figure 1: Stacked bar chart (up/down by organ) ---
make_fig1 <- function(label, outfile) {
  dt <- deg_annotated_list[[label]]
  degs <- dt[deg_flag == "yes" & tissue_class == "tissue_specific"]
  
  plot_data <- degs[, .N, by = .(predominant_broad_organ, direction)]
  plot_data <- plot_data[direction %in% c("up", "down")]
  plot_data[, predominant_broad_organ := factor(predominant_broad_organ, levels = broad_organs)]
  
  p <- ggplot(plot_data, aes(x = predominant_broad_organ, y = N, fill = direction)) +
    geom_bar(stat = "identity", position = "stack") +
    scale_fill_manual(values = c("up" = UP_COLOR, "down" = DOWN_COLOR),
                      labels = c("Upregulated", "Downregulated")) +
    labs(title = paste0("Tissue-Specific DEGs by Organ: ", label),
         x = "Broad Organ", y = "Number of DEGs", fill = "Direction") +
    theme_minimal(base_size = 14) +
    theme(text = element_text(family = "Liberation Sans"),
          plot.title = element_text(face = "bold", size = 16),
          axis.text.x = element_text(angle = 30, hjust = 1))
  
  ggsave(outfile, p, width = 8, height = 6, dpi = 300, bg = "white")
}

# --- Figure 2: 5-panel volcano plots by organ ---
make_fig2 <- function(label, outfile) {
  dt <- deg_annotated_list[[label]]
  ts <- dt[tissue_class == "tissue_specific"]
  
  plots <- list()
  for (bo in broad_organs) {
    organ_data <- ts[predominant_broad_organ == bo]
    if (nrow(organ_data) == 0) next
    organ_data[, sig := ifelse(deg_flag == "yes", "DEG", "NS")]
    
    p <- ggplot(organ_data, aes(x = log2FoldChange, y = -log10(padj), color = sig)) +
      geom_point(alpha = 0.6, size = 1.5) +
      scale_color_manual(values = c("DEG" = ORGAN_COLORS[bo], "NS" = "grey80")) +
      geom_vline(xintercept = c(-1, 1), linetype = "dashed", alpha = 0.5) +
      geom_hline(yintercept = -log10(0.05), linetype = "dashed", alpha = 0.5) +
      labs(title = bo, x = "log2 Fold Change", y = "-log10(padj)") +
      theme_minimal(base_size = 12) +
      theme(text = element_text(family = "Liberation Sans"),
            legend.position = "none",
            plot.title = element_text(face = "bold", color = ORGAN_COLORS[bo]))
    plots[[bo]] <- p
  }
  
  if (length(plots) > 0) {
    n_plots <- length(plots)
    ncol_grid <- min(n_plots, 3)
    nrow_grid <- ceiling(n_plots / ncol_grid)
    combined <- arrangeGrob(grobs = plots, ncol = ncol_grid, nrow = nrow_grid,
                            top = textGrob(paste0("Volcano Plots by Organ: ", label),
                                          gp = gpar(fontsize = 18, fontface = "bold")))
    ggsave(outfile, combined, width = 5 * ncol_grid, height = 4 * nrow_grid, 
           dpi = 300, bg = "white")
  }
}

# --- Figure 3: Top 50 DEGs heatmap ---
make_fig3 <- function(label, outfile) {
  dt <- deg_annotated_list[[label]]
  degs <- dt[deg_flag == "yes" & tissue_class == "tissue_specific"]
  
  # Top 50 by padj
  top50 <- degs[order(padj)][1:min(50, nrow(degs))]
  if (nrow(top50) < 5) return(FALSE)
  
  # Get expression data from atlas
  genes_in_atlas <- intersect(top50$gene_id, rownames(atlas$sub_tissue_expr))
  if (length(genes_in_atlas) < 5) return(FALSE)
  
  expr <- atlas$sub_tissue_expr[genes_in_atlas, ]
  # Z-score by gene
  expr_z <- t(scale(t(expr)))
  
  # Row annotation: direction
  row_ann <- data.frame(direction = top50[match(rownames(expr_z), gene_id), direction])
  row_anno <- rowAnnotation(Direction = row_anno$direction,
                            col = list(Direction = c("up" = UP_COLOR, "down" = DOWN_COLOR)))
  
  col_fun <- colorRamp2(c(-2, 0, 2), c("#0072B2", "white", "#D55E00"))
  
  ht <- Heatmap(expr_z, name = "Z-score", col = col_fun,
                show_row_names = FALSE, show_column_names = TRUE,
                column_names_gp = gpar(fontsize = 8, fontfamily = "Liberation Sans"),
                top_annotation = row_anno,
                column_title = paste0("Top DEGs x Sub-tissues: ", label),
                column_title_gp = gpar(fontsize = 14, fontface = "bold"),
                cluster_columns = FALSE)
  
  png(outfile, width = 3600, height = 2400, res = 300, bg = "white")
  draw(ht)
  dev.off()
}

# --- Figure 4: UpSet diagram ---
make_fig4 <- function(label, outfile) {
  dt <- deg_annotated_list[[label]]
  degs <- dt[deg_flag == "yes" & tissue_class == "tissue_specific"]
  
  # Build organ membership matrix
  organ_gene_lists <- list()
  for (bo in broad_organs) {
    organ_gene_lists[[bo]] <- degs[predominant_broad_organ == bo, gene_id]
  }
  
  # UpSet from list
  png(outfile, width = 3000, height = 1800, res = 300, bg = "white")
  print(upset(fromList(organ_gene_lists), order.by = "freq", 
              main.bar.color = "#2C2A26", sets.bar.color = unname(ORGAN_COLORS[broad_organs]),
              text.scale = 1.2))
  dev.off()
}

# --- Figure 5: Forest plot (DEG rates, 95% CIs) ---
make_fig5 <- function(label, outfile) {
  stats_dt <- all_enrichment_stats[[label]]
  if (is.null(stats_dt)) return(FALSE)
  
  # Calculate DEG rates and CIs
  plot_data <- stats_dt[, .(broad_organ, organ_DEGs, organ_total, organ_rate, 
                            odds_ratio, ci95_low, ci95_high, padj_bonf, enrichment)]
  plot_data[, sig_label := ifelse(padj_bonf < 0.001, "***",
                           ifelse(padj_bonf < 0.01, "**",
                           ifelse(padj_bonf < 0.05, "*", "ns")))]
  plot_data[, broad_organ := factor(broad_organ, levels = rev(broad_organs))]
  
  p <- ggplot(plot_data, aes(x = organ_rate, y = broad_organ)) +
    geom_vline(xintercept = mean(plot_data$organ_rate), linetype = "dashed", alpha = 0.5) +
    geom_errorbarh(aes(xmin = ci95_low * 100 * (organ_rate / odds_ratio), 
                       xmax = ci95_high * 100 * (organ_rate / odds_ratio)),
                   height = 0.2, alpha = 0.7) +
    geom_point(aes(color = enrichment), size = 4) +
    geom_text(aes(label = sig_label), hjust = -0.5, vjust = 0.3, size = 5) +
    scale_color_manual(values = c("enriched" = "#009E73", "depleted" = "#D55E00", "ns" = "grey60")) +
    labs(title = paste0("Tissue Enrichment Test: ", label),
         x = "DEG Rate (%)", y = "Broad Organ", color = "Result") +
    theme_minimal(base_size = 14) +
    theme(text = element_text(family = "Liberation Sans"),
          plot.title = element_text(face = "bold", size = 16))
  
  ggsave(outfile, p, width = 9, height = 5, dpi = 300, bg = "white")
}

# Generate all figures
fig_funcs <- list(make_fig1, make_fig2, make_fig3, make_fig4, make_fig5)
fig_names <- c("deg_counts_by_organ", "volcano_plots_by_organ", "heatmap_top_degs",
               "upset_organ_overlaps", "tissue_enrichment_test")

for (label in names(deg_annotated_list)) {
  cat(sprintf("\n--- %s ---\n", label))
  for (fig_num in 1:5) {
    outfile <- file.path("/mnt/results", sprintf("fig%d_%s_%s.png", fig_num, fig_names[fig_num], label))
    min_degs <- if (fig_num %in% c(3, 4)) 5 else 3  # heatmap and upset need more DEGs
    
    dt <- deg_annotated_list[[label]]
    n_degs <- sum(dt$deg_flag == "yes" & dt$tissue_class == "tissue_specific")
    
    if (n_degs < min_degs) {
      cat(sprintf("  [SKIP] fig%d: only %d tissue-specific DEGs\n", fig_num, n_degs))
      next
    }
    
    result <- tryCatch({
      fig_funcs[[fig_num]](label, outfile)
      cat(sprintf("  [OK] fig%d_%s_%s.png\n", fig_num, fig_names[fig_num], label))
      TRUE
    }, error = function(e) {
      cat(sprintf("  [ERROR] fig%d: %s\n", fig_num, e$message))
      FALSE
    })
  }
}

cat("\n=== Per-comparison figure generation complete ===\n")


Warning message:
“package ‘ggplot2’ was built under R version 4.4.3”
Loading required package: grid

ComplexHeatmap version 2.22.0
Bioconductor page: http://bioconductor.org/packages/ComplexHeatmap/
Github page: https://github.com/jokergoo/ComplexHeatmap
Documentation: http://jokergoo.github.io/ComplexHeatmap-reference

If you use it in published research, please cite either one:
- Gu, Z. Complex Heatmap Visualization. iMeta 2022.
- Gu, Z. Complex heatmaps reveal patterns and correlations in multidimensional 
    genomic data. Bioinformatics 2016.


The new InteractiveComplexHeatmap package can directly export static 
complex heatmaps into an interactive Shiny app with zero effort. Have a try!

This message can be suppressed by:
  suppressPackageStartupMessages(library(ComplexHeatmap))


Warning message:
“package ‘circlize’ was built under R version 4.4.3”
circlize version 0.4.18
CRAN page: https://cran.r-project.org/package=circlize
Github page: https://github.com/jokergoo/circlize
Do

Fix and regenerate heatmap figures

In [113]:

# Fix fig3 heatmap: the variable 'row_anno' was used as both a data.frame and a Heatmap annotation
# Rename to avoid collision

make_fig3_fixed <- function(label, outfile) {
  dt <- deg_annotated_list[[label]]
  degs <- dt[deg_flag == "yes" & tissue_class == "tissue_specific"]
  
  top50 <- degs[order(padj)][1:min(50, nrow(degs))]
  if (nrow(top50) < 5) return(FALSE)
  
  genes_in_atlas <- intersect(top50$gene_id, rownames(atlas$sub_tissue_expr))
  if (length(genes_in_atlas) < 5) return(FALSE)
  
  expr <- atlas$sub_tissue_expr[genes_in_atlas, ]
  expr_z <- t(scale(t(expr)))
  
  # Row annotation data
  dir_vec <- top50[match(rownames(expr_z), gene_id), direction]
  row_anno_obj <- rowAnnotation(Direction = dir_vec,
                                col = list(Direction = c("up" = UP_COLOR, "down" = DOWN_COLOR)))
  
  col_fun <- colorRamp2(c(-2, 0, 2), c("#0072B2", "white", "#D55E00"))
  
  ht <- Heatmap(expr_z, name = "Z-score", col = col_fun,
                show_row_names = FALSE, show_column_names = TRUE,
                column_names_gp = gpar(fontsize = 8),
                left_annotation = row_anno_obj,
                column_title = paste0("Top DEGs x Sub-tissues: ", label),
                column_title_gp = gpar(fontsize = 14, fontface = "bold"),
                cluster_columns = FALSE)
  
  png(outfile, width = 3600, height = 2400, res = 300, bg = "white")
  draw(ht)
  dev.off()
  return(TRUE)
}

# Regenerate fig3 for the 4 comparisons that had enough DEGs
for (label in c("radiation_effect", "genotype_interaction", "Cs137_100cGy", "Cs137_10cGy")) {
  dt <- deg_annotated_list[[label]]
  n_degs <- sum(dt$deg_flag == "yes" & dt$tissue_class == "tissue_specific")
  if (n_degs < 5) {
    cat(sprintf("  [SKIP] %s fig3: only %d DEGs\n", label, n_degs))
    next
  }
  
  outfile <- file.path("/mnt/results", sprintf("fig3_heatmap_top_degs_%s.png", label))
  result <- tryCatch({
    make_fig3_fixed(label, outfile)
    cat(sprintf("  [OK] fig3_heatmap_top_degs_%s.png\n", label))
  }, error = function(e) {
    cat(sprintf("  [ERROR] %s fig3: %s\n", label, e$message))
  })
}

# Count total figures generated
fig_files <- list.files("/mnt/results", pattern = "^fig[1-5]_.*\\.png$", full.names = FALSE)
cat(sprintf("\nTotal per-comparison figures: %d\n", length(fig_files)))
cat("Files:\n")
cat(fig_files, sep = "\n")


  [OK] fig3_heatmap_top_degs_radiation_effect.png
  [OK] fig3_heatmap_top_degs_genotype_interaction.png
  [OK] fig3_heatmap_top_degs_Cs137_100cGy.png
  [OK] fig3_heatmap_top_degs_Cs137_10cGy.png

Total per-comparison figures: 23
Files:
fig1_deg_counts_by_organ_Cs137_100cGy.png
fig1_deg_counts_by_organ_Cs137_10cGy.png
fig1_deg_counts_by_organ_GCR80.png
fig1_deg_counts_by_organ_genotype_interaction.png
fig1_deg_counts_by_organ_radiation_effect.png
fig2_volcano_plots_by_organ_Cs137_100cGy.png
fig2_volcano_plots_by_organ_Cs137_10cGy.png
fig2_volcano_plots_by_organ_GCR80.png
fig2_volcano_plots_by_organ_genotype_interaction.png
fig2_volcano_plots_by_organ_radiation_effect.png
fig3_heatmap_top_degs_Cs137_100cGy.png
fig3_heatmap_top_degs_Cs137_10cGy.png
fig3_heatmap_top_degs_genotype_interaction.png
fig3_heatmap_top_degs_radiation_effect.png
fig4_upset_organ_overlaps_Cs137_100cGy.png
fig4_upset_organ_overlaps_Cs137_10cGy.png
fig4_upset_organ_overlaps_genotype_interaction.png
fig4_upset_organ_o

Generate 4 cross-comparison figures

In [115]:

# ============================================================
# Step 9: Cross-comparison figures (4 PNGs)
# 1. Tissue response heatmap (5 organs x 6 comparisons, DEG rates)
# 2. Direction comparison (up/down ratios by organ across comparisons)
# 3. Dose-response (DEG count by tissue vs dose)
# 4. Radiation type comparison (gamma vs GCR)
# ============================================================

# Reload cross_summary if needed
cross_summary <- fread("/mnt/results/cross_comparison_tissue_summary.csv")

# --- Figure cross1: Tissue response heatmap ---
plot_data <- cross_summary[, .(broad_organ, comparison, deg_rate)]
plot_data[, broad_organ := factor(broad_organ, levels = broad_organs)]
# Order comparisons logically
comp_order <- c("radiation_effect", "genotype_interaction", "Cs137_10cGy", "Cs137_100cGy", "GCR40", "GCR80")
plot_data[, comparison := factor(comparison, levels = comp_order)]

# Build matrix for ComplexHeatmap
rate_matrix <- dcast(plot_data, broad_organ ~ comparison, value.var = "deg_rate")
rate_mat <- as.matrix(rate_matrix[, -1])
rownames(rate_mat) <- rate_matrix$broad_organ
rate_mat[is.na(rate_mat)] <- 0

col_fun <- colorRamp2(c(0, 5, 12), c("white", "#E69F00", "#D55E00"))

# Column annotations: radiation type
rad_types <- c("Gamma (Co-60)", "Gamma (Co-60) interaction", "Gamma (Cs-137)", "Gamma (Cs-137)", "Simulated GCR", "Simulated GCR")
col_anno <- HeatmapAnnotation(
  `Radiation type` = rad_types,
  col = list(`Radiation type` = c("Gamma (Co-60)" = "#0072B2", 
                                   "Gamma (Co-60) interaction" = "#56B4E9",
                                   "Gamma (Cs-137)" = "#009E73",
                                   "Simulated GCR" = "#CC79A7")),
  annotation_name_gp = gpar(fontsize = 10)
)

ht1 <- Heatmap(rate_mat, name = "DEG rate (%)", col = col_fun,
               show_row_names = TRUE, show_column_names = TRUE,
               row_names_gp = gpar(fontsize = 12),
               column_names_gp = gpar(fontsize = 10),
               top_annotation = col_anno,
               column_title = "Tissue-Specific DEG Rate by Organ x Comparison",
               column_title_gp = gpar(fontsize = 16, fontface = "bold"),
               cell_fun = function(j, i, x, y, width, height, fill) {
                 grid.text(sprintf("%.1f", rate_mat[i, j]), x, y, gp = gpar(fontsize = 10))
               })

png("/mnt/results/fig_cross1_tissue_response_heatmap.png", width = 3000, height = 2000, res = 300, bg = "white")
draw(ht1)
dev.off()
cat("[OK] fig_cross1_tissue_response_heatmap.png\n")

# --- Figure cross2: Direction comparison (grouped bar chart) ---
dir_data <- cross_summary[, .(broad_organ, comparison, upregulated, downregulated)]
dir_data[, broad_organ := factor(broad_organ, levels = broad_organs)]
dir_data[, comparison := factor(comparison, levels = comp_order)]
# Melt for grouped bars
dir_melt <- melt(dir_data, id.vars = c("broad_organ", "comparison"),
                 variable.name = "direction", value.name = "count")
dir_melt[, direction := factor(direction, levels = c("upregulated", "downregulated"))]

p2 <- ggplot(dir_melt, aes(x = broad_organ, y = count, fill = direction)) +
  geom_bar(stat = "identity", position = "dodge") +
  facet_wrap(~ comparison, ncol = 3) +
  scale_fill_manual(values = c("upregulated" = UP_COLOR, "downregulated" = DOWN_COLOR)) +
  labs(title = "DEG Direction by Organ Across Comparisons",
       x = "Broad Organ", y = "Number of DEGs", fill = "Direction") +
  theme_minimal(base_size = 12) +
  theme(text = element_text(family = "Liberation Sans"),
        plot.title = element_text(face = "bold", size = 16),
        axis.text.x = element_text(angle = 30, hjust = 1))

ggsave("/mnt/results/fig_cross2_direction_comparison.png", p2, width = 12, height = 8, dpi = 300, bg = "white")
cat("[OK] fig_cross2_direction_comparison.png\n")

# --- Figure cross3: Dose-response ---
# OSD782: 10cGy vs 100cGy; OSD658: 40cGy vs 80cGy
dose_data <- rbind(
  cross_summary[comparison == "Cs137_10cGy", .(broad_organ, dose = "10 cGy", source = "Cs-137", DEGs)],
  cross_summary[comparison == "Cs137_100cGy", .(broad_organ, dose = "100 cGy", source = "Cs-137", DEGs)],
  cross_summary[comparison == "GCR40", .(broad_organ, dose = "40 cGy", source = "GCR", DEGs)],
  cross_summary[comparison == "GCR80", .(broad_organ, dose = "80 cGy", source = "GCR", DEGs)]
)
dose_data[, broad_organ := factor(broad_organ, levels = broad_organs)]
dose_data[, dose := factor(dose, levels = c("10 cGy", "40 cGy", "80 cGy", "100 cGy"))]

p3 <- ggplot(dose_data, aes(x = dose, y = DEGs, group = broad_organ, color = broad_organ)) +
  geom_line(size = 1.2) +
  geom_point(size = 3) +
  facet_wrap(~ source, scales = "free_x") +
  scale_color_manual(values = ORGAN_COLORS) +
  labs(title = "Dose-Response: Tissue-Specific DEGs by Organ",
       x = "Dose", y = "Number of DEGs", color = "Broad Organ") +
  theme_minimal(base_size = 14) +
  theme(text = element_text(family = "Liberation Sans"),
        plot.title = element_text(face = "bold", size = 16))

ggsave("/mnt/results/fig_cross3_dose_response.png", p3, width = 10, height = 6, dpi = 300, bg = "white")
cat("[OK] fig_cross3_dose_response.png\n")

# --- Figure cross4: Radiation type comparison ---
# Compare Co-60 100Gy vs Cs-137 100cGy (both "high dose" gamma, different sources)
rad_type_data <- rbind(
  cross_summary[comparison == "radiation_effect", .(broad_organ, condition = "Co-60 (100 Gy)", DEGs, deg_rate)],
  cross_summary[comparison == "Cs137_100cGy", .(broad_organ, condition = "Cs-137 (100 cGy)", DEGs, deg_rate)]
)
rad_type_data[, broad_organ := factor(broad_organ, levels = broad_organs)]

p4 <- ggplot(rad_type_data, aes(x = broad_organ, y = deg_rate, fill = condition)) +
  geom_bar(stat = "identity", position = "dodge") +
  scale_fill_manual(values = c("Co-60 (100 Gy)" = "#0072B2", "Cs-137 (100 cGy)" = "#009E73")) +
  labs(title = "Radiation Type Comparison: Co-60 vs Cs-137",
       subtitle = "DEG rate (%) by organ — Co-60 produces 3.1x more tissue-specific DEGs (CMH p=1.1e-39)",
       x = "Broad Organ", y = "DEG Rate (%)", fill = "Radiation") +
  theme_minimal(base_size = 14) +
  theme(text = element_text(family = "Liberation Sans"),
        plot.title = element_text(face = "bold", size = 16),
        plot.subtitle = element_text(size = 11, color = "grey40"),
        axis.text.x = element_text(angle = 30, hjust = 1))

ggsave("/mnt/results/fig_cross4_radiation_type_comparison.png", p4, width = 10, height = 6, dpi = 300, bg = "white")
cat("[OK] fig_cross4_radiation_type_comparison.png\n")

cat("\n=== Cross-comparison figures complete ===\n")


Error in grid.Call.graphics(C_downvppath, name$path, name$name, strict): Viewport 'DEG rate (%)_heatmap_body_1_1' was not found
Traceback:

1. draw(ht1)
2. .local(object, ...)
3. draw(ht_list, ...)
4. draw(ht_list, ...)
5. .local(object, ...)
6. grid.draw(p)
7. grid.draw.gTree(p)
8. recordGraphics(drawGTree(x), list(x = x), getNamespace("grid"))
9. drawGTree(x)
10. grid.draw(x$children[[i]], recording = FALSE)
11. grid.draw.grob(x$children[[i]], recording = FALSE)
12. recordGraphics(drawGrob(x), list(x = x), getNamespace("grid"))
13. drawGrob(x)
14. preDraw(x)
15. preDraw.grob(x)
16. pushvpgp(x)
17. pushgrobvp(x$vp)
18. pushgrobvp.vpPath(x$vp)
19. downViewport(vp, strict = TRUE, recording = FALSE)
20. downViewport.vpPath(vp, strict = TRUE, recording = FALSE)
21. grid.Call.graphics(C_downvppath, name$path, name$name, strict)
22. .handleSimpleError(function (cnd) 
  . {
  .     watcher$capture_plot_and_output()
  .     cnd <- sanitize_call(cnd)
  .     watcher$push(cnd)
  .     switch(on

Fix cross-comparison heatmap with manual text overlay

In [117]:

# The cell_fun with grid.text is causing viewport issues. 
# Use a simpler approach: remove cell_fun, or use pheatmap instead.

# Fix fig_cross1: use ComplexHeatmap without cell_fun, add text via layer
library(ComplexHeatmap)

rate_matrix <- dcast(plot_data, broad_organ ~ comparison, value.var = "deg_rate")
rate_mat <- as.matrix(rate_matrix[, -1])
rownames(rate_mat) <- rate_matrix$broad_organ
rate_mat[is.na(rate_mat)] <- 0

col_fun <- colorRamp2(c(0, 5, 12), c("white", "#E69F00", "#D55E00"))

rad_types <- c("Gamma Co-60", "Co-60 interaction", "Gamma Cs-137", "Gamma Cs-137", "Sim. GCR", "Sim. GCR")
col_anno <- HeatmapAnnotation(
  `Radiation` = rad_types,
  col = list(`Radiation` = c("Gamma Co-60" = "#0072B2", 
                              "Co-60 interaction" = "#56B4E9",
                              "Gamma Cs-137" = "#009E73",
                              "Sim. GCR" = "#CC79A7")),
  annotation_name_gp = gpar(fontsize = 10),
  show_annotation_name = TRUE
)

# Use layer_fun instead of cell_fun for text overlay
ht1 <- Heatmap(rate_mat, name = "DEG_rate", col = col_fun,
               show_row_names = TRUE, show_column_names = TRUE,
               row_names_gp = gpar(fontsize = 12),
               column_names_gp = gpar(fontsize = 10),
               top_annotation = col_anno,
               column_title = "Tissue-Specific DEG Rate (%) by Organ x Comparison",
               column_title_gp = gpar(fontsize = 16, fontface = "bold"),
               cluster_rows = FALSE, cluster_columns = FALSE)

png("/mnt/results/fig_cross1_tissue_response_heatmap.png", width = 3000, height = 2000, res = 300, bg = "white")
draw(ht1)
# Add text overlay manually
for (i in 1:nrow(rate_mat)) {
  for (j in 1:ncol(rate_mat)) {
    # Use grid.text at computed positions
    seekViewport("DEG_rate_heatmap_body")
    grid.text(sprintf("%.1f", rate_mat[i, j]), 
              x = (j - 0.5) / ncol(rate_mat), 
              y = 1 - (i - 0.5) / nrow(rate_mat),
              gp = gpar(fontsize = 10))
  }
}
dev.off()
cat("[OK] fig_cross1_tissue_response_heatmap.png\n")


Error in grid.Call(C_getEngineDLon): could not open file '/mnt/results/fig_cross1_tissue_response_heatmap.png'
Traceback:

1. draw(ht1)
2. .local(object, ...)
3. draw(ht_list, ...)
4. draw(ht_list, ...)
5. .local(object, ...)
6. grid.draw(p)
7. grid.draw.gTree(p)
8. grid.Call(C_getEngineDLon)
9. .handleSimpleError(function (cnd) 
 . {
 .     watcher$capture_plot_and_output()
 .     cnd <- sanitize_call(cnd)
 .     watcher$push(cnd)
 .     switch(on_error, continue = invokeRestart("eval_continue"), 
 .         stop = invokeRestart("eval_stop"), error = NULL)
 . }, "could not open file '/mnt/results/fig_cross1_tissue_response_heatmap.png'", 
 .     base::quote(grid.Call(C_getEngineDLon)))

Generate heatmap with pheatmap and check other figures

In [119]:

# The issue is that the PNG device may have been left open from the previous failed attempt.
# Let's close any open devices first, then try a simpler approach.

# Close all open devices
while (!is.null(dev.list())) dev.off()

# Use pheatmap instead of ComplexHeatmap for this one - simpler and more reliable
library(pheatmap)

# Build annotation for columns
col_anno_df <- data.frame(Radiation = rad_types, row.names = colnames(rate_mat))
col_anno_colors <- list(Radiation = c("Gamma Co-60" = "#0072B2", 
                                       "Co-60 interaction" = "#56B4E9",
                                       "Gamma Cs-137" = "#009E73",
                                       "Sim. GCR" = "#CC79A7"))

pheatmap(rate_mat,
         color = colorRampPalette(c("white", "#E69F00", "#D55E00"))(100),
         cluster_rows = FALSE, cluster_cols = FALSE,
         annotation_col = col_anno_df,
         annotation_colors = col_anno_colors,
         display_numbers = TRUE,
         number_format = "%.1f",
         fontsize_number = 10,
         main = "Tissue-Specific DEG Rate (%) by Organ x Comparison",
         fontsize = 12,
         filename = "/mnt/results/fig_cross1_tissue_response_heatmap.png",
         width = 10, height = 5)

cat("[OK] fig_cross1_tissue_response_heatmap.png\n")

# Now generate the other 3 figures (they may have been generated already, check)
for (f in c("fig_cross2_direction_comparison.png", "fig_cross3_dose_response.png", "fig_cross4_radiation_type_comparison.png")) {
  path <- file.path("/mnt/results", f)
  if (file.exists(path)) {
    cat(sprintf("[EXISTS] %s (%d bytes)\n", f, file.size(path)))
  } else {
    cat(sprintf("[MISSING] %s\n", f))
  }
}


Warning message:
“package ‘pheatmap’ was built under R version 4.4.3”

Attaching package: ‘pheatmap’


The following object is masked from ‘package:ComplexHeatmap’:

    pheatmap


[OK] fig_cross1_tissue_response_heatmap.png
[MISSING] fig_cross2_direction_comparison.png
[MISSING] fig_cross3_dose_response.png
[MISSING] fig_cross4_radiation_type_comparison.png


Generate remaining 3 cross-comparison figures

In [121]:

# The other 3 figures weren't generated because the first error interrupted execution.
# Generate them now.

# --- Figure cross2: Direction comparison (grouped bar chart) ---
dir_data <- cross_summary[, .(broad_organ, comparison, upregulated, downregulated)]
dir_data[, broad_organ := factor(broad_organ, levels = broad_organs)]
dir_data[, comparison := factor(comparison, levels = comp_order)]
dir_melt <- melt(dir_data, id.vars = c("broad_organ", "comparison"),
                 variable.name = "direction", value.name = "count")
dir_melt[, direction := factor(direction, levels = c("upregulated", "downregulated"))]

p2 <- ggplot(dir_melt, aes(x = broad_organ, y = count, fill = direction)) +
  geom_bar(stat = "identity", position = "dodge") +
  facet_wrap(~ comparison, ncol = 3) +
  scale_fill_manual(values = c("upregulated" = UP_COLOR, "downregulated" = DOWN_COLOR)) +
  labs(title = "DEG Direction by Organ Across Comparisons",
       x = "Broad Organ", y = "Number of DEGs", fill = "Direction") +
  theme_minimal(base_size = 12) +
  theme(text = element_text(family = "Liberation Sans"),
        plot.title = element_text(face = "bold", size = 16),
        axis.text.x = element_text(angle = 30, hjust = 1))

ggsave("/mnt/results/fig_cross2_direction_comparison.png", p2, width = 12, height = 8, dpi = 300, bg = "white")
cat("[OK] fig_cross2_direction_comparison.png\n")

# --- Figure cross3: Dose-response ---
dose_data <- rbind(
  cross_summary[comparison == "Cs137_10cGy", .(broad_organ, dose = "10 cGy", source = "Cs-137", DEGs)],
  cross_summary[comparison == "Cs137_100cGy", .(broad_organ, dose = "100 cGy", source = "Cs-137", DEGs)],
  cross_summary[comparison == "GCR40", .(broad_organ, dose = "40 cGy", source = "GCR", DEGs)],
  cross_summary[comparison == "GCR80", .(broad_organ, dose = "80 cGy", source = "GCR", DEGs)]
)
dose_data[, broad_organ := factor(broad_organ, levels = broad_organs)]
dose_data[, dose := factor(dose, levels = c("10 cGy", "40 cGy", "80 cGy", "100 cGy"))]

p3 <- ggplot(dose_data, aes(x = dose, y = DEGs, group = broad_organ, color = broad_organ)) +
  geom_line(linewidth = 1.2) +
  geom_point(size = 3) +
  facet_wrap(~ source, scales = "free_x") +
  scale_color_manual(values = ORGAN_COLORS) +
  labs(title = "Dose-Response: Tissue-Specific DEGs by Organ",
       x = "Dose", y = "Number of DEGs", color = "Broad Organ") +
  theme_minimal(base_size = 14) +
  theme(text = element_text(family = "Liberation Sans"),
        plot.title = element_text(face = "bold", size = 16))

ggsave("/mnt/results/fig_cross3_dose_response.png", p3, width = 10, height = 6, dpi = 300, bg = "white")
cat("[OK] fig_cross3_dose_response.png\n")

# --- Figure cross4: Radiation type comparison ---
rad_type_data <- rbind(
  cross_summary[comparison == "radiation_effect", .(broad_organ, condition = "Co-60 (100 Gy)", DEGs, deg_rate)],
  cross_summary[comparison == "Cs137_100cGy", .(broad_organ, condition = "Cs-137 (100 cGy)", DEGs, deg_rate)]
)
rad_type_data[, broad_organ := factor(broad_organ, levels = broad_organs)]

p4 <- ggplot(rad_type_data, aes(x = broad_organ, y = deg_rate, fill = condition)) +
  geom_bar(stat = "identity", position = "dodge") +
  scale_fill_manual(values = c("Co-60 (100 Gy)" = "#0072B2", "Cs-137 (100 cGy)" = "#009E73")) +
  labs(title = "Radiation Type Comparison: Co-60 vs Cs-137",
       subtitle = "DEG rate (%) by organ - Co-60 produces 3.1x more tissue-specific DEGs (CMH p=1.1e-39)",
       x = "Broad Organ", y = "DEG Rate (%)", fill = "Radiation") +
  theme_minimal(base_size = 14) +
  theme(text = element_text(family = "Liberation Sans"),
        plot.title = element_text(face = "bold", size = 16),
        plot.subtitle = element_text(size = 11, color = "grey40"),
        axis.text.x = element_text(angle = 30, hjust = 1))

ggsave("/mnt/results/fig_cross4_radiation_type_comparison.png", p4, width = 10, height = 6, dpi = 300, bg = "white")
cat("[OK] fig_cross4_radiation_type_comparison.png\n")

cat("\n=== All 4 cross-comparison figures complete ===\n")

# Verify all files
cross_figs <- list.files("/mnt/results", pattern = "^fig_cross.*\\.png$", full.names = FALSE)
cat("\nCross-comparison figures:\n")
for (f in cross_figs) {
  cat(sprintf("  %s (%d bytes)\n", f, file.size(file.path("/mnt/results", f))))
}


[OK] fig_cross2_direction_comparison.png
[OK] fig_cross3_dose_response.png
[OK] fig_cross4_radiation_type_comparison.png

=== All 4 cross-comparison figures complete ===

Cross-comparison figures:
  fig_cross1_tissue_response_heatmap.png (200854 bytes)
  fig_cross2_direction_comparison.png (212957 bytes)
  fig_cross3_dose_response.png (163365 bytes)
  fig_cross4_radiation_type_comparison.png (151803 bytes)


Compare old normalized vs new unnormalized DEG results

In [123]:

# ============================================================
# Step 10: Old vs new comparison (OSD498_510 only)
# Old: normalized data, padj<0.05, no LFC cutoff → 6,942 DEGs
# New: unnormalized data, padj<0.05 & |log2FC|>=1 → 822 DEGs
# ============================================================

# Load old file
old_dt <- fread("/mnt/user-uploads/DEG_OSD498_510_radiation_effect.csv")
# Old file has \r in flag column, 8 columns, first unnamed = gene_id, last = flag
colnames(old_dt)[1] <- "gene_id"
colnames(old_dt)[ncol(old_dt)] <- "deg_flag_old"
# Clean the flag column
old_dt[, deg_flag_old := gsub("\r", "", deg_flag_old)]
old_dt[, deg_flag_old := trimws(deg_flag_old)]

# Load new file (already in deg_annotated_list)
new_dt <- deg_annotated_list[["radiation_effect"]]

# Merge by gene_id
comparison_dt <- merge(
  old_dt[, .(gene_id, baseMean_old = baseMean, log2FC_old = log2FoldChange, padj_old = padj, deg_flag_old)],
  new_dt[, .(gene_id, baseMean_new = baseMean, log2FC_new = log2FoldChange, padj_new = padj, deg_flag_new = deg_flag)],
  by = "gene_id", all = TRUE
)

# Classify changes
comparison_dt[, status := fifelse(
  deg_flag_old == "yes" & deg_flag_new == "yes", "both_DEG",
  fifelse(deg_flag_old == "yes" & deg_flag_new == "no", "lost_DEG",
  fifelse(deg_flag_old == "no" & deg_flag_new == "yes", "gained_DEG",
  "both_non_DEG"))
)]

# Direction change for genes that were DEG in both
comparison_dt[status == "both_DEG", direction_change := fifelse(
  sign(log2FC_old) == sign(log2FC_new), "same_direction", "flipped_direction"
)]

# Summary stats
cat("=== Old vs New Comparison (OSD498_510) ===\n\n")

cat("--- DEG Status Changes ---\n")
status_counts <- comparison_dt[, .N, by = status]
print(status_counts)

cat("\n--- Key Metrics ---\n")
cat(sprintf("Old DEGs (normalized, padj<0.05, no LFC cutoff): %d\n", 
            comparison_dt[deg_flag_old == "yes", .N]))
cat(sprintf("New DEGs (unnormalized, padj<0.05 & |LFC|>=1): %d\n",
            comparison_dt[deg_flag_new == "yes", .N]))
cat(sprintf("DEGs in both: %d\n", comparison_dt[status == "both_DEG", .N]))
cat(sprintf("DEGs lost (old=yes, new=no): %d\n", comparison_dt[status == "lost_DEG", .N]))
cat(sprintf("DEGs gained (old=no, new=yes): %d\n", comparison_dt[status == "gained_DEG", .N]))
cat(sprintf("Direction flips (among shared DEGs): %d\n",
            comparison_dt[status == "both_DEG" & direction_change == "flipped_direction", .N]))

# Why were DEGs lost? Check their |log2FC|
lost_degs <- comparison_dt[status == "lost_DEG"]
cat("\n--- Why DEGs were lost (old=yes, new=no) ---\n")
cat(sprintf("Lost DEGs with |log2FC_old| < 1 (below new threshold): %d (%.1f%%)\n",
            sum(abs(lost_degs$log2FC_old) < 1, na.rm = TRUE),
            sum(abs(lost_degs$log2FC_old) < 1, na.rm = TRUE) / nrow(lost_degs) * 100))
cat(sprintf("Lost DEGs with |log2FC_old| >= 1 but padj_new >= 0.05: %d\n",
            sum(abs(lost_degs$log2FC_old) >= 1 & lost_degs$padj_new >= 0.05, na.rm = TRUE)))
cat(sprintf("Lost DEGs with |log2FC_old| >= 1 and padj_new < 0.05 (should be DEG): %d\n",
            sum(abs(lost_degs$log2FC_old) >= 1 & lost_degs$padj_new < 0.05, na.rm = TRUE)))

cat("\n--- log2FC distribution of lost DEGs ---\n")
cat(sprintf("Median |log2FC| of lost DEGs: %.3f\n", median(abs(lost_degs$log2FC_old), na.rm = TRUE)))
cat(sprintf("Max |log2FC| of lost DEGs: %.3f\n", max(abs(lost_degs$log2FC_old), na.rm = TRUE)))
cat(sprintf("Lost DEGs with |log2FC| < 0.5: %d\n", sum(abs(lost_degs$log2FC_old) < 0.5, na.rm = TRUE)))

# baseMean comparison (normalized vs unnormalized)
cat("\n--- baseMean comparison (normalized vs unnormalized) ---\n")
cat(sprintf("Correlation of baseMean: %.4f\n", 
            cor(comparison_dt$baseMean_old, comparison_dt$baseMean_new, use = "complete.obs", method = "spearman")))
cat(sprintf("Median baseMean_old: %.1f, Median baseMean_new: %.1f\n",
            median(comparison_dt$baseMean_old, na.rm = TRUE),
            median(comparison_dt$baseMean_new, na.rm = TRUE)))

# log2FC comparison for shared DEGs
shared <- comparison_dt[status == "both_DEG"]
cat("\n--- log2FC comparison for shared DEGs ---\n")
cat(sprintf("Correlation of log2FC: %.4f\n", 
            cor(shared$log2FC_old, shared$log2FC_new, use = "complete.obs")))
cat(sprintf("Median |log2FC_old|: %.3f, Median |log2FC_new|: %.3f\n",
            median(abs(shared$log2FC_old), na.rm = TRUE),
            median(abs(shared$log2FC_new), na.rm = TRUE)))

# Tissue assignment shifts
cat("\n--- Tissue assignment shifts (shared DEGs) ---\n")
# Get tissue annotations for shared DEGs
shared_ann <- merge(
  shared[, .(gene_id, status, direction_change)],
  tissue_ann[, .(gene_id, predominant_broad_organ, predominant_subtissue, tau_subtissue)],
  by = "gene_id", all.x = TRUE
)
cat("Broad organ distribution of shared DEGs:\n")
print(shared_ann[, .N, by = predominant_broad_organ][order(-N)])

# Save comparison table
fwrite(comparison_dt, "/mnt/results/old_vs_new_comparison_OSD498_510.csv")
cat("\nSaved old_vs_new_comparison_OSD498_510.csv\n")

# Save summary
summary_dt <- data.table(
  metric = c("Old DEGs (normalized)", "New DEGs (unnormalized)", "Shared DEGs", 
             "Lost DEGs", "Gained DEGs", "Direction flips", 
             "Lost due to LFC threshold", "Lost due to padj change",
             "baseMean correlation (Spearman)", "log2FC correlation (shared)"),
  value = c(
    comparison_dt[deg_flag_old == "yes", .N],
    comparison_dt[deg_flag_new == "yes", .N],
    comparison_dt[status == "both_DEG", .N],
    comparison_dt[status == "lost_DEG", .N],
    comparison_dt[status == "gained_DEG", .N],
    comparison_dt[status == "both_DEG" & direction_change == "flipped_direction", .N],
    sum(abs(lost_degs$log2FC_old) < 1, na.rm = TRUE),
    sum(abs(lost_degs$log2FC_old) >= 1 & lost_degs$padj_new >= 0.05, na.rm = TRUE),
    round(cor(comparison_dt$baseMean_old, comparison_dt$baseMean_new, use = "complete.obs", method = "spearman"), 4),
    round(cor(shared$log2FC_old, shared$log2FC_new, use = "complete.obs"), 4)
  )
)
fwrite(summary_dt, "/mnt/results/old_vs_new_summary_OSD498_510.csv")
cat("Saved old_vs_new_summary_OSD498_510.csv\n")


=== Old vs New Comparison (OSD498_510) ===

--- DEG Status Changes ---
         status     N
         <char> <int>
1:     lost_DEG  6155
2: both_non_DEG 16536
3:     both_DEG   787
4:         <NA>  2120
5:   gained_DEG     8

--- Key Metrics ---
Old DEGs (normalized, padj<0.05, no LFC cutoff): 6942
New DEGs (unnormalized, padj<0.05 & |LFC|>=1): 822
DEGs in both: 787
DEGs lost (old=yes, new=no): 6155
DEGs gained (old=no, new=yes): 8
Direction flips (among shared DEGs): 0

--- Why DEGs were lost (old=yes, new=no) ---
Lost DEGs with |log2FC_old| < 1 (below new threshold): 6141 (99.8%)
Lost DEGs with |log2FC_old| >= 1 but padj_new >= 0.05: 14
Lost DEGs with |log2FC_old| >= 1 and padj_new < 0.05 (should be DEG): 0

--- log2FC distribution of lost DEGs ---
Median |log2FC| of lost DEGs: 0.252
Max |log2FC| of lost DEGs: 2.186
Lost DEGs with |log2FC| < 0.5: 5052

--- baseMean comparison (normalized vs unnormalized) ---
Correlation of baseMean: 1.0000
Median baseMean_old: 162.7, Median baseMean_

Generate instruction manual PDF with Phylo branding

In [125]:

# ============================================================
# Step 12: Instruction Manual PDF (all 6 comparisons)
# Mixed-level class structure with Phylo branding
# ============================================================

from reportlab.lib.pagesizes import letter
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
                                 PageBreak, HRFlowable, Image, KeepTogether)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.colors import HexColor
from reportlab.lib.enums import TA_JUSTIFY, TA_CENTER, TA_LEFT
from reportlab.lib.units import inch
import os

# Phylo brand colors
PHYLO_GOLD = HexColor("#D4A04A")
HEADING_COLOR = HexColor("#111111")
BODY_TEXT = HexColor("#2C2A26")
MUTED_TEXT = HexColor("#8A8378")
TABLE_HEADER_BG = PHYLO_GOLD
TABLE_HEADER_FG = HexColor("#FFFFFF")
TABLE_ALT_ROW = HexColor("#F9F7F3")
TABLE_BORDER = HexColor("#D5CFC5")
CALLOUT_BG = HexColor("#FAF9F3")

FONT_HEADING = "Helvetica-Bold"
FONT_BODY = "Helvetica"
FONT_ITALIC = "Helvetica-Oblique"
FONT_MONO = "Courier"

# Styles
styles = getSampleStyleSheet()
styles.add(ParagraphStyle(name="ReportTitle", fontName=FONT_HEADING, fontSize=24,
    textColor=HEADING_COLOR, spaceBefore=0, spaceAfter=6, leading=30))
styles.add(ParagraphStyle(name="Subtitle", fontName=FONT_BODY, fontSize=12,
    textColor=PHYLO_GOLD, spaceAfter=4))
styles.add(ParagraphStyle(name="Attribution", fontName=FONT_ITALIC, fontSize=10,
    textColor=MUTED_TEXT, spaceAfter=8))
styles.add(ParagraphStyle(name="SectionHead", fontName=FONT_HEADING, fontSize=16,
    textColor=HEADING_COLOR, spaceBefore=20, spaceAfter=8, leading=20))
styles.add(ParagraphStyle(name="SubHead", fontName=FONT_HEADING, fontSize=13,
    textColor=HEADING_COLOR, spaceBefore=14, spaceAfter=6))
styles.add(ParagraphStyle(name="Body", fontName=FONT_BODY, fontSize=10.5,
    textColor=BODY_TEXT, alignment=TA_JUSTIFY, spaceAfter=8, leading=15))
styles.add(ParagraphStyle(name="Caption", fontName=FONT_ITALIC, fontSize=9,
    textColor=MUTED_TEXT, alignment=TA_CENTER, spaceAfter=12))
styles.add(ParagraphStyle(name="CalloutText", fontName=FONT_BODY, fontSize=10,
    textColor=BODY_TEXT, alignment=TA_LEFT, spaceAfter=4, leading=14))
styles.add(ParagraphStyle(name="CalloutTitle", fontName=FONT_HEADING, fontSize=10.5,
    textColor=PHYLO_GOLD, spaceAfter=4))
styles.add(ParagraphStyle(name="CodeStyle", fontName=FONT_MONO, fontSize=8.5,
    textColor=BODY_TEXT, alignment=TA_LEFT, spaceAfter=4, leading=12,
    leftIndent=12, rightIndent=12))
styles.add(ParagraphStyle(name="AdvNote", fontName=FONT_ITALIC, fontSize=9,
    textColor=MUTED_TEXT, alignment=TA_LEFT, spaceAfter=8, leading=13,
    leftIndent=12))

def page_header_footer(canvas, doc):
    canvas.saveState()
    w, h = letter
    canvas.setFont("Helvetica", 9)
    canvas.setFillColor(MUTED_TEXT)
    canvas.drawString(60, h - 40, "Tissue-Specific DEG Analysis v2 - Instruction Manual")
    canvas.setStrokeColor(PHYLO_GOLD)
    canvas.setLineWidth(1)
    canvas.line(60, h - 48, w - 60, h - 48)
    canvas.setStrokeColor(TABLE_BORDER)
    canvas.setLineWidth(0.75)
    canvas.line(60, 40, w - 60, 40)
    canvas.setFont("Helvetica", 8)
    canvas.setFillColor(MUTED_TEXT)
    canvas.drawCentredString(w / 2, 26, f"Page {doc.page}")
    canvas.restoreState()

def callout(title, text):
    data = [[Paragraph(f"<b>{title}</b>", styles["CalloutTitle"])],
            [Paragraph(text, styles["CalloutText"])]]
    t = Table(data, colWidths=[440])
    t.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, -1), CALLOUT_BG),
        ("BOX", (0, 0), (-1, -1), 0.5, TABLE_BORDER),
        ("LINEBEFORE", (0, 0), (0, -1), 3, PHYLO_GOLD),
        ("TOPPADDING", (0, 0), (-1, -1), 10),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 10),
        ("LEFTPADDING", (0, 0), (-1, -1), 14),
        ("RIGHTPADDING", (0, 0), (-1, -1), 14),
    ]))
    return t

def divider():
    return HRFlowable(width=480, thickness=1, color=PHYLO_GOLD, spaceAfter=10, spaceBefore=4)

def make_table(headers, rows, col_widths=None):
    header_style = ParagraphStyle(name="hdr", fontName=FONT_HEADING, fontSize=9,
        textColor=TABLE_HEADER_FG, alignment=TA_CENTER)
    cell_style = ParagraphStyle(name="cell", fontName=FONT_BODY, fontSize=9,
        textColor=BODY_TEXT, alignment=TA_LEFT)
    data = [[Paragraph(f'<b>{h}</b>', header_style) for h in headers]]
    for row in rows:
        data.append([Paragraph(str(c), cell_style) for c in row])
    if col_widths is None:
        col_widths = [490/len(headers)] * len(headers)
    t = Table(data, colWidths=col_widths, repeatRows=1)
    t.hAlign = "CENTER"
    t.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), TABLE_HEADER_BG),
        ("TEXTCOLOR", (0, 0), (-1, 0), TABLE_HEADER_FG),
        *[("BACKGROUND", (0, i), (-1, i), TABLE_ALT_ROW) for i in range(2, len(data), 2)],
        ("GRID", (0, 0), (-1, -1), 0.5, TABLE_BORDER),
        ("BOX", (0, 0), (-1, -1), 0.75, TABLE_BORDER),
        ("TOPPADDING", (0, 0), (-1, -1), 5),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
        ("LEFTPADDING", (0, 0), (-1, -1), 6),
        ("RIGHTPADDING", (0, 0), (-1, -1), 6),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
    ]))
    return t

# Build the PDF
output_path = "/mnt/results/instruction_manual_tissue_specific_deg_v2.pdf"
doc = SimpleDocTemplate(output_path, pagesize=letter,
    topMargin=52, bottomMargin=52, leftMargin=60, rightMargin=60)

story = []

# --- Title Page ---
story.append(Spacer(1, 40))
story.append(Paragraph("Tissue-Specific DEG Analysis", styles["ReportTitle"]))
story.append(Paragraph("Instruction Manual v2 - Corrected Unnormalized DESeq2 Data", styles["Subtitle"]))
story.append(Spacer(1, 8))
story.append(Paragraph("<i>Generated by Biomni  |  2026-07-10</i>", styles["Attribution"]))
story.append(Spacer(1, 20))

story.append(callout("For Beginners",
    "This manual explains how to analyze tissue-specific gene expression changes in "
    "Arabidopsis plants exposed to different types of radiation. We use 6 separate "
    "experiments from NASA GeneLab, covering gamma rays (Co-60, Cs-137), simulated "
    "cosmic rays (GCR), and a DNA repair mutant (sog1-1). Each section has callout "
    "boxes like this one for key concepts."))

story.append(Spacer(1, 12))
story.append(callout("For Advanced Readers",
    "This version uses raw unnormalized counts as DESeq2 input (corrected from the "
    "previous version which used normalized data). DEG threshold: padj < 0.05 AND "
    "|log2FC| >= 1. Tissue assignment via AtGenExpress atlas (Schmid et al. 2005) "
    "with Tau tissue-specificity index (threshold 0.6). Cross-comparison tests use "
    "Cochran-Mantel-Haenszel and chi-square frameworks."))

story.append(PageBreak())

# --- Section 1: Introduction ---
story.append(Paragraph("1. Introduction: Arabidopsis Radiation Response", styles["SectionHead"]))
story.append(divider())

story.append(Paragraph(
    "Plants grown in space are exposed to ionizing radiation that differs from "
    "Earth's background. NASA GeneLab has deposited multiple Arabidopsis RNA-seq "
    "studies examining how plants respond to different radiation types. This analysis "
    "covers 6 experimental comparisons from 5 OSD studies (OSD-498, 508, 510, 658, 782), "
    "encompassing 158 samples across 3 radiation sources, 4 dose levels, and 3 genotypes.", styles["Body"]))

story.append(Paragraph("The 6 Experimental Comparisons", styles["SubHead"]))

comp_rows = [
    ["1", "radiation_effect", "OSD498+510", "Gamma Co-60 100Gy vs none (WT)", "~ Study + Genotype + Radiation"],
    ["2", "genotype_interaction", "OSD508+510", "sog1-1 x radiation interaction", "~ Study + Genotype + Radiation + Geno:Rad"],
    ["3", "GCR40", "OSD658", "Simulated GCR 40cGy vs none (WT)", "~ Radiation"],
    ["4", "GCR80", "OSD658", "Simulated GCR 80cGy vs none (WT)", "~ Radiation"],
    ["5", "Cs137_100cGy", "OSD782", "Gamma Cs-137 100cGy vs none (WT)", "~ Timepoint + Radiation"],
    ["6", "Cs137_10cGy", "OSD782", "Gamma Cs-137 10cGy vs none (WT)", "~ Timepoint + Radiation"],
]
story.append(make_table(["#", "Label", "Study", "Contrast", "DESeq2 Design"], comp_rows,
    col_widths=[25, 90, 65, 180, 130]))
story.append(Spacer(1, 8))

story.append(callout("Key Concept: Why 6 Separate Comparisons?",
    "Each study uses different radiation sources, doses, and genotypes. Pooling them "
    "into one model would average together biologically distinct exposures. Instead, "
    "we run focused comparisons within studies that share a consistent design, "
    "treating 'Study' as a batch covariate where needed."))

story.append(Spacer(1, 8))
story.append(Paragraph("Advanced Note: Study as a batch effect", styles["AdvNote"]))
story.append(Paragraph(
    "Different sequencing runs, years, and library preparations mean 'Study' itself "
    "can explain a large chunk of gene expression variance. In pooled comparisons "
    "(1 and 2), Study is included as a blocking term in the DESeq2 design formula. "
    "Single-study comparisons (3-6) do not need this because there is no cross-study "
    "batch to account for.", styles["AdvNote"]))

story.append(PageBreak())

# --- Section 2: DESeq2 Background ---
story.append(Paragraph("2. DESeq2 and the Raw vs Normalized Counts Issue", styles["SectionHead"]))
story.append(divider())

story.append(Paragraph(
    "DESeq2 is the standard tool for differential expression analysis of RNA-seq "
    "count data. It models read counts with a negative binomial distribution and "
    "performs its own internal normalization (median-of-ratios size factors). "
    "This means DESeq2 expects raw, unnormalized integer counts as input.", styles["Body"]))

story.append(callout("Methodological Lesson: Raw vs Normalized",
    "The first version of this analysis used GeneLab's normalized counts table as "
    "DESeq2 input. This is methodologically incorrect because DESeq2's dispersion "
    "estimation and statistical testing assume raw count data. The corrected version "
    "uses raw unnormalized counts (RSEM/STAR output), which is the statistically "
    "proper input. The impact: 6,942 DEGs (old) vs 822 DEGs (new) for the same "
    "comparison, with 99.8% of lost DEGs having |log2FC| < 1 - they were called "
    "significant only because the old threshold had no fold-change cutoff."))

story.append(Paragraph("DEG Threshold", styles["SubHead"]))
story.append(Paragraph(
    "We define differentially expressed genes (DEGs) as those with adjusted p-value "
    "(padj) < 0.05 AND absolute log2 fold change >= 1 (at least 2-fold change). "
    "This dual threshold ensures both statistical significance and biological "
    "meaningfulness - a gene can have a tiny p-value but a negligible fold change "
    "that is unlikely to matter biologically.", styles["Body"]))

story.append(Paragraph("Advanced Note: Why |log2FC| >= 1?", styles["AdvNote"]))
story.append(Paragraph(
    "The old analysis used padj < 0.05 with no fold-change cutoff, calling genes "
    "with |log2FC| as low as 0.057 as DEGs. While statistically significant, a 4% "
    "change in expression is rarely biologically meaningful. The 2-fold threshold "
    "(|log2FC| >= 1) is the standard convention in DESeq2 workflows and filters "
    "to genes with substantial expression changes.", styles["AdvNote"]))

story.append(PageBreak())

# --- Section 3: DEG Overview ---
story.append(Paragraph("3. DEG Overview: All 6 Comparisons", styles["SectionHead"]))
story.append(divider())

deg_rows = [
    ["radiation_effect", "25,519", "822", "764", "58", "3.2%"],
    ["genotype_interaction", "26,072", "433", "76", "357", "1.7%"],
    ["GCR40", "22,178", "2", "1", "1", "0.0%"],
    ["GCR80", "22,178", "11", "11", "0", "0.1%"],
    ["Cs137_100cGy", "23,667", "229", "220", "9", "1.0%"],
    ["Cs137_10cGy", "23,667", "87", "79", "8", "0.4%"],
]
story.append(make_table(
    ["Comparison", "Total Genes", "DEGs", "Up", "Down", "% DEG"],
    deg_rows, col_widths=[110, 70, 50, 50, 50, 50]))
story.append(Spacer(1, 8))

story.append(callout("Observation: Radiation Type Matters",
    "Co-60 gamma radiation at 100 Gy produces the most DEGs (822), while simulated "
    "GCR at 40-80 cGy produces almost none (2-11). This is expected: 100 Gy is "
    "1,000-2,500x higher dose than 40-80 cGy. The Cs-137 comparisons show a clear "
    "dose-response (87 at 10 cGy vs 229 at 100 cGy). The sog1-1 interaction DEGs "
    "are predominantly downregulated (357/433 = 82%), opposite to the radiation "
    "main effect (764/822 = 93% upregulated)."))

story.append(PageBreak())

# --- Section 4: Tissue-Specificity Concept ---
story.append(Paragraph("4. Tissue-Specificity: The Tau Index", styles["SectionHead"]))
story.append(divider())

story.append(Paragraph(
    "Since all 6 experiments used whole seedlings, the DEG data has no tissue "
    "information. To determine which tissues are responding to radiation, we use "
    "the AtGenExpress developmental expression atlas (Schmid et al. 2005), which "
    "profiled 237 samples across 33 sub-tissues of Arabidopsis.", styles["Body"]))

story.append(Paragraph("The Tau Tissue-Specificity Index", styles["SubHead"]))
story.append(Paragraph(
    "Tau (tau) measures how specifically a gene is expressed in one tissue versus "
    "others. It ranges from 0 (expressed equally everywhere) to 1 (expressed in "
    "only one tissue). We use the threshold tau >= 0.6 to classify genes as "
    "tissue-specific.", styles["Body"]))

story.append(Paragraph(
    "Formula: tau = sum(1 - x_i / max(x)) / (n - 1)", styles["CodeStyle"]))

story.append(callout("For Beginners: What is Tau?",
    "Imagine a gene that is only turned on in roots. Its expression in roots is "
    "high, but in leaves, flowers, and seeds it is near zero. This gene would have "
    "a Tau value close to 1 (very tissue-specific). A gene that is equally expressed "
    "everywhere (like a housekeeping gene) would have Tau close to 0. We use 0.6 "
    "as the cutoff: genes above 0.6 are 'tissue-specific'."))

story.append(Paragraph("Advanced Note: Tau vs other specificity metrics", styles["AdvNote"]))
story.append(Paragraph(
    "Tau is preferred over alternatives (Gini coefficient, entropy-based measures) "
    "because it is robust to noise, bounded [0,1], and does not require arbitrary "
    "expression thresholds. The 0.6 cutoff is consistent with Yanai et al. (2005) "
    "and Kryuchkova-Mostacci & Robinson-Rechavi (2017). In our atlas: 12,772 "
    "tissue-specific genes, 8,061 constitutive, across 5 broad organs and 33 "
    "sub-tissues.", styles["AdvNote"]))

story.append(PageBreak())

# --- Section 5: Code Walkthrough ---
story.append(Paragraph("5. Code Walkthrough: Key Steps", styles["SectionHead"]))
story.append(divider())

story.append(Paragraph("Step 1: Load and threshold DESeq2 results", styles["SubHead"]))
story.append(Paragraph(
    'dt[, deg_flag := ifelse(padj &lt; 0.05 &amp; abs(log2FoldChange) &gt;= 1, "yes", "no")]', styles["CodeStyle"]))
story.append(Paragraph(
    "This applies the dual threshold to each of the 6 DESeq2 output tables. "
    "Genes passing both criteria are flagged as DEGs.", styles["Body"]))

story.append(Paragraph("Step 2: Compute Tau from the atlas", styles["SubHead"]))
story.append(Paragraph(
    'tau &lt;- apply(expr_matrix, 1, function(x) sum(1 - x/max(x)) / (n-1))', styles["CodeStyle"]))
story.append(Paragraph(
    "For each gene, Tau is computed across all 33 sub-tissues. Genes with "
    "Tau >= 0.6 are classified as tissue-specific and assigned to their "
    "predominant tissue (highest expression).", styles["Body"]))

story.append(Paragraph("Step 3: Merge DEGs with tissue annotations", styles["SubHead"]))
story.append(Paragraph(
    'dt_ann &lt;- merge(deg_table, tissue_ann, by = "gene_id")', styles["CodeStyle"]))
story.append(Paragraph(
    "Each DEG is classified as tissue-specific (assigned to a broad organ + "
    "sub-tissue), constitutive (expressed everywhere), or unannotated (not in "
    "the atlas, including organellar genes).", styles["Body"]))

story.append(Paragraph("Step 4: Statistical tests", styles["SubHead"]))
story.append(Paragraph(
    "Four tests per comparison: (1) chi-square of independence (organ x DEG status), "
    "(2) per-tissue Fisher's exact (organ vs rest), (3) pairwise Fisher's exact "
    "(10 organ pairs), (4) direction chi-square (organ x up/down). All with "
    "Bonferroni correction.", styles["Body"]))

story.append(Paragraph("Step 5: Cross-comparison tests", styles["SubHead"]))
story.append(Paragraph(
    "Cochran-Mantel-Haenszel (CMH) test stratified by organ: compares DEG rates "
    "between radiation types (Co-60 vs Cs-137) and doses (10 vs 100 cGy). "
    "Chi-square tests compare organ distributions and direction patterns across "
    "comparisons.", styles["Body"]))

story.append(PageBreak())

# --- Section 6: Results ---
story.append(Paragraph("6. Key Results", styles["SectionHead"]))
story.append(divider())

story.append(Paragraph("Per-Comparison Tissue Enrichment", styles["SubHead"]))

result_rows = [
    ["radiation_effect", "233.6", "2.2e-49", "Seedling (12.7%)", "Root, Flower, Seed"],
    ["genotype_interaction", "14.4", "0.006", "Flower (3.1%)", "Leaf_Shoot"],
    ["GCR40", "SKIP", "-", "Too few DEGs (2)", "-"],
    ["GCR80", "SKIP", "-", "Too few DEGs (4)", "-"],
    ["Cs137_100cGy", "53.9", "5.7e-11", "Seedling (3.0%), Root (2.4%)", "Flower, Leaf_Shoot"],
    ["Cs137_10cGy", "44.4", "5.3e-9", "Seedling (1.6%), Root (0.9%)", "Flower"],
]
story.append(make_table(
    ["Comparison", "Chi-sq", "p-value", "Enriched", "Depleted"],
    result_rows, col_widths=[110, 55, 60, 130, 100]))
story.append(Spacer(1, 10))

story.append(Paragraph("Cross-Comparison Tests", styles["SubHead"]))

cross_rows = [
    ["CMH: Co-60 vs Cs-137", "173.7", "1.1e-39", "Co-60 = 3.1x more DEGs"],
    ["CMH: Cs-137 10 vs 100cGy", "52.0", "5.6e-13", "100cGy = 2.8x more DEGs"],
    ["Chi-sq: Genotype organ dist.", "87.1", "5.5e-18", "Different tissue profiles"],
    ["Chi-sq: Direction (rad vs int.)", "456.5", "2.8e-101", "Up-dominant vs down-dominant"],
    ["Chi-sq: Overall (4 comparisons)", "209.2", "4.0e-38", "Profiles differ across conditions"],
]
story.append(make_table(
    ["Test", "Statistic", "p-value", "Interpretation"],
    cross_rows, col_widths=[130, 60, 65, 200]))
story.append(Spacer(1, 10))

story.append(callout("Key Finding: sog1-1 Dampens the Radiation Response",
    "The genotype x radiation interaction DEGs are 82% downregulated, while the "
    "radiation main effect is 93% upregulated. This means sog1-1 (a DNA damage "
    "response mutant) reduces the plant's ability to upregulate radiation-responsive "
    "genes. The tissue distribution also differs significantly (chi-sq p=5.5e-18), "
    "with Flower and Seed_Silique showing the most interaction DEGs."))

story.append(PageBreak())

# --- Section 7: Old vs New Comparison ---
story.append(Paragraph("7. Methodological Lesson: Old vs New Comparison", styles["SectionHead"]))
story.append(divider())

story.append(Paragraph(
    "For the OSD498_510 radiation_effect comparison, we can directly compare the "
    "old (normalized data, no LFC cutoff) and new (unnormalized data, |LFC| >= 1) "
    "results to quantify the impact of both corrections.", styles["Body"]))

oldnew_rows = [
    ["Old DEGs (normalized, padj<0.05)", "6,942"],
    ["New DEGs (unnormalized, padj<0.05 & |LFC|>=1)", "822"],
    ["Shared DEGs (in both)", "787"],
    ["Lost DEGs (old=yes, new=no)", "6,155"],
    ["Gained DEGs (old=no, new=yes)", "8"],
    ["Direction flips (among shared)", "0"],
    ["Lost due to LFC threshold (|log2FC_old| < 1)", "6,141 (99.8%)"],
    ["Lost due to normalization change", "14"],
    ["log2FC correlation (shared DEGs)", "0.95"],
    ["baseMean correlation (Spearman)", "1.00"],
]
story.append(make_table(["Metric", "Value"], oldnew_rows, col_widths=[300, 150]))
story.append(Spacer(1, 10))

story.append(callout("What This Tells Us",
    "The 88% reduction in DEGs is almost entirely from the stricter fold-change "
    "threshold, not from the normalization correction. Only 14 genes were lost "
    "specifically due to switching from normalized to unnormalized counts. The "
    "787 shared DEGs have a log2FC correlation of 0.95 and zero direction flips, "
    "confirming that the biological signal is consistent - the corrections made "
    "the results more stringent, not different."))

story.append(PageBreak())

# --- Section 8: Outputs Guide ---
story.append(Paragraph("8. Output Files Guide", styles["SectionHead"]))
story.append(divider())

story.append(Paragraph("Per-Comparison Outputs", styles["SubHead"]))
output_rows = [
    ["tissue_specific_degs_v2/<label>/", "Hierarchical CSV folders (5 organs + sub-tissues)"],
    ["tissue_deg_summary_<label>.csv", "Hierarchical summary: organ x sub-tissue DEG counts"],
    ["tissue_enrichment_stats_<label>.csv", "Fisher's exact test results per organ"],
    ["pairwise_comparisons_<label>.csv", "Pairwise Fisher's tests (10 organ pairs)"],
    ["fig1-5_*_<label>.png", "5 figures per comparison (bar, volcano, heatmap, UpSet, forest)"],
]
story.append(make_table(["File Pattern", "Description"], output_rows, col_widths=[200, 280]))
story.append(Spacer(1, 8))

story.append(Paragraph("Cross-Comparison Outputs", styles["SubHead"]))
cross_output_rows = [
    ["cross_comparison_tissue_summary.csv", "DEG counts: organ x comparison matrix"],
    ["cross_comparison_direction.csv", "DEG rates: organ x comparison matrix"],
    ["cross_comparison_statistics.csv", "All Fisher's exact results across 6 comparisons"],
    ["cross_comparison_statistical_tests.csv", "CMH and chi-square cross-comparison tests"],
    ["fig_cross1-4_*.png", "4 cross-comparison figures"],
    ["old_vs_new_comparison_OSD498_510.csv", "Gene-level old vs new comparison"],
]
story.append(make_table(["File", "Description"], cross_output_rows, col_widths=[220, 260]))
story.append(Spacer(1, 8))

story.append(Paragraph("R Script", styles["SubHead"]))
story.append(Paragraph(
    "tissue_specific_deg_analysis_v2.R - Complete parameterized R script covering "
    "all 12 sections of the analysis. Can be re-run with different thresholds by "
    "changing the configuration variables at the top.", styles["Body"]))

story.append(PageBreak())

# --- Section 9: Interpretation Guide ---
story.append(Paragraph("9. How to Interpret the Results", styles["SectionHead"]))
story.append(divider())

story.append(Paragraph("Reading the Tissue Enrichment Forest Plot", styles["SubHead"]))
story.append(Paragraph(
    "Each forest plot (fig5) shows the DEG rate (%) for each organ with 95% "
    "confidence intervals. Organs to the right of the dashed line (overall mean) "
    "have higher DEG rates. Significance markers: *** (p<0.001), ** (p<0.01), "
    "* (p<0.05), ns (not significant), after Bonferroni correction.", styles["Body"]))

story.append(Paragraph("Reading the Cross-Comparison Heatmap", styles["SubHead"]))
story.append(Paragraph(
    "The tissue response heatmap (fig_cross1) shows DEG rates for each organ "
    "across all 6 comparisons. Darker colors = more DEGs. This reveals which "
    "tissues respond most to each radiation condition. Seedling shows the highest "
    "rate (11.96%) under Co-60 100Gy, while GCR conditions produce almost no "
    "tissue-specific DEGs.", styles["Body"]))

story.append(Paragraph("Reading the Dose-Response Plot", styles["SubHead"]))
story.append(Paragraph(
    "The dose-response figure (fig_cross3) plots DEG count vs dose for each organ, "
    "separated by radiation source (Cs-137 vs GCR). For Cs-137, all organs show "
    "a clear increase from 10 to 100 cGy. For GCR, the increase from 40 to 80 cGy "
    "is minimal, reflecting the very low DEG counts at these doses.", styles["Body"]))

story.append(callout("Caveat: Low-DEG Comparisons",
    "GCR40 (2 DEGs) and GCR80 (11 DEGs) have insufficient DEGs for meaningful "
    "tissue enrichment statistics. Chi-square and direction tests were skipped "
    "for these comparisons. This is a real biological signal (low-dose mixed-"
    "particle radiation produces minimal 2-fold transcriptional changes), not "
    "a data quality issue. Consider lowering the LFC threshold for these "
    "comparisons if exploratory analysis is desired."))

story.append(PageBreak())

# --- Section 10: Exercises ---
story.append(Paragraph("10. Exercises", styles["SectionHead"]))
story.append(divider())

story.append(Paragraph("Beginner Exercises", styles["SubHead"]))
story.append(Paragraph(
    "1. Open the tissue_deg_summary_radiation_effect.csv file. Which sub-tissue "
    "has the most DEGs? Is it up- or down-regulated?<br/><br/>"
    "2. Look at fig1 for Cs137_100cGy. Which organ has the most DEGs? Compare "
    "this to fig1 for radiation_effect - how does the organ distribution differ "
    "between Co-60 and Cs-137 radiation?<br/><br/>"
    "3. In the old_vs_new comparison, what percentage of old DEGs were lost "
    "because their fold change was below the new threshold?", styles["Body"]))

story.append(Paragraph("Intermediate Exercises", styles["SubHead"]))
story.append(Paragraph(
    "4. The CMH test for Co-60 vs Cs-137 gives OR=0.328. Interpret this odds "
    "ratio: what does it mean biologically?<br/><br/>"
    "5. The genotype_interaction DEGs are 82% downregulated while radiation_effect "
    "is 93% upregulated. What does this tell you about the role of SOG1 in the "
    "radiation response?<br/><br/>"
    "6. Compare the tissue enrichment results for Cs137_10cGy and Cs137_100cGy. "
    "Does the enriched tissue change with dose, or just the magnitude?", styles["Body"]))

story.append(Paragraph("Advanced Exercises", styles["SubHead"]))
story.append(Paragraph(
    "7. The chi-square test for genotype organ distribution gives X2=87.1, "
    "p=5.5e-18. Examine the residuals: which organs contribute most to this "
    "significant result?<br/><br/>"
    "8. Re-run the analysis with |log2FC| >= 0.5 instead of 1.0. How many more "
    "DEGs do you get for GCR40 and GCR80? Does this change the tissue enrichment "
    "results?<br/><br/>"
    "9. The baseMean correlation between old and new is 1.0 (Spearman), but the "
    "log2FC correlation is 0.95. Why might log2FC be less correlated than baseMean "
    "when switching from normalized to unnormalized counts?", styles["Body"]))

# Build PDF
doc.build(story, onFirstPage=page_header_footer, onLaterPages=page_header_footer)

# Validate
from pypdf import PdfReader
reader = PdfReader(output_path)
page_count = len(reader.pages)
file_size = os.path.getsize(output_path)
print(f"PDF generated: {output_path}")
print(f"Pages: {page_count}, Size: {file_size:,} bytes")
assert page_count >= 5, f"Only {page_count} pages"
assert file_size > 5000, f"Only {file_size} bytes"
first_text = reader.pages[0].extract_text()
assert len(first_text.strip()) > 0, "No text on first page"
print("Validation passed.")


PDF generated: /mnt/results/instruction_manual_tissue_specific_deg_v2.pdf
Pages: 11, Size: 24,580 bytes
Validation passed.


Generate statistical walkthrough PDF with all 6 comparisons

In [127]:

# ============================================================
# Step 13: Statistical Walkthrough PDF (all 6 comparisons)
# Part I: Figures explained with basic/intermediate stats terminology
# Part II: Code blocks explained
# Part III: Statistical glossary
# ============================================================

from reportlab.lib.pagesizes import letter
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
                                 PageBreak, HRFlowable, Image, KeepTogether)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.colors import HexColor
from reportlab.lib.enums import TA_JUSTIFY, TA_CENTER, TA_LEFT
import os

# Reuse same brand colors and styles from Step 12
PHYLO_GOLD = HexColor("#D4A04A")
HEADING_COLOR = HexColor("#111111")
BODY_TEXT = HexColor("#2C2A26")
MUTED_TEXT = HexColor("#8A8378")
TABLE_HEADER_BG = PHYLO_GOLD
TABLE_HEADER_FG = HexColor("#FFFFFF")
TABLE_ALT_ROW = HexColor("#F9F7F3")
TABLE_BORDER = HexColor("#D5CFC5")
CALLOUT_BG = HexColor("#FAF9F3")

FONT_HEADING = "Helvetica-Bold"
FONT_BODY = "Helvetica"
FONT_ITALIC = "Helvetica-Oblique"
FONT_MONO = "Courier"

styles = getSampleStyleSheet()
styles.add(ParagraphStyle(name="ReportTitle", fontName=FONT_HEADING, fontSize=24,
    textColor=HEADING_COLOR, spaceBefore=0, spaceAfter=6, leading=30))
styles.add(ParagraphStyle(name="Subtitle", fontName=FONT_BODY, fontSize=12,
    textColor=PHYLO_GOLD, spaceAfter=4))
styles.add(ParagraphStyle(name="Attribution", fontName=FONT_ITALIC, fontSize=10,
    textColor=MUTED_TEXT, spaceAfter=8))
styles.add(ParagraphStyle(name="SectionHead", fontName=FONT_HEADING, fontSize=16,
    textColor=HEADING_COLOR, spaceBefore=20, spaceAfter=8, leading=20))
styles.add(ParagraphStyle(name="SubHead", fontName=FONT_HEADING, fontSize=13,
    textColor=HEADING_COLOR, spaceBefore=14, spaceAfter=6))
styles.add(ParagraphStyle(name="Body", fontName=FONT_BODY, fontSize=10.5,
    textColor=BODY_TEXT, alignment=TA_JUSTIFY, spaceAfter=8, leading=15))
styles.add(ParagraphStyle(name="Caption", fontName=FONT_ITALIC, fontSize=9,
    textColor=MUTED_TEXT, alignment=TA_CENTER, spaceAfter=12))
styles.add(ParagraphStyle(name="CalloutText", fontName=FONT_BODY, fontSize=10,
    textColor=BODY_TEXT, alignment=TA_LEFT, spaceAfter=4, leading=14))
styles.add(ParagraphStyle(name="CalloutTitle", fontName=FONT_HEADING, fontSize=10.5,
    textColor=PHYLO_GOLD, spaceAfter=4))
styles.add(ParagraphStyle(name="CodeStyle", fontName=FONT_MONO, fontSize=8.5,
    textColor=BODY_TEXT, alignment=TA_LEFT, spaceAfter=4, leading=12,
    leftIndent=12, rightIndent=12))
styles.add(ParagraphStyle(name="AdvNote", fontName=FONT_ITALIC, fontSize=9,
    textColor=MUTED_TEXT, alignment=TA_LEFT, spaceAfter=8, leading=13,
    leftIndent=12))
styles.add(ParagraphStyle(name="GlossaryTerm", fontName=FONT_HEADING, fontSize=10.5,
    textColor=HEADING_COLOR, spaceBefore=8, spaceAfter=2))
styles.add(ParagraphStyle(name="GlossaryDef", fontName=FONT_BODY, fontSize=10,
    textColor=BODY_TEXT, alignment=TA_JUSTIFY, spaceAfter=6, leading=14,
    leftIndent=12))

def page_header_footer(canvas, doc):
    canvas.saveState()
    w, h = letter
    canvas.setFont("Helvetica", 9)
    canvas.setFillColor(MUTED_TEXT)
    canvas.drawString(60, h - 40, "Statistical Walkthrough - Figures and Code v2")
    canvas.setStrokeColor(PHYLO_GOLD)
    canvas.setLineWidth(1)
    canvas.line(60, h - 48, w - 60, h - 48)
    canvas.setStrokeColor(TABLE_BORDER)
    canvas.setLineWidth(0.75)
    canvas.line(60, 40, w - 60, 40)
    canvas.setFont("Helvetica", 8)
    canvas.setFillColor(MUTED_TEXT)
    canvas.drawCentredString(w / 2, 26, f"Page {doc.page}")
    canvas.restoreState()

def callout(title, text):
    data = [[Paragraph(f"<b>{title}</b>", styles["CalloutTitle"])],
            [Paragraph(text, styles["CalloutText"])]]
    t = Table(data, colWidths=[440])
    t.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, -1), CALLOUT_BG),
        ("BOX", (0, 0), (-1, -1), 0.5, TABLE_BORDER),
        ("LINEBEFORE", (0, 0), (0, -1), 3, PHYLO_GOLD),
        ("TOPPADDING", (0, 0), (-1, -1), 10),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 10),
        ("LEFTPADDING", (0, 0), (-1, -1), 14),
        ("RIGHTPADDING", (0, 0), (-1, -1), 14),
    ]))
    return t

def divider():
    return HRFlowable(width=480, thickness=1, color=PHYLO_GOLD, spaceAfter=10, spaceBefore=4)

def make_table(headers, rows, col_widths=None):
    header_style = ParagraphStyle(name="hdr2", fontName=FONT_HEADING, fontSize=9,
        textColor=TABLE_HEADER_FG, alignment=TA_CENTER)
    cell_style = ParagraphStyle(name="cell2", fontName=FONT_BODY, fontSize=9,
        textColor=BODY_TEXT, alignment=TA_LEFT)
    data = [[Paragraph(f'<b>{h}</b>', header_style) for h in headers]]
    for row in rows:
        data.append([Paragraph(str(c), cell_style) for c in row])
    if col_widths is None:
        col_widths = [490/len(headers)] * len(headers)
    t = Table(data, colWidths=col_widths, repeatRows=1)
    t.hAlign = "CENTER"
    t.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), TABLE_HEADER_BG),
        ("TEXTCOLOR", (0, 0), (-1, 0), TABLE_HEADER_FG),
        *[("BACKGROUND", (0, i), (-1, i), TABLE_ALT_ROW) for i in range(2, len(data), 2)],
        ("GRID", (0, 0), (-1, -1), 0.5, TABLE_BORDER),
        ("BOX", (0, 0), (-1, -1), 0.75, TABLE_BORDER),
        ("TOPPADDING", (0, 0), (-1, -1), 5),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
        ("LEFTPADDING", (0, 0), (-1, -1), 6),
        ("RIGHTPADDING", (0, 0), (-1, -1), 6),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
    ]))
    return t

# Build PDF
output_path = "/mnt/results/statistical_walkthrough_figures_and_code_v2.pdf"
doc = SimpleDocTemplate(output_path, pagesize=letter,
    topMargin=52, bottomMargin=52, leftMargin=60, rightMargin=60)

story = []

# --- Title Page ---
story.append(Spacer(1, 40))
story.append(Paragraph("Statistical Walkthrough", styles["ReportTitle"]))
story.append(Paragraph("Figures and Code Explained v2 - 6 Comparisons with Corrected Data", styles["Subtitle"]))
story.append(Spacer(1, 8))
story.append(Paragraph("<i>Generated by Biomni  |  2026-07-10</i>", styles["Attribution"]))
story.append(Spacer(1, 20))

story.append(callout("Purpose",
    "This document explains the statistical concepts behind each figure and code "
    "block in the tissue-specific DEG analysis. It uses basic and intermediate "
    "statistical terminology, with callout boxes for key concepts. All 6 radiation "
    "comparisons are covered, plus the new cross-comparison analyses."))

story.append(PageBreak())

# ============================================================
# PART I: Figures Explained
# ============================================================
story.append(Paragraph("Part I: Figures Explained", styles["SectionHead"]))
story.append(divider())

# Figure 1: Stacked Bar Chart
story.append(Paragraph("Figure 1: Stacked Bar Chart - DEG Counts by Organ", styles["SubHead"]))
story.append(Paragraph(
    "<b>What it shows:</b> For each comparison, a stacked bar chart displays the "
    "number of tissue-specific DEGs in each of the 5 broad organs, colored by "
    "direction (blue = upregulated, orange = downregulated).", styles["Body"]))
story.append(Paragraph(
    "<b>Statistical concept - Frequency distribution:</b> A bar chart is a visual "
    "representation of a frequency distribution - it shows how many items (DEGs) "
    "fall into each category (organ). The height of each bar represents the count. "
    "The stacked colors show a secondary categorical variable (direction).", styles["Body"]))
story.append(callout("Basic Term: Frequency",
    "Frequency is simply how many times something occurs. If Root has 99 DEGs, "
    "the frequency for Root is 99. The bar chart makes it easy to compare "
    "frequencies across categories at a glance."))
story.append(Paragraph(
    "<b>What to look for:</b> In radiation_effect, Seedling has the tallest bar "
    "(148 DEGs, almost all upregulated). In genotype_interaction, Flower and "
    "Seed_Silique dominate, and the bars are mostly orange (downregulated). "
    "This visual difference between comparisons is quantified by the chi-square "
    "tests in Part II.", styles["Body"]))

story.append(Spacer(1, 8))

# Figure 2: Volcano Plots
story.append(Paragraph("Figure 2: Volcano Plots by Organ", styles["SubHead"]))
story.append(Paragraph(
    "<b>What it shows:</b> A 5-panel volcano plot (one per organ) for each "
    "comparison. The x-axis is log2 fold change (effect size), the y-axis is "
    "-log10(padj) (statistical significance). Each point is a gene. Colored "
    "points are DEGs (passing both thresholds); grey points are not significant.", styles["Body"]))
story.append(Paragraph(
    "<b>Statistical concept - Effect size vs significance:</b> A volcano plot "
    "visualizes the two dimensions of differential expression simultaneously. "
    "The x-axis shows the magnitude of change (effect size): genes far from "
    "center have large fold changes. The y-axis shows statistical significance: "
    "genes high on the plot have small p-values. The dashed lines mark the "
    "thresholds (|log2FC| = 1 and padj = 0.05).", styles["Body"]))
story.append(callout("Intermediate Term: Why both axes matter",
    "A gene can be statistically significant (high on y-axis) but have a tiny "
    "effect size (near center on x-axis). This happens with large sample sizes "
    "where even small differences become 'significant.' The dual threshold "
    "(both axes) ensures we only call genes that are both significant AND "
    "biologically meaningful (2-fold change)."))
story.append(Paragraph(
    "<b>What to look for:</b> In radiation_effect, Seedling and Leaf_Shoot show "
    "many points in the upper-right (upregulated DEGs with large effect sizes). "
    "In genotype_interaction, points cluster in the upper-left (downregulated). "
    "GCR comparisons have almost no colored points - very few genes pass both "
    "thresholds at these low doses.", styles["Body"]))

story.append(PageBreak())

# Figure 3: Heatmap
story.append(Paragraph("Figure 3: Heatmap - Top 50 DEGs x 33 Sub-tissues", styles["SubHead"]))
story.append(Paragraph(
    "<b>What it shows:</b> For each comparison, the top 50 DEGs (by padj) are "
    "displayed as rows, with the 33 AtGenExpress sub-tissues as columns. Colors "
    "represent z-scored expression (blue = below average, red = above average). "
    "The left annotation shows direction (blue = up, orange = down).", styles["Body"]))
story.append(Paragraph(
    "<b>Statistical concept - Z-score normalization:</b> A z-score transforms "
    "each value to show how many standard deviations it is from the mean. "
    "z = (x - mean) / sd. This allows comparing expression patterns across "
    "genes with very different absolute expression levels. A z-score of +2 "
    "means the gene is expressed 2 standard deviations above its own average.", styles["Body"]))
story.append(callout("Basic Term: Standard Deviation",
    "Standard deviation (SD) measures how spread out values are. If most values "
    "are close to the mean, SD is small. If values vary widely, SD is large. "
    "Z-scores use SD as a ruler: z=1 means '1 SD above average,' z=-2 means "
    "'2 SD below average.' This puts all genes on the same scale."))
story.append(Paragraph(
    "<b>What to look for:</b> Tissue-specific DEGs should show a block of red "
    "(high expression) in their assigned tissue and blue elsewhere. For "
    "radiation_effect, many top DEGs are seedling-specific (red in seedling "
    "columns). For genotype_interaction, the pattern is more mixed, reflecting "
    "the interaction effect rather than a simple tissue-specific response.", styles["Body"]))

story.append(Spacer(1, 8))

# Figure 4: UpSet Diagram
story.append(Paragraph("Figure 4: UpSet Diagram - Organ Overlaps", styles["SubHead"]))
story.append(Paragraph(
    "<b>What it shows:</b> An UpSet diagram visualizes set intersections. Each "
    "organ is a set of DEGs. The bottom-left bars show set sizes (total DEGs "
    "per organ). The main bar chart shows the size of each intersection - "
    "how many DEGs are shared between specific combinations of organs.", styles["Body"]))
story.append(Paragraph(
    "<b>Statistical concept - Set intersection analysis:</b> Unlike a Venn "
    "diagram (which becomes unreadable with more than 3 sets), an UpSet diagram "
    "scales to any number of sets. The connected dots below each bar indicate "
    "which sets are included in that intersection. A bar with a single filled "
    "dot represents genes unique to that organ; bars with multiple connected "
    "dots represent genes shared between those organs.", styles["Body"]))
story.append(callout("Intermediate Term: Set intersection",
    "An intersection is the set of items that belong to all specified sets "
    "simultaneously. If 10 DEGs are in both Root and Seedling, the Root-and-"
    "Seedling intersection has 10 elements. Since each DEG is assigned to only "
    "one predominant organ, most intersections will be empty - the bars with "
    "single dots (unique to each organ) will dominate."))
story.append(Paragraph(
    "<b>What to look for:</b> Because each tissue-specific gene is assigned to "
    "only one predominant organ, the largest bars should be the single-organ "
    "sets (no overlaps). This confirms that the Tau-based tissue assignment "
    "is working correctly - genes are being assigned to distinct tissues.", styles["Body"]))

story.append(PageBreak())

# Figure 5: Forest Plot
story.append(Paragraph("Figure 5: Forest Plot - Tissue Enrichment Test", styles["SubHead"]))
story.append(Paragraph(
    "<b>What it shows:</b> For each comparison, a forest plot displays the DEG "
    "rate (%) for each organ (dots), with the overall mean as a dashed reference "
    "line. Significance markers (*, **, ***) indicate Bonferroni-corrected "
    "p-values from Fisher's exact tests. Colors indicate enrichment (green), "
    "depletion (orange), or not significant (grey).", styles["Body"]))
story.append(Paragraph(
    "<b>Statistical concept - Proportions, confidence intervals, and odds ratios:</b> "
    "The DEG rate is a proportion: (DEGs in organ) / (total genes in organ). "
    "Fisher's exact test compares this proportion to the rest of the genome. "
    "The odds ratio (OR) quantifies the difference: OR > 1 means the organ is "
    "enriched for DEGs, OR < 1 means depleted. The 95% confidence interval "
    "shows the range of plausible OR values.", styles["Body"]))
story.append(callout("Basic Term: Odds Ratio",
    "An odds ratio compares the odds of an event in two groups. If Seedling has "
    "148 DEGs out of 1,167 genes (12.7%) and the rest has 374 out of 9,616 (3.9%), "
    "the odds ratio is about 3.7. This means a tissue-specific Seedling gene is "
    "3.7x more likely to be a DEG than a gene from any other organ."))
story.append(Paragraph(
    "<b>What to look for:</b> In radiation_effect, Seedling (12.7%, ***) and "
    "Leaf_Shoot (8.1%, *) are enriched; Root (3.1%, ***), Flower (3.4%, *), and "
    "Seed_Silique (3.4%, *) are depleted. In Cs137_100cGy, Root (2.4%, **) and "
    "Seedling (3.0%, **) are enriched. The pattern differs by radiation type, "
    "which the CMH test in Part II quantifies.", styles["Body"]))

story.append(PageBreak())

# Cross-comparison figures
story.append(Paragraph("Cross-Comparison Figures", styles["SubHead"]))

story.append(Paragraph("Figure cross1: Tissue Response Heatmap", styles["SubHead"]))
story.append(Paragraph(
    "<b>What it shows:</b> A heatmap of DEG rates (%) for each organ (rows) "
    "across all 6 comparisons (columns). Darker orange = higher DEG rate. "
    "The top annotation bar shows radiation type. Numbers in each cell are "
    "the exact DEG rate.", styles["Body"]))
story.append(Paragraph(
    "<b>Statistical concept - Contingency table visualization:</b> This heatmap "
    "is a visual representation of a 5x6 contingency table. Each cell is a "
    "proportion (DEG rate) that can be compared across rows (organs) and "
    "columns (comparisons). The color scale makes patterns visible that would "
    "be hard to see in a raw table of numbers.", styles["Body"]))
story.append(callout("What to look for",
    "Seedling (row 2) has the darkest cell under radiation_effect (11.96%). "
    "GCR40 and GCR80 columns are nearly white (0% DEG rate). The Cs-137 "
    "columns show intermediate values with a dose gradient (10cGy < 100cGy). "
    "This visual pattern is what the CMH and chi-square tests quantify."))

story.append(Spacer(1, 8))

story.append(Paragraph("Figure cross2: Direction Comparison", styles["SubHead"]))
story.append(Paragraph(
    "<b>What it shows:</b> A faceted grouped bar chart with up/down DEG counts "
    "for each organ, across all 6 comparisons. Blue bars = upregulated, orange "
    "bars = downregulated.", styles["Body"]))
story.append(Paragraph(
    "<b>Statistical concept - Conditional distributions:</b> Each facet (panel) "
    "shows the conditional distribution of direction given organ for one "
    "comparison. Comparing these conditional distributions across facets reveals "
    "whether the direction pattern changes by radiation condition. The chi-square "
    "test for direction (Test 4) formally tests whether these conditional "
    "distributions differ.", styles["Body"]))

story.append(Spacer(1, 8))

story.append(Paragraph("Figure cross3: Dose-Response", styles["SubHead"]))
story.append(Paragraph(
    "<b>What it shows:</b> Line plots of DEG count vs dose for each organ, "
    "separated by radiation source (Cs-137 vs GCR). Each line is an organ, "
    "colored consistently with the organ color palette.", styles["Body"]))
story.append(Paragraph(
    "<b>Statistical concept - Dose-response relationship:</b> A dose-response "
    "curve shows how a biological outcome (DEG count) changes with increasing "
    "dose of a treatment (radiation). A steeper slope means greater sensitivity "
    "to dose. The CMH test for dose-response (OR=2.846 for Cs-137) formally "
    "tests whether the odds of being a DEG increase with dose, after accounting "
    "for tissue differences.", styles["Body"]))

story.append(Spacer(1, 8))

story.append(Paragraph("Figure cross4: Radiation Type Comparison", styles["SubHead"]))
story.append(Paragraph(
    "<b>What it shows:</b> A grouped bar chart comparing DEG rates by organ "
    "between Co-60 100 Gy and Cs-137 100 cGy. Blue bars = Co-60, green bars = "
    "Cs-137.", styles["Body"]))
story.append(Paragraph(
    "<b>Statistical concept - Stratified comparison:</b> This figure visualizes "
    "the comparison that the CMH test quantifies. The CMH test stratifies by "
    "organ (treating each organ as a separate stratum) and asks: after accounting "
    "for organ-specific baseline rates, is there an overall difference in DEG "
    "rate between the two radiation types? The answer is yes (OR=0.328, "
    "p=1.1e-39): Co-60 produces 3.1x more tissue-specific DEGs than Cs-137.", styles["Body"]))
story.append(callout("Intermediate Term: Stratification",
    "Stratification means analyzing each subgroup (organ) separately, then "
    "combining the results. This is more powerful than ignoring organ differences "
    "because it removes the confounding effect of organs having different baseline "
    "DEG rates. The CMH test is the stratified version of the chi-square test."))

story.append(PageBreak())

# ============================================================
# PART II: Code Blocks Explained
# ============================================================
story.append(Paragraph("Part II: Code Blocks Explained", styles["SectionHead"]))
story.append(divider())

# Code 1: Tau Index
story.append(Paragraph("Code Block 1: Tau Tissue-Specificity Index", styles["SubHead"]))
story.append(Paragraph(
    "compute_tau &lt;- function(expr_matrix) {<br/>"
    "&nbsp;&nbsp;tau &lt;- apply(expr_matrix, 1, function(x) {<br/>"
    "&nbsp;&nbsp;&nbsp;&nbsp;x &lt;- as.numeric(x)<br/>"
    "&nbsp;&nbsp;&nbsp;&nbsp;if (max(x) == 0) return(NA)<br/>"
    "&nbsp;&nbsp;&nbsp;&nbsp;sum(1 - x / max(x)) / (length(x) - 1)<br/>"
    "&nbsp;&nbsp;})<br/>"
    "&nbsp;&nbsp;return(tau)<br/>"
    "}", styles["CodeStyle"]))
story.append(Paragraph(
    "<b>What it does:</b> Computes the Tau index for each gene across all tissues. "
    "The apply() function loops over rows (genes). For each gene, it normalizes "
    "expression by the maximum, subtracts from 1, sums, and divides by (n-1).", styles["Body"]))
story.append(Paragraph(
    "<b>Statistical concept - Tissue-specificity metric:</b> Tau is a "
    "tissue-specificity score ranging from 0 to 1. The formula "
    "tau = sum(1 - x_i/max(x)) / (n-1) measures how unevenly a gene is expressed "
    "across tissues. If a gene is expressed only in one tissue, all other values "
    "are near 0, so each (1 - 0/max) = 1, and tau approaches 1. If expression is "
    "uniform, each (1 - x/max) is near 0, and tau approaches 0.", styles["Body"]))
story.append(callout("Basic Term: Normalization",
    "Normalization means putting different things on the same scale. Here, "
    "dividing by max(x) normalizes all expression values to [0, 1], so genes "
    "with very different absolute expression levels can be compared. A gene "
    "expressed at 1000 in one tissue and 10 elsewhere gets the same Tau as one "
    "expressed at 100 and 1."))

story.append(Spacer(1, 8))

# Code 2: Chi-square test
story.append(Paragraph("Code Block 2: Chi-Square Test of Independence", styles["SubHead"]))
story.append(Paragraph(
    "contig &lt;- table(ts_genes$predominant_broad_organ, ts_genes$deg_flag)<br/>"
    "chi_test &lt;- chisq.test(contig)", styles["CodeStyle"]))
story.append(Paragraph(
    "<b>What it does:</b> Builds a contingency table (organ x DEG status) and "
    "runs a chi-square test of independence.", styles["Body"]))
story.append(Paragraph(
    "<b>Statistical concept - Chi-square test of independence:</b> This test "
    "asks: is the proportion of DEGs the same across all organs, or does it "
    "differ? The null hypothesis is that organ and DEG status are independent "
    "(no tissue enrichment). The test statistic X2 measures how much the "
    "observed counts deviate from what we'd expect under independence. A large "
    "X2 with a small p-value means the organs have significantly different DEG "
    "rates.", styles["Body"]))
story.append(callout("Intermediate Term: Expected vs Observed",
    "The chi-square test compares observed counts to expected counts. If organs "
    "and DEG status were independent, we'd expect each organ to have the same "
    "DEG rate (the overall average). The test measures how far the actual data "
    "deviates from this expectation. X2 = sum((observed - expected)2 / expected)."))
story.append(Paragraph(
    "<b>Results:</b> radiation_effect: X2=233.6, p=2.2e-49 (highly significant). "
    "genotype_interaction: X2=14.4, p=0.006 (significant). Cs137_100cGy: X2=53.9, "
    "p=5.7e-11. Cs137_10cGy: X2=44.4, p=5.3e-9. GCR40/GCR80: skipped (too few DEGs).", styles["Body"]))

story.append(PageBreak())

# Code 3: Fisher's exact test
story.append(Paragraph("Code Block 3: Per-Tissue Fisher's Exact Test", styles["SubHead"]))
story.append(Paragraph(
    "fisher_mat &lt;- matrix(c(organ_deg, organ_non, rest_deg, rest_non), nrow=2)<br/>"
    "ft &lt;- fisher.test(fisher_mat)<br/>"
    "padj &lt;- p.adjust(ft$p.value, method='bonferroni', n=5)", styles["CodeStyle"]))
story.append(Paragraph(
    "<b>What it does:</b> For each organ, builds a 2x2 table (organ vs rest, "
    "DEG vs non-DEG) and runs Fisher's exact test. Then applies Bonferroni "
    "correction for 5 tests (one per organ).", styles["Body"]))
story.append(Paragraph(
    "<b>Statistical concept - Fisher's exact test:</b> Fisher's exact test is "
    "an alternative to the chi-square test for 2x2 tables. It is preferred when "
    "sample sizes are small because it calculates the exact probability rather "
    "than relying on a large-sample approximation. The test returns an odds "
    "ratio (OR) and a p-value.", styles["Body"]))
story.append(callout("Intermediate Term: Bonferroni Correction",
    "When running multiple tests, the chance of a false positive increases. "
    "The Bonferroni correction adjusts for this by multiplying each p-value "
    "by the number of tests (here, 5). If you run 5 tests at alpha=0.05, the "
    "Bonferroni-adjusted threshold is 0.05/5 = 0.01. This is conservative - "
    "it minimizes false positives but may miss true positives."))
story.append(Paragraph(
    "<b>Results:</b> In radiation_effect: Seedling OR=3.67 (enriched, padj=1.5e-30), "
    "Leaf_Shoot OR=1.94 (enriched, padj=1.7e-6), Root OR=0.56 (depleted, "
    "padj=5.8e-7). In Cs137_100cGy: Root OR=1.82 (enriched, padj=9.3e-4), "
    "Seedling OR=2.14 (enriched, padj=1.3e-3).", styles["Body"]))

story.append(Spacer(1, 8))

# Code 4: Pairwise Fisher's
story.append(Paragraph("Code Block 4: Pairwise Fisher's Exact Tests", styles["SubHead"]))
story.append(Paragraph(
    "for (pr in combn(BROAD_ORGANS, 2, simplify=FALSE)) {<br/>"
    "&nbsp;&nbsp;mat &lt;- matrix(c(contig[pr[1],'yes'], contig[pr[1],'no'],<br/>"
    "&nbsp;&nbsp;&nbsp;&nbsp;contig[pr[2],'yes'], contig[pr[2],'no']), nrow=2)<br/>"
    "&nbsp;&nbsp;ft &lt;- fisher.test(mat)<br/>"
    "&nbsp;&nbsp;padj &lt;- p.adjust(ft$p.value, method='bonferroni', n=10)<br/>"
    "}", styles["CodeStyle"]))
story.append(Paragraph(
    "<b>What it does:</b> Runs all 10 pairwise comparisons between organs "
    "(5 choose 2 = 10 pairs) using Fisher's exact test, with Bonferroni "
    "correction for 10 tests.", styles["Body"]))
story.append(Paragraph(
    "<b>Statistical concept - Multiple pairwise comparisons:</b> The chi-square "
    "test tells us that organs differ overall, but not which specific pairs differ. "
    "Pairwise Fisher's tests identify exactly which organ pairs have significantly "
    "different DEG rates. With 5 organs, there are 10 possible pairs. The "
    "Bonferroni correction is stricter here (n=10) to account for the larger "
    "number of tests.", styles["Body"]))
story.append(Paragraph(
    "<b>Results:</b> radiation_effect: 7/10 pairs significant. Seedling (12.7%) > "
    "Leaf_Shoot (8.1%) > Seed_Silique (3.4%) = Flower (3.4%) > Root (3.1%). "
    "Cs137_100cGy: 5/10 significant. Cs137_10cGy: 4/10 significant.", styles["Body"]))

story.append(PageBreak())

# Code 5: Direction chi-square
story.append(Paragraph("Code Block 5: Direction-of-Response Chi-Square", styles["SubHead"]))
story.append(Paragraph(
    "dir_contig &lt;- table(degs_only$predominant_broad_organ, degs_only$direction)<br/>"
    "dir_chi &lt;- chisq.test(dir_contig)", styles["CodeStyle"]))
story.append(Paragraph(
    "<b>What it does:</b> Tests whether the up/down ratio differs across organs "
    "for DEGs within a comparison.", styles["Body"]))
story.append(Paragraph(
    "<b>Statistical concept - Testing conditional distributions:</b> This is "
    "another chi-square test, but instead of testing DEG vs non-DEG, it tests "
    "up vs down. The question is: do all organs have the same up/down ratio, "
    "or do some organs respond differently in direction? A significant result "
    "means the direction of response is tissue-dependent.", styles["Body"]))
story.append(Paragraph(
    "<b>Results:</b> radiation_effect: X2=62.7, p=8.0e-13. Root is 97% up, "
    "Seedling is 100% up, Flower is 76% up (more downregulated than others). "
    "genotype_interaction: X2=47.7, p=1.1e-9. Root is 89% down, Seedling is "
    "100% down - the opposite pattern from radiation_effect.", styles["Body"]))

story.append(Spacer(1, 8))

# Code 6: CMH test (NEW)
story.append(Paragraph("Code Block 6: Cochran-Mantel-Haenszel Test (NEW)", styles["SubHead"]))
story.append(Paragraph(
    "# Build 2x2xK array: condition x DEG x organ strata<br/>"
    "rad_array &lt;- array(0, dim=c(2, 2, 5),<br/>"
    "&nbsp;&nbsp;dimnames=list(c('Co60','Cs137'), c('no','yes'), BROAD_ORGANS))<br/>"
    "cmh_rad &lt;- mantelhaen.test(rad_array)", styles["CodeStyle"]))
story.append(Paragraph(
    "<b>What it does:</b> Tests whether the odds ratio between condition (Co-60 "
    "vs Cs-137) and DEG status is consistent across organ strata. Returns a "
    "common odds ratio and p-value.", styles["Body"]))
story.append(Paragraph(
    "<b>Statistical concept - Stratified analysis with CMH:</b> The "
    "Cochran-Mantel-Haenszel (CMH) test is a stratified version of the chi-square "
    "test. Instead of pooling all organs together (which would confound organ "
    "differences with radiation type differences), it analyzes each organ "
    "separately and combines the results. The 'common odds ratio' is a weighted "
    "average of the organ-specific odds ratios.", styles["Body"]))
story.append(callout("Intermediate Term: Confounding",
    "Confounding occurs when an outside variable distorts the relationship "
    "between the variables you're studying. If Seedling has more DEGs than Root, "
    "and Co-60 happens to have more Seedling samples, a simple comparison would "
    "make Co-60 look more potent - but this could be due to the tissue difference, "
    "not the radiation type. Stratification (CMH) removes this confounding by "
    "comparing within each organ first."))
story.append(Paragraph(
    "<b>Results:</b> Co-60 vs Cs-137: common OR=0.328 (p=1.1e-39), meaning "
    "Cs-137 has 1/0.328 = 3.1x lower odds of producing a tissue-specific DEG. "
    "Cs-137 dose-response: common OR=2.846 (p=5.6e-13), meaning 100 cGy has "
    "2.8x higher odds than 10 cGy.", styles["Body"]))

story.append(PageBreak())

# Code 7: Cross-comparison chi-square (NEW)
story.append(Paragraph("Code Block 7: Cross-Comparison Chi-Square (NEW)", styles["SubHead"]))
story.append(Paragraph(
    "# Compare organ distribution between radiation_effect and genotype_interaction<br/>"
    "genotype_mat &lt;- rbind(organ_dist_rad, organ_dist_int)<br/>"
    "gen_chi &lt;- chisq.test(genotype_mat)<br/><br/>"
    "# Compare direction patterns<br/>"
    "dir_mat &lt;- rbind(dir_rad, dir_int)<br/>"
    "dir_chi &lt;- chisq.test(dir_mat)", styles["CodeStyle"]))
story.append(Paragraph(
    "<b>What it does:</b> Two chi-square tests comparing the tissue distribution "
    "and direction patterns between the radiation main effect and the sog1-1 "
    "interaction effect.", styles["Body"]))
story.append(Paragraph(
    "<b>Statistical concept - Comparing distributions across conditions:</b> "
    "These tests use a 2x5 (condition x organ) or 2x2 (condition x direction) "
    "contingency table. The first asks: does the organ distribution of DEGs "
    "differ between the two comparisons? The second asks: does the up/down "
    "ratio differ?", styles["Body"]))
story.append(Paragraph(
    "<b>Results:</b> Organ distribution: X2=87.1, p=5.5e-18 (significantly "
    "different). Direction: X2=456.5, p=2.8e-101 (extremely different). The "
    "direction result is one of the strongest signals in the entire analysis - "
    "the sog1-1 interaction reverses the direction of the radiation response.", styles["Body"]))

story.append(PageBreak())

# ============================================================
# PART III: Statistical Glossary
# ============================================================
story.append(Paragraph("Part III: Statistical Glossary", styles["SectionHead"]))
story.append(divider())

glossary = [
    ("Adjusted p-value (padj)",
     "A p-value corrected for multiple testing. DESeq2 uses the Benjamini-Hochberg "
     "procedure, which controls the false discovery rate (FDR). A padj of 0.05 means "
     "we expect 5% of called DEGs to be false positives."),
    ("Bonferroni Correction",
     "A conservative multiple testing correction that multiplies each p-value by "
     "the number of tests. If you run 5 tests, the threshold becomes 0.05/5 = 0.01. "
     "It minimizes false positives but may miss true positives."),
    ("Chi-Square Test of Independence",
     "Tests whether two categorical variables are associated. Here, it tests whether "
     "organ membership and DEG status are independent. A large X2 statistic with a "
     "small p-value means the variables are associated (organs have different DEG rates)."),
    ("Cochran-Mantel-Haenszel (CMH) Test",
     "A stratified version of the chi-square test. Instead of pooling all data, it "
     "analyzes each stratum (organ) separately and combines results. This removes "
     "confounding by the stratifying variable. Returns a common odds ratio."),
    ("Confidence Interval (CI)",
     "A range of values that likely contains the true parameter. A 95% CI means "
     "that if you repeated the experiment many times, 95% of the intervals would "
     "contain the true value. Narrower CIs indicate more precise estimates."),
    ("Confounding",
     "When a third variable distorts the relationship between two studied variables. "
     "For example, if organs have different baseline DEG rates, comparing radiation "
     "types without accounting for organ would confound tissue effects with radiation effects."),
    ("Contingency Table",
     "A table showing the frequency distribution of two or more categorical variables. "
     "Rows and columns are categories; cells are counts. The chi-square test operates on "
     "contingency tables."),
    ("DESeq2",
     "An R package for differential expression analysis of RNA-seq count data. It models "
     "counts with a negative binomial distribution and performs internal normalization "
     "(median-of-ratios). Requires raw integer counts as input."),
    ("Effect Size",
     "The magnitude of a difference or relationship. In differential expression, the "
     "effect size is the log2 fold change. A large effect size means a big change in "
     "expression, regardless of statistical significance."),
    ("Fisher's Exact Test",
     "An exact test for 2x2 contingency tables. Preferred over chi-square when sample "
     "sizes are small. Returns an odds ratio and p-value. Used here for per-organ and "
     "pairwise comparisons."),
    ("Frequency Distribution",
     "A summary of how often each value or category occurs. Bar charts visualize "
     "frequency distributions for categorical variables."),
    ("Log2 Fold Change (log2FC)",
     "The log base 2 of the fold change. A log2FC of 1 means 2-fold upregulation; "
     "-1 means 2-fold downregulation. Log scale makes up and down changes symmetric: "
     "+1 and -1 are equally far from 0."),
    ("Negative Binomial Distribution",
     "A probability distribution for count data that allows for overdispersion (variance "
     "greater than the mean). DESeq2 uses this to model RNA-seq read counts."),
    ("Odds Ratio (OR)",
     "The ratio of odds between two groups. OR=2 means the event is twice as likely in "
     "group 1 vs group 2. OR=0.5 means half as likely. OR=1 means no difference. Used "
     "in Fisher's exact and CMH tests."),
    ("P-value",
     "The probability of observing data as extreme as what you got, assuming the null "
     "hypothesis is true. A small p-value (< 0.05) suggests the null hypothesis is unlikely. "
     "It does NOT measure the probability that the null is true."),
    ("Stratification",
     "Analyzing data within subgroups (strata) separately, then combining results. "
     "The CMH test stratifies by organ. This removes confounding by the stratifying "
     "variable and is more powerful than ignoring subgroup differences."),
    ("Tau Tissue-Specificity Index",
     "A score from 0 to 1 measuring how specifically a gene is expressed in one tissue. "
     "0 = expressed equally everywhere; 1 = expressed in only one tissue. Threshold of "
     "0.6 classifies genes as tissue-specific."),
    ("Z-score",
     "The number of standard deviations a value is from the mean. z = (x - mean) / sd. "
     "Used in heatmaps to normalize expression across genes with different absolute levels. "
     "Positive z = above average; negative = below average."),
]

for term, definition in glossary:
    story.append(Paragraph(term, styles["GlossaryTerm"]))
    story.append(Paragraph(definition, styles["GlossaryDef"]))

# Build PDF
doc.build(story, onFirstPage=page_header_footer, onLaterPages=page_header_footer)

# Validate
from pypdf import PdfReader
reader = PdfReader(output_path)
page_count = len(reader.pages)
file_size = os.path.getsize(output_path)
print(f"PDF generated: {output_path}")
print(f"Pages: {page_count}, Size: {file_size:,} bytes")
assert page_count >= 5, f"Only {page_count} pages"
assert file_size > 5000, f"Only {file_size} bytes"
first_text = reader.pages[0].extract_text()
assert len(first_text.strip()) > 0, "No text on first page"
print("Validation passed.")


PDF generated: /mnt/results/statistical_walkthrough_figures_and_code_v2.pdf
Pages: 13, Size: 29,118 bytes
Validation passed.
